# Software-Engineered Chest X-Ray Analysis and Explanation System

**Notebook:** Grounded Language Model Adaptation and Integration  
**Subject:** AIMLCZG546 - Software Engineering for Machine Learning  
**Assessment:** Assignment II  
**Group:** 89

## Group Details & Contribution

| BITS ID | Name | Engineering contribution | Weight |
|---|---|---|---:|
| 2024ac05672@wilp.bits-pilani.ac.in | Ananda Vadivel D | Requirements formulation, GR4ML view preparation, quality requirement justification | 25% |
| 2024ac05968@wilp.bits-pilani.ac.in | Adurti Sai Venkatesh | Dataset preparation, exploratory data analysis, visualization support, data quality validation | 25% |
| 2024ac05653@wilp.bits-pilani.ac.in | D Mallikarjuna Reddy | ML pipeline implementation, model comparison, final evaluation, report integration | 25% |
| 2024ac05055@wilp.bits-pilani.ac.in | Krupashankar Subramani | System architecture, architectural pattern mapping, model registry artifacts, inference and application validation | 25% |

## Notebook Index

1. [Grounded Language Model Adaptation and Integration](#grounded-language-model-adaptation-and-integration)
2. [1. Language Integration Configuration](#1-language-integration-configuration)
3. [2. Grounded Finding Ontology and Safety Boundaries](#2-grounded-finding-ontology-and-safety-boundaries)
4. [3. Versioned Language Task and Prompt Registry](#3-versioned-language-task-and-prompt-registry)
5. [4. Leakage-Resistant Language Dataset Design](#4-leakage-resistant-language-dataset-design)
6. [5. Frozen Computer-Vision Grounding Contract](#5-frozen-computer-vision-grounding-contract)
7. [6. Controlled Confidence and Scenario Generation](#6-controlled-confidence-and-scenario-generation)
8. [7. Structured Grounding Input Serialization](#7-structured-grounding-input-serialization)
9. [8. Controlled Target Response Construction](#8-controlled-target-response-construction)
10. [9. Balanced Grounded Language Record Generation](#9-balanced-grounded-language-record-generation)
11. [10. Dataset-Wide Grounding and Safety Audit](#10-dataset-wide-grounding-and-safety-audit)
12. [11. Versioned Dataset Export and Integrity Verification](#11-versioned-dataset-export-and-integrity-verification)
13. [12. Tokenizer Retrieval and Sequence-Length Analysis](#12-tokenizer-retrieval-and-sequence-length-analysis)
14. [13. Frozen Tokenization Contract and Dataset Materialization](#13-frozen-tokenization-contract-and-dataset-materialization)
15. [14. FLAN-T5-Small Model Initialization](#14-flan-t5-small-model-initialization)
16. [15. Fine-Tuning and MLflow Configuration](#15-fine-tuning-and-mlflow-configuration)
17. [16. Trainer Assembly and Pre-Training Gate](#16-trainer-assembly-and-pre-training-gate)
18. [17. Versioned Full-Model Fine-Tuning](#17-versioned-full-model-fine-tuning)
19. [18. Versioned Fine-Tuned Model Export](#18-versioned-fine-tuned-model-export)
20. [19. Held-Out Language Evaluation](#19-held-out-language-evaluation)
21. [20. Held-Out Error Analysis and Representative Generations](#20-held-out-error-analysis-and-representative-generations)
22. [21. Deterministic Output Guardrail and Safe Fallback](#21-deterministic-output-guardrail-and-safe-fallback)
23. [22. Language Integration Artifact Registry and Final Readiness Gate](#22-language-integration-artifact-registry-and-final-readiness-gate)
24. [23. Grounded Language Integration Summary](#23-grounded-language-integration-summary)

---

The notebook records the executable implementation, retained runtime outputs, and verifiable engineering evidence for this component.


# Grounded Language Model Adaptation and Integration

This notebook develops the grounded language component of the Software-Engineered Chest X-Ray Analysis and Explanation System. A compact FLAN-T5 model is fine-tuned using controlled ChestMNIST-domain records for structured report generation, plain-language explanation, grounded question answering, and educational follow-up guidance.

The language model receives only structured information derived from the frozen computer-vision contract, approved finding descriptions, confidence values, thresholds, and explicit safety limitations. It does not inspect chest X-ray images independently. All generated content is intended for educational decision support and must not be interpreted as a clinical diagnosis or a substitute for professional medical review.

## 1. Language Integration Configuration

This section restores the registered solution paths and Hugging Face cache routing required by the current notebook. It initializes the dedicated directories for derived language data, the versioned fine-tuned model, and language outputs. A storage gate also protects a minimum reserve of 4 GiB before any model download or training activity begins.


In [1]:
import os
import shutil
from pathlib import Path

import yaml


# -------------------------------------------------------------------------
# Load the registered solution paths
# -------------------------------------------------------------------------
PATH_REGISTRY_PATH = Path(
    "/home/jovyan/chest-xray-ai-assistant/configs/paths.yaml"
)

if not PATH_REGISTRY_PATH.is_file():
    raise FileNotFoundError(
        f"Path registry was not found: {PATH_REGISTRY_PATH}"
    )

with PATH_REGISTRY_PATH.open("r", encoding="utf-8") as file:
    path_registry = yaml.safe_load(file)

if not isinstance(path_registry, dict):
    raise ValueError("The path registry must contain a YAML mapping.")


def collect_registered_paths(value):
    """Recursively collect path-like string values from the registry."""
    collected = []

    if isinstance(value, dict):
        for nested_value in value.values():
            collected.extend(collect_registered_paths(nested_value))
    elif isinstance(value, list):
        for nested_value in value:
            collected.extend(collect_registered_paths(nested_value))
    elif isinstance(value, str) and value.startswith("/"):
        collected.append(Path(value))

    return collected


registered_paths = collect_registered_paths(path_registry)

expected_solution_root = Path(
    "/home/jovyan/chest-xray-ai-assistant"
)
expected_data_root = Path(
    "/home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data"
)

if expected_solution_root not in registered_paths:
    raise KeyError(
        "The registered solution root does not match the expected path: "
        f"{expected_solution_root}"
    )

if expected_data_root not in registered_paths:
    raise KeyError(
        "The registered data root does not match the expected path: "
        f"{expected_data_root}"
    )

SOLUTION_ROOT = expected_solution_root
DATA_ROOT = expected_data_root


# -------------------------------------------------------------------------
# Define the language-model contract
# -------------------------------------------------------------------------
BASE_LANGUAGE_MODEL = "google/flan-t5-small"
LANGUAGE_MODEL_VERSION = "flan-t5-small-chestmnist-v1"

LANGUAGE_DATA_DIR = DATA_ROOT / "processed" / "grounded_language"
LANGUAGE_MODEL_DIR = DATA_ROOT / "models" / LANGUAGE_MODEL_VERSION
LANGUAGE_OUTPUT_DIR = DATA_ROOT / "outputs" / "language"

for directory in (
    LANGUAGE_DATA_DIR,
    LANGUAGE_MODEL_DIR,
    LANGUAGE_OUTPUT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


# -------------------------------------------------------------------------
# Restore runtime artifact routing
# -------------------------------------------------------------------------
HF_CACHE_DIR = DATA_ROOT / "hf-cache"
HF_DATASETS_CACHE_DIR = HF_CACHE_DIR / "datasets"
TORCH_CACHE_DIR = DATA_ROOT / "models" / "torch-cache"

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_DATASETS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TORCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE_DIR)
os.environ["TORCH_HOME"] = str(TORCH_CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# -------------------------------------------------------------------------
# Apply the protected storage gate
# -------------------------------------------------------------------------
GIB = 1024 ** 3
PROTECTED_RESERVE_GIB = 4.0
ESTIMATED_LANGUAGE_STAGE_GIB = 1.5

storage = shutil.disk_usage(DATA_ROOT)
free_storage_gib = storage.free / GIB
usable_storage_gib = max(
    0.0,
    free_storage_gib - PROTECTED_RESERVE_GIB,
)

storage_ready = usable_storage_gib >= ESTIMATED_LANGUAGE_STAGE_GIB


# -------------------------------------------------------------------------
# Report the resolved configuration
# -------------------------------------------------------------------------
print("GROUNDED LANGUAGE CONFIGURATION")
print("-" * 100)
print(f"Path registry             : {PATH_REGISTRY_PATH}")
print(f"Solution root             : {SOLUTION_ROOT}")
print(f"Data root                 : {DATA_ROOT}")
print(f"Base language model       : {BASE_LANGUAGE_MODEL}")
print(f"Language model version    : {LANGUAGE_MODEL_VERSION}")
print(f"Derived language data     : {LANGUAGE_DATA_DIR}")
print(f"Versioned model directory : {LANGUAGE_MODEL_DIR}")
print(f"Language output directory : {LANGUAGE_OUTPUT_DIR}")
print(f"Hugging Face cache        : {HF_CACHE_DIR}")
print(f"Current free storage      : {free_storage_gib:.2f} GiB")
print(f"Protected reserve         : {PROTECTED_RESERVE_GIB:.2f} GiB")
print(f"Usable after reserve      : {usable_storage_gib:.2f} GiB")
print(f"Estimated stage need      : {ESTIMATED_LANGUAGE_STAGE_GIB:.2f} GiB")
print(f"Storage readiness         : {'PASS' if storage_ready else 'FAIL'}")

if not storage_ready:
    raise RuntimeError(
        "Insufficient storage for the grounded language stage while "
        "preserving the required 4 GiB reserve."
    )

print("-" * 100)
print("STATUS: READY FOR GROUNDED LANGUAGE DATA DESIGN")

GROUNDED LANGUAGE CONFIGURATION
----------------------------------------------------------------------------------------------------
Path registry             : /home/jovyan/chest-xray-ai-assistant/configs/paths.yaml
Solution root             : /home/jovyan/chest-xray-ai-assistant
Data root                 : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data
Base language model       : google/flan-t5-small
Language model version    : flan-t5-small-chestmnist-v1
Derived language data     : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed/grounded_language
Versioned model directory : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/models/flan-t5-small-chestmnist-v1
Language output directory : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/language
Hugging Face cache        : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/hf-cache
Current free storage      : 11.35 GiB
Protected reserve         : 4.00 GiB
Usab

## 2. Grounded Finding Ontology and Safety Boundaries

The language model requires a controlled ontology that maps every ChestMNIST label to its training prevalence and an approved plain-language description. These descriptions provide contextual meaning without claiming that a finding has been clinically confirmed.

This section also establishes the fixed language-safety boundary used during dataset construction, fine-tuning, evaluation, and application inference. The special state **no target finding** means that none of the 14 supported findings crossed its frozen threshold; it does not mean that the chest X-ray is clinically normal.


In [2]:
import pandas as pd


# -------------------------------------------------------------------------
# Define the controlled ChestMNIST finding ontology
# -------------------------------------------------------------------------
FINDING_ONTOLOGY = [
    {
        "label_id": 0,
        "label_name": "atelectasis",
        "display_name": "Atelectasis",
        "training_prevalence": 0.1019,
        "approved_description": (
            "A pattern associated with reduced expansion or partial collapse "
            "of part of the lung."
        ),
    },
    {
        "label_id": 1,
        "label_name": "cardiomegaly",
        "display_name": "Cardiomegaly",
        "training_prevalence": 0.0249,
        "approved_description": (
            "A pattern associated with an enlarged appearance of the heart."
        ),
    },
    {
        "label_id": 2,
        "label_name": "effusion",
        "display_name": "Effusion",
        "training_prevalence": 0.1180,
        "approved_description": (
            "A pattern associated with fluid collecting in the space around "
            "the lungs."
        ),
    },
    {
        "label_id": 3,
        "label_name": "infiltration",
        "display_name": "Infiltration",
        "training_prevalence": 0.1773,
        "approved_description": (
            "A broad dataset pattern associated with increased material or "
            "opacity within lung tissue."
        ),
    },
    {
        "label_id": 4,
        "label_name": "mass",
        "display_name": "Mass",
        "training_prevalence": 0.0508,
        "approved_description": (
            "A dataset pattern associated with a larger focal opacity or "
            "mass-like appearance."
        ),
    },
    {
        "label_id": 5,
        "label_name": "nodule",
        "display_name": "Nodule",
        "training_prevalence": 0.0558,
        "approved_description": (
            "A dataset pattern associated with a small, rounded focal opacity."
        ),
    },
    {
        "label_id": 6,
        "label_name": "pneumonia",
        "display_name": "Pneumonia",
        "training_prevalence": 0.0125,
        "approved_description": (
            "A pattern associated with lung opacity that may occur with "
            "pneumonia, without confirming infection."
        ),
    },
    {
        "label_id": 7,
        "label_name": "pneumothorax",
        "display_name": "Pneumothorax",
        "training_prevalence": 0.0472,
        "approved_description": (
            "A pattern associated with air in the space between the lung and "
            "chest wall."
        ),
    },
    {
        "label_id": 8,
        "label_name": "consolidation",
        "display_name": "Consolidation",
        "training_prevalence": 0.0416,
        "approved_description": (
            "A pattern associated with an area of lung airspace becoming "
            "filled and appearing denser."
        ),
    },
    {
        "label_id": 9,
        "label_name": "edema",
        "display_name": "Edema",
        "training_prevalence": 0.0215,
        "approved_description": (
            "A pattern associated with increased fluid within the lungs."
        ),
    },
    {
        "label_id": 10,
        "label_name": "emphysema",
        "display_name": "Emphysema",
        "training_prevalence": 0.0229,
        "approved_description": (
            "A pattern associated with over-expanded lungs and changes in "
            "lung airspaces."
        ),
    },
    {
        "label_id": 11,
        "label_name": "fibrosis",
        "display_name": "Fibrosis",
        "training_prevalence": 0.0148,
        "approved_description": (
            "A pattern associated with scarring or fibrotic change in lung "
            "tissue."
        ),
    },
    {
        "label_id": 12,
        "label_name": "pleural",
        "display_name": "Pleural Abnormality",
        "training_prevalence": 0.0290,
        "approved_description": (
            "A broad dataset label associated with an abnormal appearance of "
            "the lining around the lungs."
        ),
    },
    {
        "label_id": 13,
        "label_name": "hernia",
        "display_name": "Hernia",
        "training_prevalence": 0.0018,
        "approved_description": (
            "A pattern associated with tissue or an organ projecting through "
            "an opening near the diaphragm."
        ),
    },
]


# -------------------------------------------------------------------------
# Define the fixed grounding and safety language
# -------------------------------------------------------------------------
NO_TARGET_FINDING_DESCRIPTION = (
    "None of the 14 supported ChestMNIST findings crossed its frozen "
    "decision threshold. This does not establish that the chest X-ray is "
    "clinically normal."
)

EDUCATIONAL_USE_LIMITATION = (
    "This output is generated by an educational decision-support prototype. "
    "It is not a diagnosis and should not replace review by a qualified "
    "healthcare professional."
)

GRADCAM_LIMITATION = (
    "Grad-CAM highlights image regions that influenced a model output. "
    "It does not confirm a lesion, provide segmentation, or establish a "
    "clinical diagnosis."
)

PROFESSIONAL_REVIEW_GUIDANCE = (
    "A qualified healthcare professional can interpret the image together "
    "with symptoms, history, examination findings, and other tests."
)


# -------------------------------------------------------------------------
# Validate the ontology contract
# -------------------------------------------------------------------------
ontology_df = pd.DataFrame(FINDING_ONTOLOGY)

expected_label_ids = list(range(14))
expected_label_names = [
    "atelectasis",
    "cardiomegaly",
    "effusion",
    "infiltration",
    "mass",
    "nodule",
    "pneumonia",
    "pneumothorax",
    "consolidation",
    "edema",
    "emphysema",
    "fibrosis",
    "pleural",
    "hernia",
]

validation_checks = {
    "Fourteen findings represented": len(ontology_df) == 14,
    "Label identifiers preserve order": (
        ontology_df["label_id"].tolist() == expected_label_ids
    ),
    "Label names preserve model contract": (
        ontology_df["label_name"].tolist() == expected_label_names
    ),
    "Label names are unique": ontology_df["label_name"].is_unique,
    "Descriptions are complete": (
        ontology_df["approved_description"].str.strip().ne("").all()
    ),
    "Prevalence values are valid": (
        ontology_df["training_prevalence"].between(0.0, 1.0).all()
    ),
    "No-target-finding boundary defined": bool(
        NO_TARGET_FINDING_DESCRIPTION.strip()
    ),
    "Educational limitation defined": bool(
        EDUCATIONAL_USE_LIMITATION.strip()
    ),
    "Grad-CAM limitation defined": bool(GRADCAM_LIMITATION.strip()),
    "Professional review wording defined": bool(
        PROFESSIONAL_REVIEW_GUIDANCE.strip()
    ),
}

print("GROUNDING ONTOLOGY VALIDATION")
print("-" * 100)

for check_name, passed in validation_checks.items():
    print(f"{check_name:<48}: {'PASS' if passed else 'FAIL'}")

print("-" * 100)
print(
    ontology_df[
        [
            "label_id",
            "label_name",
            "training_prevalence",
            "approved_description",
        ]
    ].to_string(index=False)
)

if not all(validation_checks.values()):
    raise RuntimeError(
        "The controlled finding ontology or safety contract is incomplete."
    )

print("-" * 100)
print("STATUS: APPROVED ONTOLOGY AND SAFETY BOUNDARIES ESTABLISHED")

GROUNDING ONTOLOGY VALIDATION
----------------------------------------------------------------------------------------------------
Fourteen findings represented                   : PASS
Label identifiers preserve order                : PASS
Label names preserve model contract             : PASS
Label names are unique                          : PASS
Descriptions are complete                       : PASS
Prevalence values are valid                     : PASS
No-target-finding boundary defined              : PASS
Educational limitation defined                  : PASS
Grad-CAM limitation defined                     : PASS
Professional review wording defined             : PASS
----------------------------------------------------------------------------------------------------
 label_id    label_name  training_prevalence                                                                                approved_description
        0   atelectasis               0.1019                A pattern ass

## 3. Versioned Language Task and Prompt Registry

The fine-tuned model will use four explicit instruction prefixes so that each language function remains identifiable, testable, and reproducible. This section defines the permitted input fields, required output structure, and task-specific grounding rules.

The registry is versioned independently from the computer-vision model. This allows generated outputs to retain both model lineage and prompt-contract lineage when they are later delivered through the API.


In [3]:
from datetime import datetime, timezone


# -------------------------------------------------------------------------
# Define the versioned task and prompt contract
# -------------------------------------------------------------------------
PROMPT_REGISTRY_VERSION = "grounded-language-prompts-v1"
PROMPT_REGISTRY_PATH = SOLUTION_ROOT / "configs" / "prompt_registry.yaml"

COMMON_INPUT_FIELDS = [
    "task_type",
    "finding_names",
    "probabilities",
    "frozen_thresholds",
    "threshold_decisions",
    "no_target_finding",
    "confidence_categories",
    "model_version",
    "approved_descriptions",
    "limitation_boundary",
    "user_question",
]

TASK_REGISTRY = {
    "structured_report": {
        "instruction_prefix": "generate structured report:",
        "purpose": (
            "Generate a cautious preliminary report from the supplied "
            "computer-vision findings."
        ),
        "required_sections": [
            "PRELIMINARY MODEL REPORT",
            "MODEL FINDINGS",
            "LIMITATIONS",
        ],
        "task_rules": [
            "Mention only findings supplied in the structured input.",
            "Preserve probability, threshold, and uncertainty relationships.",
            "Do not present a model finding as a confirmed diagnosis.",
            "Include the educational-use limitation.",
        ],
    },
    "plain_language_explanation": {
        "instruction_prefix": "explain in simple language:",
        "purpose": (
            "Explain the supplied model output using concise and accessible "
            "non-technical language."
        ),
        "required_sections": [
            "EXPLANATION",
            "LIMITATIONS",
        ],
        "task_rules": [
            "Explain only supplied findings and approved descriptions.",
            "Avoid unsupported symptoms, measurements, or clinical conclusions.",
            "Preserve uncertainty and the no-target-finding boundary.",
            "Include the educational-use limitation.",
        ],
    },
    "grounded_question_answering": {
        "instruction_prefix": "answer grounded question:",
        "purpose": (
            "Answer a user question only when the supplied model context "
            "supports the answer."
        ),
        "required_sections": [
            "ANSWER",
            "LIMITATIONS",
        ],
        "task_rules": [
            "Use only the supplied findings, confidence data, and descriptions.",
            "State when the question cannot be answered from the supplied context.",
            "Do not infer symptoms, causes, prognosis, or treatment.",
            "Include the educational-use limitation.",
        ],
    },
    "educational_follow_up": {
        "instruction_prefix": "generate educational follow-up:",
        "purpose": (
            "Provide controlled educational next-step guidance without "
            "prescribing treatment or medication."
        ),
        "required_sections": [
            "EDUCATIONAL FOLLOW-UP",
            "LIMITATIONS",
        ],
        "task_rules": [
            "Use only controlled professional-review wording.",
            "Do not provide treatment, medication, or emergency instructions.",
            "Do not claim that model output confirms a medical condition.",
            "Include the educational-use limitation.",
        ],
    },
}

prompt_registry = {
    "registry_version": PROMPT_REGISTRY_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "base_language_model": BASE_LANGUAGE_MODEL,
    "language_model_version": LANGUAGE_MODEL_VERSION,
    "computer_vision_model_version": "resnet18-chestmnist-v1",
    "input_serialization": {
        "format": "ordered key-value text",
        "field_order": COMMON_INPUT_FIELDS,
        "missing_optional_value": "not provided",
    },
    "global_generation_rules": [
        "Mention only findings present in the supplied context.",
        "Do not independently inspect or interpret the image.",
        "Do not provide a definitive diagnosis.",
        "Do not invent symptoms, measurements, or medical history.",
        "Do not provide treatment or medication instructions.",
        "Do not claim that Grad-CAM confirms or localizes a lesion.",
        "Preserve confidence and threshold uncertainty.",
        "Use no target finding only for the defined 14-label state.",
        "Include the educational-use limitation.",
    ],
    "controlled_language": {
        "no_target_finding": NO_TARGET_FINDING_DESCRIPTION,
        "educational_use_limitation": EDUCATIONAL_USE_LIMITATION,
        "gradcam_limitation": GRADCAM_LIMITATION,
        "professional_review_guidance": PROFESSIONAL_REVIEW_GUIDANCE,
    },
    "tasks": TASK_REGISTRY,
}


# -------------------------------------------------------------------------
# Validate the task registry before saving it
# -------------------------------------------------------------------------
expected_task_names = {
    "structured_report",
    "plain_language_explanation",
    "grounded_question_answering",
    "educational_follow_up",
}

instruction_prefixes = [
    task_config["instruction_prefix"]
    for task_config in TASK_REGISTRY.values()
]

registry_checks = {
    "Four language tasks registered": len(TASK_REGISTRY) == 4,
    "Expected task names preserved": (
        set(TASK_REGISTRY) == expected_task_names
    ),
    "Instruction prefixes are unique": (
        len(instruction_prefixes) == len(set(instruction_prefixes))
    ),
    "Every prefix ends with a colon": all(
        prefix.endswith(":") for prefix in instruction_prefixes
    ),
    "Every task has required sections": all(
        bool(task_config["required_sections"])
        for task_config in TASK_REGISTRY.values()
    ),
    "Every task has grounding rules": all(
        bool(task_config["task_rules"])
        for task_config in TASK_REGISTRY.values()
    ),
    "Input field order is unique": (
        len(COMMON_INPUT_FIELDS) == len(set(COMMON_INPUT_FIELDS))
    ),
    "Global safety rules are defined": bool(
        prompt_registry["global_generation_rules"]
    ),
}

if not all(registry_checks.values()):
    failed_checks = [
        name for name, passed in registry_checks.items() if not passed
    ]
    raise RuntimeError(
        "Prompt registry validation failed: " + ", ".join(failed_checks)
    )


# -------------------------------------------------------------------------
# Persist the approved registry
# -------------------------------------------------------------------------
PROMPT_REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)

with PROMPT_REGISTRY_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        prompt_registry,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Report registry status
# -------------------------------------------------------------------------
print("VERSIONED LANGUAGE TASK REGISTRY")
print("-" * 100)
print(f"Registry version          : {PROMPT_REGISTRY_VERSION}")
print(f"Registry path             : {PROMPT_REGISTRY_PATH}")
print(f"Language tasks            : {len(TASK_REGISTRY)}")
print(f"Common input fields       : {len(COMMON_INPUT_FIELDS)}")
print(f"Global generation rules   : {len(prompt_registry['global_generation_rules'])}")
print("-" * 100)

for task_name, task_config in TASK_REGISTRY.items():
    print(f"{task_name:<30}: {task_config['instruction_prefix']}")

print("-" * 100)

for check_name, passed in registry_checks.items():
    print(f"{check_name:<42}: {'PASS' if passed else 'FAIL'}")

print("-" * 100)
print("STATUS: VERSIONED LANGUAGE TASK CONTRACT REGISTERED")

VERSIONED LANGUAGE TASK REGISTRY
----------------------------------------------------------------------------------------------------
Registry version          : grounded-language-prompts-v1
Registry path             : /home/jovyan/chest-xray-ai-assistant/configs/prompt_registry.yaml
Language tasks            : 4
Common input fields       : 11
Global generation rules   : 9
----------------------------------------------------------------------------------------------------
structured_report             : generate structured report:
plain_language_explanation    : explain in simple language:
grounded_question_answering   : answer grounded question:
educational_follow_up         : generate educational follow-up:
----------------------------------------------------------------------------------------------------
Four language tasks registered            : PASS
Expected task names preserved             : PASS
Instruction prefixes are unique           : PASS
Every prefix ends with a colon   

## 4. Leakage-Resistant Language Dataset Design

The grounded language dataset is constructed independently of the isolated ChestMNIST imaging-test predictions. It uses the approved 14-label ontology, training-partition prevalence, controlled threshold relationships, confidence categories, and safe response templates.

All four language tasks receive equal representation. Five controlled scenario profiles are included within every partition, while template families remain mutually exclusive across training, validation, and held-out language-test data. The held-out language-test partition created here is a synthetic language-evaluation partition and is separate from the isolated ChestMNIST image-test partition used previously.


In [4]:
# -------------------------------------------------------------------------
# Define the deterministic dataset design
# -------------------------------------------------------------------------
LANGUAGE_DATASET_VERSION = "chestmnist-grounded-language-v1"
LANGUAGE_DATASET_SEED = 42

TASK_NAMES = list(TASK_REGISTRY.keys())

SCENARIO_PROFILES = {
    "no_target_finding": {
        "description": (
            "All supplied finding probabilities remain below their frozen "
            "thresholds."
        ),
        "expected_state": "no_target_finding",
    },
    "single_below_threshold": {
        "description": (
            "One finding is supplied as a candidate but remains below its "
            "frozen threshold."
        ),
        "expected_state": "below_threshold",
    },
    "single_above_threshold": {
        "description": (
            "One supplied finding crosses its frozen threshold."
        ),
        "expected_state": "one_target_finding",
    },
    "multiple_above_threshold": {
        "description": (
            "Two or three supplied findings cross their frozen thresholds "
            "with controlled confidence variation."
        ),
        "expected_state": "multiple_target_findings",
    },
    "mixed_threshold_relationship": {
        "description": (
            "At least one supplied finding crosses its threshold while "
            "another remains below threshold."
        ),
        "expected_state": "mixed_threshold_state",
    },
}

CONFIDENCE_CATEGORIES = {
    "below_threshold": {
        "relationship": "probability < threshold",
        "description": "The model probability remains below the decision threshold.",
    },
    "borderline": {
        "relationship": "0.00 <= probability - threshold < 0.05",
        "description": "The probability is only slightly above the threshold.",
    },
    "moderate": {
        "relationship": "0.05 <= probability - threshold < 0.15",
        "description": "The probability is moderately above the threshold.",
    },
    "higher": {
        "relationship": "probability - threshold >= 0.15",
        "description": "The probability is more clearly above the threshold.",
    },
}

SPLIT_DESIGN = {
    "train": {
        "records_per_task_scenario": 180,
        "template_families": [
            "train_direct_v1",
            "train_context_first_v1",
            "train_compact_v1",
            "train_explanatory_v1",
            "train_threshold_first_v1",
            "train_model_first_v1",
        ],
    },
    "validation": {
        "records_per_task_scenario": 30,
        "template_families": [
            "validation_summary_first_v1",
            "validation_evidence_first_v1",
        ],
    },
    "language_test": {
        "records_per_task_scenario": 30,
        "template_families": [
            "test_question_first_v1",
            "test_decision_first_v1",
        ],
    },
}

dataset_design = {
    "dataset_version": LANGUAGE_DATASET_VERSION,
    "random_seed": LANGUAGE_DATASET_SEED,
    "source_policy": {
        "allowed_sources": [
            "ChestMNIST 14-label ontology",
            "ChestMNIST training-partition prevalence",
            "approved finding descriptions",
            "frozen computer-vision model contract",
            "controlled probability and threshold scenarios",
            "versioned safe response templates",
        ],
        "prohibited_sources": [
            "Notebook 4 isolated-test prediction records",
            "ChestMNIST isolated-test targets",
            "ChestMNIST isolated-test probabilities",
            "Notebook 5 representative test-case predictions",
        ],
    },
    "tasks": TASK_NAMES,
    "scenario_profiles": SCENARIO_PROFILES,
    "confidence_categories": CONFIDENCE_CATEGORIES,
    "splits": SPLIT_DESIGN,
}


# -------------------------------------------------------------------------
# Calculate expected record counts
# -------------------------------------------------------------------------
split_counts = {}

for split_name, split_config in SPLIT_DESIGN.items():
    records_per_combination = split_config["records_per_task_scenario"]
    split_counts[split_name] = (
        len(TASK_NAMES)
        * len(SCENARIO_PROFILES)
        * records_per_combination
    )

total_expected_records = sum(split_counts.values())

task_count_per_split = {
    split_name: (
        len(SCENARIO_PROFILES)
        * split_config["records_per_task_scenario"]
    )
    for split_name, split_config in SPLIT_DESIGN.items()
}


# -------------------------------------------------------------------------
# Validate template separation and dataset balance
# -------------------------------------------------------------------------
template_family_sets = {
    split_name: set(split_config["template_families"])
    for split_name, split_config in SPLIT_DESIGN.items()
}

template_overlap = (
    template_family_sets["train"]
    & template_family_sets["validation"]
) | (
    template_family_sets["train"]
    & template_family_sets["language_test"]
) | (
    template_family_sets["validation"]
    & template_family_sets["language_test"]
)

dataset_design_checks = {
    "Deterministic seed is 42": LANGUAGE_DATASET_SEED == 42,
    "Four tasks are represented": len(TASK_NAMES) == 4,
    "Five scenario profiles are represented": (
        len(SCENARIO_PROFILES) == 5
    ),
    "All splits cover every task": all(
        task_count > 0 for task_count in task_count_per_split.values()
    ),
    "Template families are split-exclusive": len(template_overlap) == 0,
    "Training records equal 3600": split_counts["train"] == 3600,
    "Validation records equal 600": (
        split_counts["validation"] == 600
    ),
    "Held-out language-test records equal 600": (
        split_counts["language_test"] == 600
    ),
    "Total records equal 4800": total_expected_records == 4800,
    "Isolated image-test artifacts are prohibited": any(
        "isolated-test" in source.lower()
        for source in dataset_design["source_policy"]["prohibited_sources"]
    ),
}


# -------------------------------------------------------------------------
# Persist the dataset-design contract
# -------------------------------------------------------------------------
LANGUAGE_DATASET_DESIGN_PATH = (
    LANGUAGE_DATA_DIR / "language_dataset_design.yaml"
)

with LANGUAGE_DATASET_DESIGN_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        dataset_design,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Report the dataset design
# -------------------------------------------------------------------------
print("GROUNDED LANGUAGE DATASET DESIGN")
print("-" * 100)
print(f"Dataset version             : {LANGUAGE_DATASET_VERSION}")
print(f"Random seed                 : {LANGUAGE_DATASET_SEED}")
print(f"Registered tasks            : {len(TASK_NAMES)}")
print(f"Scenario profiles           : {len(SCENARIO_PROFILES)}")
print(f"Confidence categories       : {len(CONFIDENCE_CATEGORIES)}")
print(f"Dataset design path         : {LANGUAGE_DATASET_DESIGN_PATH}")
print("-" * 100)

for split_name in ("train", "validation", "language_test"):
    print(
        f"{split_name:<28}: "
        f"{split_counts[split_name]:>4} records | "
        f"{task_count_per_split[split_name]:>3} per task | "
        f"{len(SPLIT_DESIGN[split_name]['template_families'])} template families"
    )

print("-" * 100)
print(f"Total expected records      : {total_expected_records}")
print("-" * 100)

for check_name, passed in dataset_design_checks.items():
    print(f"{check_name:<48}: {'PASS' if passed else 'FAIL'}")

if not all(dataset_design_checks.values()):
    failed_checks = [
        name for name, passed in dataset_design_checks.items() if not passed
    ]
    raise RuntimeError(
        "Language dataset design validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: LEAKAGE-RESISTANT LANGUAGE DATASET DESIGN APPROVED")

GROUNDED LANGUAGE DATASET DESIGN
----------------------------------------------------------------------------------------------------
Dataset version             : chestmnist-grounded-language-v1
Random seed                 : 42
Registered tasks            : 4
Scenario profiles           : 5
Confidence categories       : 4
Dataset design path         : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed/grounded_language/language_dataset_design.yaml
----------------------------------------------------------------------------------------------------
train                       : 3600 records | 900 per task | 6 template families
validation                  :  600 records | 150 per task | 2 template families
language_test               :  600 records | 150 per task | 2 template families
----------------------------------------------------------------------------------------------------
Total expected records      : 4800
--------------------------------------------------

## 5. Frozen Computer-Vision Grounding Contract

Language records must use the actual per-label decision thresholds frozen during computer-vision model development. This section retrieves those thresholds from the versioned ResNet-18 metadata rather than recreating or recalibrating them.

The resulting mapping connects each approved finding to its frozen threshold and becomes the only threshold source used for controlled language-scenario generation. No isolated image-test predictions are accessed.


In [5]:
import numbers

import numpy as np


# -------------------------------------------------------------------------
# Load the frozen computer-vision model metadata
# -------------------------------------------------------------------------
COMPUTER_VISION_MODEL_VERSION = "resnet18-chestmnist-v1"
COMPUTER_VISION_MODEL_DIR = (
    DATA_ROOT / "models" / COMPUTER_VISION_MODEL_VERSION
)
COMPUTER_VISION_METADATA_PATH = (
    COMPUTER_VISION_MODEL_DIR / "model_metadata.yaml"
)

if not COMPUTER_VISION_METADATA_PATH.is_file():
    raise FileNotFoundError(
        "Computer-vision model metadata was not found: "
        f"{COMPUTER_VISION_METADATA_PATH}"
    )

with COMPUTER_VISION_METADATA_PATH.open("r", encoding="utf-8") as file:
    computer_vision_metadata = yaml.safe_load(file)

if not isinstance(computer_vision_metadata, dict):
    raise ValueError(
        "The computer-vision metadata must contain a YAML mapping."
    )


# -------------------------------------------------------------------------
# Locate a 14-value threshold vector without assuming one YAML layout
# -------------------------------------------------------------------------
label_names = ontology_df["label_name"].tolist()


def extract_numeric_vector(value):
    """Return a 14-value numeric vector when one is present."""
    if isinstance(value, (list, tuple)) and len(value) == 14:
        if all(isinstance(item, numbers.Real) for item in value):
            return np.asarray(value, dtype=np.float64)

    if isinstance(value, dict):
        if all(
            label_name in value
            and isinstance(value[label_name], numbers.Real)
            for label_name in label_names
        ):
            return np.asarray(
                [value[label_name] for label_name in label_names],
                dtype=np.float64,
            )

        string_indices = [str(index) for index in range(14)]
        if all(
            index in value and isinstance(value[index], numbers.Real)
            for index in string_indices
        ):
            return np.asarray(
                [value[index] for index in string_indices],
                dtype=np.float64,
            )

    return None


def find_numeric_vectors(value, path):
    """Search inside a threshold-related value for valid vectors."""
    vectors = []

    direct_vector = extract_numeric_vector(value)
    if direct_vector is not None:
        vectors.append((path, direct_vector))

    if isinstance(value, dict):
        for key, nested_value in value.items():
            vectors.extend(
                find_numeric_vectors(
                    nested_value,
                    f"{path}.{key}",
                )
            )

    return vectors


def collect_threshold_candidates(value, path="root"):
    """Collect threshold vectors from threshold-related metadata fields."""
    candidates = []

    if isinstance(value, dict):
        for key, nested_value in value.items():
            current_path = f"{path}.{key}"

            if "threshold" in str(key).lower():
                candidates.extend(
                    find_numeric_vectors(nested_value, current_path)
                )

            candidates.extend(
                collect_threshold_candidates(
                    nested_value,
                    current_path,
                )
            )

    return candidates


threshold_candidates = collect_threshold_candidates(
    computer_vision_metadata
)

# Remove duplicate path-and-value candidates.
unique_candidates = []
seen_candidates = set()

for candidate_path, candidate_values in threshold_candidates:
    candidate_key = (
        candidate_path,
        tuple(np.round(candidate_values, 10)),
    )
    if candidate_key not in seen_candidates:
        seen_candidates.add(candidate_key)
        unique_candidates.append(
            (candidate_path, candidate_values)
        )

if not unique_candidates:
    raise KeyError(
        "No 14-value per-label threshold vector was found in "
        f"{COMPUTER_VISION_METADATA_PATH}. "
        f"Top-level keys: {list(computer_vision_metadata.keys())}"
    )


# Prefer paths explicitly describing frozen, calibrated, or per-label values.
def threshold_candidate_priority(candidate):
    candidate_path = candidate[0].lower()
    priority_terms = {
        "frozen": 4,
        "calibrated": 3,
        "per_label": 2,
        "decision": 2,
        "thresholds": 1,
    }
    return sum(
        score
        for term, score in priority_terms.items()
        if term in candidate_path
    )


unique_candidates.sort(
    key=threshold_candidate_priority,
    reverse=True,
)

selected_threshold_path, frozen_threshold_values = (
    unique_candidates[0]
)


# -------------------------------------------------------------------------
# Validate against the previously frozen threshold summary
# -------------------------------------------------------------------------
documented_threshold_summary = np.asarray(
    [
        0.6797,
        0.8438,
        0.6758,
        0.5977,
        0.8672,
        0.8125,
        0.7695,
        0.8281,
        0.7461,
        0.8789,
        0.8750,
        0.8594,
        0.8594,
        0.6992,
    ],
    dtype=np.float64,
)

threshold_checks = {
    "Metadata file is available": (
        COMPUTER_VISION_METADATA_PATH.is_file()
    ),
    "Threshold vector contains 14 values": (
        frozen_threshold_values.shape == (14,)
    ),
    "All thresholds are finite": (
        np.isfinite(frozen_threshold_values).all()
    ),
    "All thresholds are within zero and one": (
        np.logical_and(
            frozen_threshold_values >= 0.0,
            frozen_threshold_values <= 1.0,
        ).all()
    ),
    "Thresholds match frozen evaluation summary": (
        np.allclose(
            frozen_threshold_values,
            documented_threshold_summary,
            atol=0.001,
            rtol=0.0,
        )
    ),
}

if not all(threshold_checks.values()):
    failed_checks = [
        name for name, passed in threshold_checks.items() if not passed
    ]
    raise RuntimeError(
        "Frozen threshold validation failed: "
        + ", ".join(failed_checks)
    )


# -------------------------------------------------------------------------
# Bind thresholds to the approved ontology
# -------------------------------------------------------------------------
FROZEN_THRESHOLDS = {
    label_name: float(threshold)
    for label_name, threshold in zip(
        label_names,
        frozen_threshold_values,
    )
}

grounding_contract_df = ontology_df.copy()
grounding_contract_df["frozen_threshold"] = (
    grounding_contract_df["label_name"].map(FROZEN_THRESHOLDS)
)


# -------------------------------------------------------------------------
# Report the frozen grounding contract
# -------------------------------------------------------------------------
print("FROZEN COMPUTER-VISION GROUNDING CONTRACT")
print("-" * 100)
print(f"Model version             : {COMPUTER_VISION_MODEL_VERSION}")
print(f"Metadata path             : {COMPUTER_VISION_METADATA_PATH}")
print(f"Threshold candidates found: {len(unique_candidates)}")
print(f"Selected threshold field  : {selected_threshold_path}")
print("-" * 100)

for check_name, passed in threshold_checks.items():
    print(f"{check_name:<48}: {'PASS' if passed else 'FAIL'}")

print("-" * 100)
print(
    grounding_contract_df[
        [
            "label_id",
            "label_name",
            "training_prevalence",
            "frozen_threshold",
        ]
    ].to_string(
        index=False,
        formatters={
            "training_prevalence": "{:.4f}".format,
            "frozen_threshold": "{:.4f}".format,
        },
    )
)

print("-" * 100)
print("STATUS: FROZEN THRESHOLD CONTRACT READY FOR LANGUAGE DATA GENERATION")

FROZEN COMPUTER-VISION GROUNDING CONTRACT
----------------------------------------------------------------------------------------------------
Model version             : resnet18-chestmnist-v1
Metadata path             : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/models/resnet18-chestmnist-v1/model_metadata.yaml
Threshold candidates found: 1
Selected threshold field  : root.output_contract.thresholds
----------------------------------------------------------------------------------------------------
Metadata file is available                      : PASS
Threshold vector contains 14 values             : PASS
All thresholds are finite                       : PASS
All thresholds are within zero and one          : PASS
Thresholds match frozen evaluation summary      : PASS
----------------------------------------------------------------------------------------------------
 label_id    label_name training_prevalence frozen_threshold
        0   atelectasis              0.

## 6. Controlled Confidence and Scenario Generation

The frozen thresholds differ considerably across findings, with several thresholds above 0.85. Confidence bands are therefore finalized using achievable probability margins relative to each finding’s own threshold.

This section implements deterministic utilities for the five approved scenario profiles. Each generated scenario preserves the probability, threshold, threshold decision, confidence category, approved description, model version, and no-target-finding state required for grounded language generation.


In [6]:
# -------------------------------------------------------------------------
# Finalize achievable threshold-relative confidence bands
# -------------------------------------------------------------------------
CONFIDENCE_CATEGORIES = {
    "below_threshold": {
        "relationship": "probability < threshold",
        "description": (
            "The supplied probability remains below the frozen threshold."
        ),
    },
    "borderline": {
        "relationship": "0.00 < probability - threshold < 0.03",
        "description": (
            "The supplied probability is only slightly above the frozen threshold."
        ),
    },
    "moderate": {
        "relationship": "0.03 <= probability - threshold < 0.10",
        "description": (
            "The supplied probability is moderately above the frozen threshold."
        ),
    },
    "higher": {
        "relationship": "probability - threshold >= 0.10",
        "description": (
            "The supplied probability is more clearly above the frozen threshold."
        ),
    },
}

dataset_design["confidence_categories"] = CONFIDENCE_CATEGORIES

with LANGUAGE_DATASET_DESIGN_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        dataset_design,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Create reusable ontology lookups
# -------------------------------------------------------------------------
ONTOLOGY_BY_NAME = {
    record["label_name"]: record
    for record in FINDING_ONTOLOGY
}


def assign_confidence_category(probability, threshold):
    """Assign a deterministic category from the threshold relationship."""
    margin = probability - threshold

    if margin < 0.0:
        return "below_threshold"
    if margin < 0.03:
        return "borderline"
    if margin < 0.10:
        return "moderate"
    return "higher"


def sample_probability(threshold, crossed_threshold, rng):
    """Sample a controlled probability relative to a frozen threshold."""
    if not crossed_threshold:
        lower_bound = max(0.01, threshold - 0.18)
        upper_bound = max(lower_bound + 0.001, threshold - 0.01)
        probability = rng.uniform(lower_bound, upper_bound)
    else:
        confidence_category = rng.choice(
            ["borderline", "moderate", "higher"]
        )

        margin_ranges = {
            "borderline": (0.005, 0.025),
            "moderate": (0.040, 0.090),
            "higher": (0.105, 0.150),
        }

        lower_margin, upper_margin = margin_ranges[confidence_category]
        maximum_margin = 0.995 - threshold
        upper_margin = min(upper_margin, maximum_margin)

        if upper_margin < lower_margin:
            raise ValueError(
                f"Threshold {threshold:.4f} cannot support the requested "
                f"{confidence_category} confidence band."
            )

        probability = threshold + rng.uniform(
            lower_margin,
            upper_margin,
        )

    return round(float(probability), 4)


def create_finding_evidence(label_name, crossed_threshold, rng):
    """Create one grounded finding-evidence object."""
    ontology_record = ONTOLOGY_BY_NAME[label_name]
    threshold = FROZEN_THRESHOLDS[label_name]
    probability = sample_probability(
        threshold=threshold,
        crossed_threshold=crossed_threshold,
        rng=rng,
    )

    actual_decision = probability >= threshold

    if actual_decision != crossed_threshold:
        raise RuntimeError(
            f"Rounded probability changed the intended threshold decision "
            f"for {label_name}."
        )

    return {
        "label_id": int(ontology_record["label_id"]),
        "label_name": label_name,
        "display_name": ontology_record["display_name"],
        "probability": probability,
        "frozen_threshold": round(float(threshold), 4),
        "crossed_threshold": bool(actual_decision),
        "confidence_category": assign_confidence_category(
            probability,
            threshold,
        ),
        "training_prevalence": float(
            ontology_record["training_prevalence"]
        ),
        "approved_description": ontology_record[
            "approved_description"
        ],
    }


def build_controlled_scenario(
    scenario_profile,
    rng,
    preferred_labels=None,
):
    """Build one scenario that follows an approved threshold profile."""
    if scenario_profile not in SCENARIO_PROFILES:
        raise KeyError(
            f"Unknown scenario profile: {scenario_profile}"
        )

    available_labels = label_names.copy()

    if preferred_labels is None:
        selected_labels = []
    else:
        selected_labels = list(dict.fromkeys(preferred_labels))

    required_label_counts = {
        "no_target_finding": 2,
        "single_below_threshold": 1,
        "single_above_threshold": 1,
        "multiple_above_threshold": int(rng.integers(2, 4)),
        "mixed_threshold_relationship": int(rng.integers(2, 4)),
    }

    required_count = required_label_counts[scenario_profile]

    remaining_labels = [
        label_name
        for label_name in available_labels
        if label_name not in selected_labels
    ]

    if len(selected_labels) < required_count:
        additional_labels = rng.choice(
            remaining_labels,
            size=required_count - len(selected_labels),
            replace=False,
        ).tolist()
        selected_labels.extend(additional_labels)

    selected_labels = selected_labels[:required_count]

    if scenario_profile in {
        "no_target_finding",
        "single_below_threshold",
    }:
        intended_decisions = [False] * required_count
    elif scenario_profile in {
        "single_above_threshold",
        "multiple_above_threshold",
    }:
        intended_decisions = [True] * required_count
    else:
        intended_decisions = [True] + [False] * (required_count - 1)
        rng.shuffle(intended_decisions)

    findings = [
        create_finding_evidence(
            label_name=label_name,
            crossed_threshold=decision,
            rng=rng,
        )
        for label_name, decision in zip(
            selected_labels,
            intended_decisions,
        )
    ]

    crossed_count = sum(
        finding["crossed_threshold"]
        for finding in findings
    )

    return {
        "scenario_profile": scenario_profile,
        "findings": findings,
        "no_target_finding": crossed_count == 0,
        "crossed_finding_count": int(crossed_count),
        "computer_vision_model_version": (
            COMPUTER_VISION_MODEL_VERSION
        ),
        "limitation_boundary": EDUCATIONAL_USE_LIMITATION,
    }


# -------------------------------------------------------------------------
# Run one deterministic smoke test for every scenario profile
# -------------------------------------------------------------------------
scenario_rng = np.random.default_rng(LANGUAGE_DATASET_SEED)

scenario_smoke_tests = {
    scenario_name: build_controlled_scenario(
        scenario_profile=scenario_name,
        rng=scenario_rng,
    )
    for scenario_name in SCENARIO_PROFILES
}

scenario_checks = {
    "Five smoke-test scenarios created": (
        len(scenario_smoke_tests) == 5
    ),
    "No-target scenario has no crossed findings": (
        scenario_smoke_tests["no_target_finding"][
            "crossed_finding_count"
        ] == 0
    ),
    "Single-below scenario remains below": (
        scenario_smoke_tests["single_below_threshold"][
            "crossed_finding_count"
        ] == 0
    ),
    "Single-above scenario has one finding": (
        scenario_smoke_tests["single_above_threshold"][
            "crossed_finding_count"
        ] == 1
    ),
    "Multiple-above scenario has multiple findings": (
        scenario_smoke_tests["multiple_above_threshold"][
            "crossed_finding_count"
        ] >= 2
    ),
    "Mixed scenario contains both decisions": (
        0
        < scenario_smoke_tests["mixed_threshold_relationship"][
            "crossed_finding_count"
        ]
        < len(
            scenario_smoke_tests["mixed_threshold_relationship"][
                "findings"
            ]
        )
    ),
    "All generated probabilities are valid": all(
        0.0 <= finding["probability"] <= 1.0
        for scenario in scenario_smoke_tests.values()
        for finding in scenario["findings"]
    ),
}


# -------------------------------------------------------------------------
# Report the controlled scenario smoke test
# -------------------------------------------------------------------------
print("CONTROLLED LANGUAGE SCENARIO GENERATION")
print("-" * 100)
print(f"Deterministic seed        : {LANGUAGE_DATASET_SEED}")
print(f"Scenario profiles         : {len(SCENARIO_PROFILES)}")
print(f"Confidence categories     : {len(CONFIDENCE_CATEGORIES)}")
print(f"Dataset design updated    : {LANGUAGE_DATASET_DESIGN_PATH}")
print("-" * 100)

for scenario_name, scenario in scenario_smoke_tests.items():
    decision_summary = ", ".join(
        (
            f"{finding['label_name']}="
            f"{finding['probability']:.4f}/"
            f"{finding['frozen_threshold']:.4f} "
            f"({finding['confidence_category']})"
        )
        for finding in scenario["findings"]
    )

    print(
        f"{scenario_name:<30}: "
        f"crossed={scenario['crossed_finding_count']} | "
        f"{decision_summary}"
    )

print("-" * 100)

for check_name, passed in scenario_checks.items():
    print(f"{check_name:<52}: {'PASS' if passed else 'FAIL'}")

if not all(scenario_checks.values()):
    failed_checks = [
        name for name, passed in scenario_checks.items() if not passed
    ]
    raise RuntimeError(
        "Controlled scenario generation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: CONTROLLED THRESHOLD SCENARIOS READY")

CONTROLLED LANGUAGE SCENARIO GENERATION
----------------------------------------------------------------------------------------------------
Deterministic seed        : 42
Scenario profiles         : 5
Confidence categories     : 4
Dataset design updated    : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed/grounded_language/language_dataset_design.yaml
----------------------------------------------------------------------------------------------------
no_target_finding             : crossed=0 | pneumonia=0.7081/0.7695 (below_threshold), consolidation=0.5821/0.7461 (below_threshold)
single_below_threshold        : crossed=0 | hernia=0.6486/0.6992 (below_threshold)
single_above_threshold        : crossed=1 | pneumothorax=0.8421/0.8281 (borderline)
multiple_above_threshold      : crossed=3 | emphysema=0.9867/0.8750 (higher), effusion=0.7435/0.6758 (moderate), pleural=0.8657/0.8594 (borderline)
mixed_threshold_relationship  : crossed=1 | infiltration=0.7428/0.5977 (h

## 7. Structured Grounding Input Serialization

Each fine-tuning example requires a consistent textual representation that FLAN-T5 can process while retaining the complete grounding contract. The instruction prefix identifies the requested task, and the remaining fields describe only the supplied model evidence.

Finding names, probabilities, thresholds, decisions, confidence categories, approved descriptions, model version, limitation boundary, and any applicable user question are serialized in a fixed order. This format will also be reused by the later FastAPI language service to preserve training–inference consistency.


In [7]:
# -------------------------------------------------------------------------
# Define controlled questions for serialization validation
# -------------------------------------------------------------------------
CONTROLLED_QUESTION_EXAMPLES = {
    "output_summary": (
        "What does the supplied model information indicate?"
    ),
    "threshold_meaning": (
        "Did the supplied finding cross its decision threshold?"
    ),
    "confidence_meaning": (
        "How should the supplied confidence category be understood?"
    ),
    "gradcam_boundary": (
        "Does a Grad-CAM highlighted region confirm a lesion?"
    ),
}


def serialize_name_value_pairs(findings, value_key, formatter=None):
    """Serialize one finding attribute in the registered finding order."""
    values = []

    for finding in findings:
        value = finding[value_key]

        if formatter is not None:
            value = formatter(value)

        values.append(f"{finding['label_name']}={value}")

    return " | ".join(values)


def serialize_descriptions(findings):
    """Serialize approved descriptions without adding external content."""
    return " || ".join(
        (
            f"{finding['label_name']}: "
            f"{finding['approved_description']}"
        )
        for finding in findings
    )


def serialize_grounded_input(
    task_name,
    scenario,
    user_question=None,
):
    """Convert one controlled scenario into the model input format."""
    if task_name not in TASK_REGISTRY:
        raise KeyError(f"Unknown language task: {task_name}")

    findings = scenario["findings"]
    instruction_prefix = TASK_REGISTRY[task_name][
        "instruction_prefix"
    ]

    serialized_fields = {
        "task_type": task_name,
        "finding_names": " | ".join(
            finding["label_name"] for finding in findings
        ),
        "probabilities": serialize_name_value_pairs(
            findings,
            "probability",
            formatter=lambda value: f"{value:.4f}",
        ),
        "frozen_thresholds": serialize_name_value_pairs(
            findings,
            "frozen_threshold",
            formatter=lambda value: f"{value:.4f}",
        ),
        "threshold_decisions": serialize_name_value_pairs(
            findings,
            "crossed_threshold",
            formatter=lambda value: (
                "crossed" if value else "not_crossed"
            ),
        ),
        "no_target_finding": str(
            scenario["no_target_finding"]
        ).lower(),
        "confidence_categories": serialize_name_value_pairs(
            findings,
            "confidence_category",
        ),
        "model_version": scenario[
            "computer_vision_model_version"
        ],
        "approved_descriptions": serialize_descriptions(findings),
        "limitation_boundary": scenario["limitation_boundary"],
        "user_question": (
            user_question.strip()
            if user_question and user_question.strip()
            else "not provided"
        ),
    }

    missing_fields = [
        field_name
        for field_name in COMMON_INPUT_FIELDS
        if field_name not in serialized_fields
    ]

    if missing_fields:
        raise RuntimeError(
            "The serialized input is missing registered fields: "
            + ", ".join(missing_fields)
        )

    input_lines = [instruction_prefix]

    input_lines.extend(
        f"{field_name}: {serialized_fields[field_name]}"
        for field_name in COMMON_INPUT_FIELDS
    )

    return "\n".join(input_lines)


# -------------------------------------------------------------------------
# Create one serialized smoke test for every task
# -------------------------------------------------------------------------
serialization_scenarios = {
    "structured_report": scenario_smoke_tests[
        "multiple_above_threshold"
    ],
    "plain_language_explanation": scenario_smoke_tests[
        "single_above_threshold"
    ],
    "grounded_question_answering": scenario_smoke_tests[
        "mixed_threshold_relationship"
    ],
    "educational_follow_up": scenario_smoke_tests[
        "no_target_finding"
    ],
}

serialized_smoke_tests = {}

for task_name, scenario in serialization_scenarios.items():
    question = None

    if task_name == "grounded_question_answering":
        question = CONTROLLED_QUESTION_EXAMPLES[
            "threshold_meaning"
        ]

    serialized_smoke_tests[task_name] = (
        serialize_grounded_input(
            task_name=task_name,
            scenario=scenario,
            user_question=question,
        )
    )


# -------------------------------------------------------------------------
# Validate serialization completeness and prefix compliance
# -------------------------------------------------------------------------
serialization_checks = {
    "Four task inputs serialized": (
        len(serialized_smoke_tests) == 4
    ),
    "Every input starts with its task prefix": all(
        serialized_smoke_tests[task_name].startswith(
            TASK_REGISTRY[task_name]["instruction_prefix"]
        )
        for task_name in TASK_REGISTRY
    ),
    "Every input contains all registered fields": all(
        all(
            f"{field_name}:" in serialized_input
            for field_name in COMMON_INPUT_FIELDS
        )
        for serialized_input in serialized_smoke_tests.values()
    ),
    "Every input includes the model version": all(
        COMPUTER_VISION_MODEL_VERSION in serialized_input
        for serialized_input in serialized_smoke_tests.values()
    ),
    "Every input includes the limitation boundary": all(
        EDUCATIONAL_USE_LIMITATION in serialized_input
        for serialized_input in serialized_smoke_tests.values()
    ),
    "Question is included for grounded QA": (
        CONTROLLED_QUESTION_EXAMPLES["threshold_meaning"]
        in serialized_smoke_tests[
            "grounded_question_answering"
        ]
    ),
    "Non-QA tasks use the optional-value marker": all(
        "user_question: not provided"
        in serialized_smoke_tests[task_name]
        for task_name in TASK_REGISTRY
        if task_name != "grounded_question_answering"
    ),
}


# -------------------------------------------------------------------------
# Report serialization results
# -------------------------------------------------------------------------
print("STRUCTURED GROUNDING INPUT SERIALIZATION")
print("-" * 100)

for task_name, serialized_input in serialized_smoke_tests.items():
    print(
        f"{task_name:<30}: "
        f"{len(serialized_input.splitlines())} lines | "
        f"{len(serialized_input):>4} characters"
    )

print("-" * 100)

for check_name, passed in serialization_checks.items():
    print(f"{check_name:<52}: {'PASS' if passed else 'FAIL'}")

if not all(serialization_checks.values()):
    failed_checks = [
        name for name, passed in serialization_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Grounding input serialization failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("GROUNDED QA SERIALIZATION EXAMPLE")
print("-" * 100)
print(
    serialized_smoke_tests[
        "grounded_question_answering"
    ]
)
print("-" * 100)
print("STATUS: TRAINING–INFERENCE INPUT CONTRACT READY")

STRUCTURED GROUNDING INPUT SERIALIZATION
----------------------------------------------------------------------------------------------------
structured_report             : 12 lines |  975 characters
plain_language_explanation    : 12 lines |  644 characters
grounded_question_answering   : 12 lines | 1108 characters
educational_follow_up         : 12 lines |  886 characters
----------------------------------------------------------------------------------------------------
Four task inputs serialized                         : PASS
Every input starts with its task prefix             : PASS
Every input contains all registered fields          : PASS
Every input includes the model version              : PASS
Every input includes the limitation boundary        : PASS
Question is included for grounded QA                : PASS
Non-QA tasks use the optional-value marker          : PASS
----------------------------------------------------------------------------------------------------
GROUNDE

## 8. Controlled Target Response Construction

### 8.1 Preliminary Report and Plain-Language Targets

Fine-tuning targets must preserve the supplied evidence without converting model scores into clinical conclusions. This block defines split-specific language styles and constructs targets for preliminary report generation and plain-language explanation.

The required section headings remain stable for deterministic compliance evaluation, while wording styles are assigned through the split-exclusive template families approved earlier. Every finding statement retains its probability, frozen threshold, threshold decision, confidence category, and approved description.


In [9]:
import re


# -------------------------------------------------------------------------
# Define the split-exclusive response style families
# -------------------------------------------------------------------------
LANGUAGE_STYLE_FAMILIES = {
    "train_direct_v1": {
        "evidence_lead": "The supplied model output reports",
        "no_target_lead": "The supplied probabilities did not cross",
        "explanation_lead": "In simple terms, the model assigned",
    },
    "train_context_first_v1": {
        "evidence_lead": "Based on the provided model context",
        "no_target_lead": "Based on the provided model context, none crossed",
        "explanation_lead": "Using the provided model information, the result contains",
    },
    "train_compact_v1": {
        "evidence_lead": "The model evidence records",
        "no_target_lead": "The model evidence records no crossing of",
        "explanation_lead": "The supplied result gives",
    },
    "train_explanatory_v1": {
        "evidence_lead": "The structured model evidence indicates",
        "no_target_lead": "The structured evidence indicates no crossing of",
        "explanation_lead": "The result can be explained as",
    },
    "train_threshold_first_v1": {
        "evidence_lead": "Relative to the frozen thresholds, the output records",
        "no_target_lead": "Relative to the frozen thresholds, none crossed",
        "explanation_lead": "When compared with the frozen thresholds",
    },
    "train_model_first_v1": {
        "evidence_lead": "The versioned model output contains",
        "no_target_lead": "The versioned model output contains no crossing of",
        "explanation_lead": "The versioned model produced",
    },
    "validation_summary_first_v1": {
        "evidence_lead": "The supplied evidence summary records",
        "no_target_lead": "The evidence summary records no crossing of",
        "explanation_lead": "The evidence summary means",
    },
    "validation_evidence_first_v1": {
        "evidence_lead": "The available model evidence describes",
        "no_target_lead": "The available model evidence shows no crossing of",
        "explanation_lead": "From the available evidence",
    },
    "test_question_first_v1": {
        "evidence_lead": "For the requested interpretation, the model reports",
        "no_target_lead": "For the requested interpretation, none crossed",
        "explanation_lead": "For this explanation, the supplied result contains",
    },
    "test_decision_first_v1": {
        "evidence_lead": "The recorded threshold decisions identify",
        "no_target_lead": "The recorded threshold decisions identify no crossing of",
        "explanation_lead": "The recorded decision can be understood as",
    },
}


# -------------------------------------------------------------------------
# Validate style-family alignment with the split design
# -------------------------------------------------------------------------
registered_template_families = {
    family_name
    for split_config in SPLIT_DESIGN.values()
    for family_name in split_config["template_families"]
}

style_family_checks = {
    "All dataset families have response styles": (
        registered_template_families
        == set(LANGUAGE_STYLE_FAMILIES)
    ),
    "Ten response style families are registered": (
        len(LANGUAGE_STYLE_FAMILIES) == 10
    ),
    "Every family contains three style fields": all(
        set(style_config) == {
            "evidence_lead",
            "no_target_lead",
            "explanation_lead",
        }
        for style_config in LANGUAGE_STYLE_FAMILIES.values()
    ),
}


def render_finding_evidence(finding, simple_language=False):
    """Render one supplied finding without changing its evidence."""
    probability = finding["probability"]
    threshold = finding["frozen_threshold"]
    display_name = finding["display_name"]
    description = finding["approved_description"]
    category = finding["confidence_category"]

    if finding["crossed_threshold"]:
        relationship_text = (
            f"crossed its frozen threshold and was categorized as "
            f"{category.replace('_', ' ')}"
        )
    else:
        relationship_text = (
            "did not cross its frozen threshold"
        )

    if simple_language:
        return (
            f"For {display_name}, the model score was "
            f"{probability:.4f}, compared with a decision threshold of "
            f"{threshold:.4f}. The score {relationship_text}. "
            f"This label refers to {description[0].lower() + description[1:]}"
        )

    return (
        f"{display_name}: model probability {probability:.4f}; "
        f"frozen threshold {threshold:.4f}; the probability "
        f"{relationship_text}. {description}"
    )


def render_no_target_statement(scenario, style_config):
    """Render the controlled no-target-finding interpretation."""
    supplied_names = ", ".join(
        finding["display_name"]
        for finding in scenario["findings"]
    )

    return (
        f"{style_config['no_target_lead']} the frozen thresholds "
        f"for the supplied candidates: {supplied_names}. "
        f"{NO_TARGET_FINDING_DESCRIPTION}"
    )


def build_structured_report_target(scenario, template_family):
    """Build a structured preliminary-report fine-tuning target."""
    if template_family not in LANGUAGE_STYLE_FAMILIES:
        raise KeyError(
            f"Unknown template family: {template_family}"
        )

    style_config = LANGUAGE_STYLE_FAMILIES[template_family]

    if scenario["no_target_finding"]:
        findings_text = render_no_target_statement(
            scenario,
            style_config,
        )
    else:
        evidence_lines = [
            render_finding_evidence(finding)
            for finding in scenario["findings"]
        ]
        findings_text = (
            f"{style_config['evidence_lead']} the following "
            f"threshold relationships:\n"
            + "\n".join(
                f"- {evidence_line}"
                for evidence_line in evidence_lines
            )
        )

    return (
        "PRELIMINARY MODEL REPORT\n"
        "MODEL FINDINGS\n"
        f"{findings_text}\n"
        "LIMITATIONS\n"
        f"{EDUCATIONAL_USE_LIMITATION} "
        f"{PROFESSIONAL_REVIEW_GUIDANCE}"
    )


def build_plain_explanation_target(scenario, template_family):
    """Build a plain-language explanation fine-tuning target."""
    if template_family not in LANGUAGE_STYLE_FAMILIES:
        raise KeyError(
            f"Unknown template family: {template_family}"
        )

    style_config = LANGUAGE_STYLE_FAMILIES[template_family]

    if scenario["no_target_finding"]:
        explanation_text = render_no_target_statement(
            scenario,
            style_config,
        )
    else:
        finding_explanations = [
            render_finding_evidence(
                finding,
                simple_language=True,
            )
            for finding in scenario["findings"]
        ]
        explanation_text = (
            f"{style_config['explanation_lead']} the following "
            f"information. "
            + " ".join(finding_explanations)
        )

    return (
        "EXPLANATION\n"
        f"{explanation_text}\n"
        "LIMITATIONS\n"
        f"{EDUCATIONAL_USE_LIMITATION} "
        f"{PROFESSIONAL_REVIEW_GUIDANCE}"
    )


# -------------------------------------------------------------------------
# Validate grounding and safety in generated targets
# -------------------------------------------------------------------------
def contains_required_sections(target_text, required_sections):
    return all(
        section in target_text
        for section in required_sections
    )


def identify_unsupported_findings(target_text, scenario):
    supplied_names = {
        finding["label_name"]
        for finding in scenario["findings"]
    }

    mentioned_names = {
        label_name
        for label_name in label_names
        if re.search(
            rf"\b{re.escape(label_name)}\b",
            target_text,
            flags=re.IGNORECASE,
        )
    }

    return sorted(mentioned_names - supplied_names)


report_smoke_scenario = scenario_smoke_tests[
    "multiple_above_threshold"
]
explanation_smoke_scenario = scenario_smoke_tests[
    "mixed_threshold_relationship"
]

report_target_smoke_test = build_structured_report_target(
    scenario=report_smoke_scenario,
    template_family="train_direct_v1",
)

explanation_target_smoke_test = (
    build_plain_explanation_target(
        scenario=explanation_smoke_scenario,
        template_family="train_context_first_v1",
    )
)

target_checks = {
    **style_family_checks,
    "Report contains all required sections": (
        contains_required_sections(
            report_target_smoke_test,
            TASK_REGISTRY["structured_report"][
                "required_sections"
            ],
        )
    ),
    "Explanation contains all required sections": (
        contains_required_sections(
            explanation_target_smoke_test,
            TASK_REGISTRY["plain_language_explanation"][
                "required_sections"
            ],
        )
    ),
    "Report contains no unsupported findings": (
        not identify_unsupported_findings(
            report_target_smoke_test,
            report_smoke_scenario,
        )
    ),
    "Explanation contains no unsupported findings": (
        not identify_unsupported_findings(
            explanation_target_smoke_test,
            explanation_smoke_scenario,
        )
    ),
    "Both targets contain the educational limitation": all(
        EDUCATIONAL_USE_LIMITATION in target_text
        for target_text in (
            report_target_smoke_test,
            explanation_target_smoke_test,
        )
    ),
    "Targets avoid definitive-diagnosis wording": all(
        forbidden_phrase not in target_text.lower()
        for target_text in (
            report_target_smoke_test,
            explanation_target_smoke_test,
        )
        for forbidden_phrase in (
            "confirmed diagnosis",
            "definitely has",
            "proves that",
            "confirms a lesion",
        )
    ),
}


# -------------------------------------------------------------------------
# Report target-construction results
# -------------------------------------------------------------------------
print("CONTROLLED REPORT AND EXPLANATION TARGETS")
print("-" * 100)
print(f"Response style families   : {len(LANGUAGE_STYLE_FAMILIES)}")
print(f"Report target characters  : {len(report_target_smoke_test)}")
print(
    f"Explanation characters    : "
    f"{len(explanation_target_smoke_test)}"
)
print("-" * 100)

for check_name, passed in target_checks.items():
    print(f"{check_name:<52}: {'PASS' if passed else 'FAIL'}")

if not all(target_checks.values()):
    failed_checks = [
        name for name, passed in target_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Controlled target construction failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STRUCTURED REPORT TARGET EXAMPLE")
print("-" * 100)
print(report_target_smoke_test)
print("-" * 100)
print("PLAIN-LANGUAGE TARGET EXAMPLE")
print("-" * 100)
print(explanation_target_smoke_test)
print("-" * 100)
print("STATUS: REPORT AND EXPLANATION TARGET BUILDERS READY")

CONTROLLED REPORT AND EXPLANATION TARGETS
----------------------------------------------------------------------------------------------------
Response style families   : 10
Report target characters  : 1098
Explanation characters    : 1154
----------------------------------------------------------------------------------------------------
All dataset families have response styles           : PASS
Ten response style families are registered          : PASS
Every family contains three style fields            : PASS
Report contains all required sections               : PASS
Explanation contains all required sections          : PASS
Report contains no unsupported findings             : PASS
Explanation contains no unsupported findings        : PASS
Both targets contain the educational limitation     : PASS
Targets avoid definitive-diagnosis wording          : PASS
----------------------------------------------------------------------------------------------------
STRUCTURED REPORT TARGET EX

### 8.2 Grounded Question Answering and Educational Follow-Up

Grounded question answering must distinguish between questions supported by the supplied model context and questions that require unavailable clinical information. Controlled question intents cover model summaries, threshold decisions, confidence interpretation, the no-target-finding state, Grad-CAM limitations, diagnosis requests, and treatment requests.

Educational follow-up targets use only approved professional-review wording. They do not prescribe treatment, recommend medication, confirm a condition, or interpret Grad-CAM as lesion evidence.


In [10]:
# -------------------------------------------------------------------------
# Define controlled grounded-question intents
# -------------------------------------------------------------------------
GROUNDING_QUESTION_REGISTRY = {
    "output_summary": {
        "question": (
            "What does the supplied model information indicate?"
        ),
        "response_policy": (
            "Summarize only the supplied threshold relationships."
        ),
    },
    "threshold_meaning": {
        "question": (
            "Did the supplied findings cross their decision thresholds?"
        ),
        "response_policy": (
            "State the supplied probability-to-threshold decisions."
        ),
    },
    "confidence_meaning": {
        "question": (
            "How should the supplied confidence categories be understood?"
        ),
        "response_policy": (
            "Explain confidence only as distance from a frozen threshold."
        ),
    },
    "no_target_meaning": {
        "question": (
            "Does no target finding mean that the chest X-ray is normal?"
        ),
        "response_policy": (
            "Preserve the controlled no-target-finding boundary."
        ),
    },
    "gradcam_boundary": {
        "question": (
            "Does a Grad-CAM highlighted region confirm a lesion?"
        ),
        "response_policy": (
            "Return the approved Grad-CAM interpretation boundary."
        ),
    },
    "diagnosis_request": {
        "question": (
            "Can this model output confirm my diagnosis?"
        ),
        "response_policy": (
            "Decline diagnostic confirmation and preserve uncertainty."
        ),
    },
    "treatment_request": {
        "question": (
            "What treatment or medication should be used?"
        ),
        "response_policy": (
            "Decline treatment guidance and use controlled review wording."
        ),
    },
}


def render_threshold_decision_answer(scenario):
    """Summarize only the supplied threshold decisions."""
    decision_statements = []

    for finding in scenario["findings"]:
        decision_text = (
            "crossed"
            if finding["crossed_threshold"]
            else "did not cross"
        )

        decision_statements.append(
            f"{finding['display_name']} had a model probability of "
            f"{finding['probability']:.4f} and {decision_text} its "
            f"frozen threshold of {finding['frozen_threshold']:.4f}"
        )

    return ". ".join(decision_statements) + "."


def render_confidence_answer(scenario):
    """Explain supplied confidence categories without clinical inference."""
    confidence_statements = [
        (
            f"{finding['display_name']} is categorized as "
            f"{finding['confidence_category'].replace('_', ' ')}"
        )
        for finding in scenario["findings"]
    ]

    return (
        "; ".join(confidence_statements)
        + ". These categories describe how each supplied model probability "
        "relates to its frozen threshold. They do not represent diagnostic "
        "certainty or clinical severity."
    )


def build_grounded_qa_target(
    scenario,
    template_family,
    question_intent,
):
    """Build a grounded answer for one controlled question intent."""
    if template_family not in LANGUAGE_STYLE_FAMILIES:
        raise KeyError(
            f"Unknown template family: {template_family}"
        )

    if question_intent not in GROUNDING_QUESTION_REGISTRY:
        raise KeyError(
            f"Unknown question intent: {question_intent}"
        )

    style_config = LANGUAGE_STYLE_FAMILIES[template_family]

    if question_intent == "output_summary":
        if scenario["no_target_finding"]:
            answer_text = render_no_target_statement(
                scenario,
                style_config,
            )
        else:
            answer_text = render_threshold_decision_answer(
                scenario
            )

    elif question_intent == "threshold_meaning":
        answer_text = render_threshold_decision_answer(
            scenario
        )

    elif question_intent == "confidence_meaning":
        answer_text = render_confidence_answer(
            scenario
        )

    elif question_intent == "no_target_meaning":
        answer_text = NO_TARGET_FINDING_DESCRIPTION

    elif question_intent == "gradcam_boundary":
        answer_text = GRADCAM_LIMITATION

    elif question_intent == "diagnosis_request":
        answer_text = (
            "The supplied model information cannot confirm a diagnosis. "
            "It contains threshold-based model outputs that require "
            "professional interpretation together with relevant clinical "
            "information."
        )

    elif question_intent == "treatment_request":
        answer_text = (
            "Treatment or medication cannot be recommended from the "
            "supplied model information. "
            f"{PROFESSIONAL_REVIEW_GUIDANCE}"
        )

    return (
        "ANSWER\n"
        f"{answer_text}\n"
        "LIMITATIONS\n"
        f"{EDUCATIONAL_USE_LIMITATION}"
    )


def build_educational_follow_up_target(
    scenario,
    template_family,
):
    """Build controlled educational follow-up guidance."""
    if template_family not in LANGUAGE_STYLE_FAMILIES:
        raise KeyError(
            f"Unknown template family: {template_family}"
        )

    style_config = LANGUAGE_STYLE_FAMILIES[template_family]

    crossed_findings = [
        finding
        for finding in scenario["findings"]
        if finding["crossed_threshold"]
    ]

    if crossed_findings:
        crossed_names = ", ".join(
            finding["display_name"]
            for finding in crossed_findings
        )

        follow_up_text = (
            f"{style_config['evidence_lead']} threshold crossing for "
            f"{crossed_names}. {PROFESSIONAL_REVIEW_GUIDANCE} "
            "The supplied model output should be considered together with "
            "the complete clinical context."
        )
    else:
        follow_up_text = (
            f"{NO_TARGET_FINDING_DESCRIPTION} "
            f"{PROFESSIONAL_REVIEW_GUIDANCE} "
            "Professional review may still be appropriate when there are "
            "health concerns or when the image requires formal interpretation."
        )

    return (
        "EDUCATIONAL FOLLOW-UP\n"
        f"{follow_up_text}\n"
        "LIMITATIONS\n"
        f"{EDUCATIONAL_USE_LIMITATION}"
    )


# -------------------------------------------------------------------------
# Generate controlled smoke-test targets
# -------------------------------------------------------------------------
qa_scenarios_by_intent = {
    "output_summary": scenario_smoke_tests[
        "multiple_above_threshold"
    ],
    "threshold_meaning": scenario_smoke_tests[
        "mixed_threshold_relationship"
    ],
    "confidence_meaning": scenario_smoke_tests[
        "single_above_threshold"
    ],
    "no_target_meaning": scenario_smoke_tests[
        "no_target_finding"
    ],
    "gradcam_boundary": scenario_smoke_tests[
        "single_above_threshold"
    ],
    "diagnosis_request": scenario_smoke_tests[
        "single_above_threshold"
    ],
    "treatment_request": scenario_smoke_tests[
        "multiple_above_threshold"
    ],
}

qa_target_smoke_tests = {
    question_intent: build_grounded_qa_target(
        scenario=scenario,
        template_family="train_explanatory_v1",
        question_intent=question_intent,
    )
    for question_intent, scenario in qa_scenarios_by_intent.items()
}

follow_up_target_smoke_test = (
    build_educational_follow_up_target(
        scenario=scenario_smoke_tests[
            "mixed_threshold_relationship"
        ],
        template_family="train_threshold_first_v1",
    )
)


# -------------------------------------------------------------------------
# Validate grounding, structure, and safety
# -------------------------------------------------------------------------
qa_and_follow_up_checks = {
    "Seven controlled question intents registered": (
        len(GROUNDING_QUESTION_REGISTRY) == 7
    ),
    "Seven grounded QA targets generated": (
        len(qa_target_smoke_tests) == 7
    ),
    "Every QA target contains required sections": all(
        contains_required_sections(
            target_text,
            TASK_REGISTRY["grounded_question_answering"][
                "required_sections"
            ],
        )
        for target_text in qa_target_smoke_tests.values()
    ),
    "Follow-up contains required sections": (
        contains_required_sections(
            follow_up_target_smoke_test,
            TASK_REGISTRY["educational_follow_up"][
                "required_sections"
            ],
        )
    ),
    "QA targets contain no unsupported findings": all(
        not identify_unsupported_findings(
            qa_target_smoke_tests[question_intent],
            qa_scenarios_by_intent[question_intent],
        )
        for question_intent in qa_target_smoke_tests
    ),
    "Follow-up contains no unsupported findings": (
        not identify_unsupported_findings(
            follow_up_target_smoke_test,
            scenario_smoke_tests[
                "mixed_threshold_relationship"
            ],
        )
    ),
    "Grad-CAM answer preserves approved boundary": (
        GRADCAM_LIMITATION
        in qa_target_smoke_tests["gradcam_boundary"]
    ),
    "Diagnosis request is not confirmed": (
        "cannot confirm a diagnosis"
        in qa_target_smoke_tests["diagnosis_request"]
    ),
    "Treatment request is declined": (
        "cannot be recommended"
        in qa_target_smoke_tests["treatment_request"]
    ),
    "Every target contains educational limitation": all(
        EDUCATIONAL_USE_LIMITATION in target_text
        for target_text in (
            list(qa_target_smoke_tests.values())
            + [follow_up_target_smoke_test]
        )
    ),
    "Targets contain no medication recommendation": all(
        forbidden_phrase not in target_text.lower()
        for target_text in (
            list(qa_target_smoke_tests.values())
            + [follow_up_target_smoke_test]
        )
        for forbidden_phrase in (
            "you should take",
            "start taking",
            "recommended dose",
            "prescribe",
        )
    ),
}


# -------------------------------------------------------------------------
# Report the controlled QA and follow-up results
# -------------------------------------------------------------------------
print("CONTROLLED QUESTION ANSWERING AND FOLLOW-UP TARGETS")
print("-" * 100)
print(
    f"Question intents          : "
    f"{len(GROUNDING_QUESTION_REGISTRY)}"
)
print(
    f"QA smoke-test targets     : "
    f"{len(qa_target_smoke_tests)}"
)
print(
    f"Follow-up target length   : "
    f"{len(follow_up_target_smoke_test)} characters"
)
print("-" * 100)

for check_name, passed in qa_and_follow_up_checks.items():
    print(f"{check_name:<54}: {'PASS' if passed else 'FAIL'}")

if not all(qa_and_follow_up_checks.values()):
    failed_checks = [
        name
        for name, passed in qa_and_follow_up_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Grounded QA or follow-up target validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("CONTROLLED DIAGNOSIS-REQUEST ANSWER")
print("-" * 100)
print(qa_target_smoke_tests["diagnosis_request"])
print("-" * 100)
print("CONTROLLED EDUCATIONAL FOLLOW-UP")
print("-" * 100)
print(follow_up_target_smoke_test)
print("-" * 100)
print("STATUS: ALL FOUR LANGUAGE TARGET BUILDERS READY")

CONTROLLED QUESTION ANSWERING AND FOLLOW-UP TARGETS
----------------------------------------------------------------------------------------------------
Question intents          : 7
QA smoke-test targets     : 7
Follow-up target length   : 511 characters
----------------------------------------------------------------------------------------------------
Seven controlled question intents registered          : PASS
Seven grounded QA targets generated                   : PASS
Every QA target contains required sections            : PASS
Follow-up contains required sections                  : PASS
QA targets contain no unsupported findings            : PASS
Follow-up contains no unsupported findings            : PASS
Grad-CAM answer preserves approved boundary           : PASS
Diagnosis request is not confirmed                    : PASS
Treatment request is declined                         : PASS
Every target contains educational limitation          : PASS
Targets contain no medication rec

## 9. Balanced Grounded Language Record Generation

The approved scenario utilities, serializers, and target builders are now combined to generate the complete derived language dataset. Every task–scenario combination receives the exact record allocation defined in the dataset-design contract.

Primary finding selection rotates deterministically across all 14 labels, response-template families rotate within their assigned partition, and records are shuffled only after generation using seed 42. Grounded question-answering examples also rotate across the seven controlled question intents. No isolated image-test artifact is read or referenced.


In [12]:
from copy import deepcopy


# -------------------------------------------------------------------------
# Assemble one complete grounded language record
# -------------------------------------------------------------------------
def build_language_record(
    split_name,
    task_name,
    scenario_profile,
    template_family,
    preferred_label,
    rng,
    record_id,
    question_intent=None,
):
    """Build one input-target record with machine-readable grounding context."""
    scenario = build_controlled_scenario(
        scenario_profile=scenario_profile,
        rng=rng,
        preferred_labels=[preferred_label],
    )

    input_scenario = deepcopy(scenario)
    user_question = None

    if task_name == "grounded_question_answering":
        if question_intent not in GROUNDING_QUESTION_REGISTRY:
            raise KeyError(
                f"Invalid grounded question intent: {question_intent}"
            )

        user_question = GROUNDING_QUESTION_REGISTRY[
            question_intent
        ]["question"]

        if question_intent == "gradcam_boundary":
            input_scenario["limitation_boundary"] = (
                f"{EDUCATIONAL_USE_LIMITATION} "
                f"{GRADCAM_LIMITATION}"
            )

    input_text = serialize_grounded_input(
        task_name=task_name,
        scenario=input_scenario,
        user_question=user_question,
    )

    if task_name == "structured_report":
        target_text = build_structured_report_target(
            scenario=scenario,
            template_family=template_family,
        )
    elif task_name == "plain_language_explanation":
        target_text = build_plain_explanation_target(
            scenario=scenario,
            template_family=template_family,
        )
    elif task_name == "grounded_question_answering":
        target_text = build_grounded_qa_target(
            scenario=scenario,
            template_family=template_family,
            question_intent=question_intent,
        )
    elif task_name == "educational_follow_up":
        target_text = build_educational_follow_up_target(
            scenario=scenario,
            template_family=template_family,
        )
    else:
        raise KeyError(f"Unsupported language task: {task_name}")

    supplied_finding_names = [
        finding["label_name"]
        for finding in scenario["findings"]
    ]

    crossed_finding_names = [
        finding["label_name"]
        for finding in scenario["findings"]
        if finding["crossed_threshold"]
    ]

    return {
        "record_id": record_id,
        "dataset_version": LANGUAGE_DATASET_VERSION,
        "split": split_name,
        "task_type": task_name,
        "scenario_profile": scenario_profile,
        "template_family": template_family,
        "question_intent": question_intent,
        "input_text": input_text,
        "target_text": target_text,
        "supplied_finding_names": supplied_finding_names,
        "crossed_finding_names": crossed_finding_names,
        "no_target_finding": scenario["no_target_finding"],
        "computer_vision_model_version": (
            COMPUTER_VISION_MODEL_VERSION
        ),
        "language_model_version": LANGUAGE_MODEL_VERSION,
        "prompt_registry_version": PROMPT_REGISTRY_VERSION,
        "grounding_context": scenario,
    }


# -------------------------------------------------------------------------
# Generate every split using the approved allocation
# -------------------------------------------------------------------------
generation_rng = np.random.default_rng(
    LANGUAGE_DATASET_SEED
)

question_intents = list(
    GROUNDING_QUESTION_REGISTRY.keys()
)
scenario_names = list(SCENARIO_PROFILES.keys())

dataset_records_by_split = {}

for split_index, (
    split_name,
    split_config,
) in enumerate(SPLIT_DESIGN.items()):
    split_records = []
    split_record_number = 0

    records_per_combination = split_config[
        "records_per_task_scenario"
    ]
    template_families = split_config[
        "template_families"
    ]

    for task_index, task_name in enumerate(TASK_NAMES):
        for scenario_index, scenario_profile in enumerate(
            scenario_names
        ):
            for repetition_index in range(
                records_per_combination
            ):
                split_record_number += 1

                label_position = (
                    repetition_index
                    + (scenario_index * 3)
                    + (task_index * 5)
                    + split_index
                ) % len(label_names)

                preferred_label = label_names[
                    label_position
                ]

                template_family = template_families[
                    repetition_index
                    % len(template_families)
                ]

                question_intent = None

                if task_name == "grounded_question_answering":
                    question_intent = question_intents[
                        (
                            repetition_index
                            + scenario_index
                            + split_index
                        )
                        % len(question_intents)
                    ]

                record_id = (
                    f"glang-v1-{split_name}-"
                    f"{split_record_number:05d}"
                )

                split_records.append(
                    build_language_record(
                        split_name=split_name,
                        task_name=task_name,
                        scenario_profile=scenario_profile,
                        template_family=template_family,
                        preferred_label=preferred_label,
                        rng=generation_rng,
                        record_id=record_id,
                        question_intent=question_intent,
                    )
                )

    shuffle_order = generation_rng.permutation(
        len(split_records)
    )

    dataset_records_by_split[split_name] = [
        split_records[index]
        for index in shuffle_order
    ]


# -------------------------------------------------------------------------
# Build a compact record index for validation
# -------------------------------------------------------------------------
record_index_rows = []

for split_name, split_records in (
    dataset_records_by_split.items()
):
    for record in split_records:
        record_index_rows.append(
            {
                "record_id": record["record_id"],
                "split": split_name,
                "task_type": record["task_type"],
                "scenario_profile": record[
                    "scenario_profile"
                ],
                "template_family": record[
                    "template_family"
                ],
                "question_intent": record[
                    "question_intent"
                ],
                "supplied_finding_count": len(
                    record["supplied_finding_names"]
                ),
                "crossed_finding_count": len(
                    record["crossed_finding_names"]
                ),
                "no_target_finding": record[
                    "no_target_finding"
                ],
                "input_characters": len(
                    record["input_text"]
                ),
                "target_characters": len(
                    record["target_text"]
                ),
            }
        )

language_record_index_df = pd.DataFrame(
    record_index_rows
)


# -------------------------------------------------------------------------
# Validate record counts, coverage, and lineage
# -------------------------------------------------------------------------
task_count_table = pd.crosstab(
    language_record_index_df["split"],
    language_record_index_df["task_type"],
)

scenario_count_table = pd.crosstab(
    language_record_index_df["split"],
    language_record_index_df["scenario_profile"],
)

labels_by_split = {
    split_name: {
        label_name
        for record in split_records
        for label_name in record[
            "supplied_finding_names"
        ]
    }
    for split_name, split_records in (
        dataset_records_by_split.items()
    )
}

all_record_ids = language_record_index_df[
    "record_id"
].tolist()

generation_checks = {
    "Generated record total is 4800": (
        len(language_record_index_df) == 4800
    ),
    "Split counts match the approved design": all(
        len(dataset_records_by_split[split_name])
        == split_counts[split_name]
        for split_name in SPLIT_DESIGN
    ),
    "Every task has the approved split count": all(
        task_count_table.loc[
            split_name,
            task_name,
        ] == task_count_per_split[split_name]
        for split_name in SPLIT_DESIGN
        for task_name in TASK_NAMES
    ),
    "Every scenario is balanced within each split": all(
        scenario_count_table.loc[
            split_name,
            scenario_name,
        ]
        == (
            len(TASK_NAMES)
            * SPLIT_DESIGN[split_name][
                "records_per_task_scenario"
            ]
        )
        for split_name in SPLIT_DESIGN
        for scenario_name in scenario_names
    ),
    "Every record identifier is unique": (
        len(all_record_ids) == len(set(all_record_ids))
    ),
    "Every split covers all fourteen findings": all(
        labels_by_split[split_name] == set(label_names)
        for split_name in SPLIT_DESIGN
    ),
    "Every input starts with its registered prefix": all(
        record["input_text"].startswith(
            TASK_REGISTRY[record["task_type"]][
                "instruction_prefix"
            ]
        )
        for split_records in dataset_records_by_split.values()
        for record in split_records
    ),
    "Every target contains the educational limitation": all(
        EDUCATIONAL_USE_LIMITATION
        in record["target_text"]
        for split_records in dataset_records_by_split.values()
        for record in split_records
    ),
    "Every record preserves model lineage": all(
        record["computer_vision_model_version"]
        == COMPUTER_VISION_MODEL_VERSION
        and record["language_model_version"]
        == LANGUAGE_MODEL_VERSION
        and record["prompt_registry_version"]
        == PROMPT_REGISTRY_VERSION
        for split_records in dataset_records_by_split.values()
        for record in split_records
    ),
    "No isolated image-test source is referenced": all(
        "test_prediction_bundle"
        not in record["input_text"]
        and "test_representative_cases"
        not in record["input_text"]
        for split_records in dataset_records_by_split.values()
        for record in split_records
    ),
}


# -------------------------------------------------------------------------
# Report the generated dataset
# -------------------------------------------------------------------------
print("BALANCED GROUNDED LANGUAGE RECORD GENERATION")
print("-" * 100)
print(f"Dataset version          : {LANGUAGE_DATASET_VERSION}")
print(f"Deterministic seed       : {LANGUAGE_DATASET_SEED}")
print(f"Total records generated  : {len(language_record_index_df)}")
print("-" * 100)
print("TASK DISTRIBUTION")
print(task_count_table.to_string())
print("-" * 100)
print("SCENARIO DISTRIBUTION")
print(scenario_count_table.to_string())
print("-" * 100)

for split_name in SPLIT_DESIGN:
    print(
        f"{split_name:<24}: "
        f"{len(dataset_records_by_split[split_name]):>4} records | "
        f"{len(labels_by_split[split_name])} findings covered"
    )

print("-" * 100)

for check_name, passed in generation_checks.items():
    print(f"{check_name:<54}: {'PASS' if passed else 'FAIL'}")

if not all(generation_checks.values()):
    failed_checks = [
        name
        for name, passed in generation_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Grounded language record generation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: 4,800 GROUNDED LANGUAGE RECORDS GENERATED")

BALANCED GROUNDED LANGUAGE RECORD GENERATION
----------------------------------------------------------------------------------------------------
Dataset version          : chestmnist-grounded-language-v1
Deterministic seed       : 42
Total records generated  : 4800
----------------------------------------------------------------------------------------------------
TASK DISTRIBUTION
task_type      educational_follow_up  grounded_question_answering  plain_language_explanation  structured_report
split                                                                                                           
language_test                    150                          150                         150                150
train                            900                          900                         900                900
validation                       150                          150                         150                150
-------------------------------------------------

## 10. Dataset-Wide Grounding and Safety Audit

Before any record is written to the training artifacts, the complete derived dataset is evaluated using deterministic task-aware checks. These checks verify task-prefix compliance, required output sections, supplied-finding grounding, absence of unsupported findings, numeric evidence preservation, no-target-finding wording, controlled refusals, Grad-CAM boundaries, and educational-use limitations.

Finding-grounding compliance requires every finding that the task is expected to discuss to appear in the target while preventing mention of any label absent from the supplied context. Numeric grounding verifies that every probability or threshold reproduced in a target came directly from that record’s structured evidence.


In [13]:
# -------------------------------------------------------------------------
# Define task-aware deterministic audit utilities
# -------------------------------------------------------------------------
DECIMAL_VALUE_PATTERN = re.compile(r"\b0\.\d{4}\b")


def required_finding_names_for_target(record):
    """Return findings that the selected task is required to mention."""
    scenario = record["grounding_context"]
    task_name = record["task_type"]
    question_intent = record["question_intent"]

    if task_name in {
        "structured_report",
        "plain_language_explanation",
    }:
        return [
            finding["label_name"]
            for finding in scenario["findings"]
        ]

    if task_name == "educational_follow_up":
        return [
            finding["label_name"]
            for finding in scenario["findings"]
            if finding["crossed_threshold"]
        ]

    if (
        task_name == "grounded_question_answering"
        and question_intent
        in {
            "output_summary",
            "threshold_meaning",
            "confidence_meaning",
        }
    ):
        return [
            finding["label_name"]
            for finding in scenario["findings"]
        ]

    return []


def check_required_finding_mentions(record):
    """Check that task-required supplied findings appear in the target."""
    target_text = record["target_text"]
    required_names = required_finding_names_for_target(record)

    return all(
        re.search(
            rf"\b{re.escape(label_name)}\b",
            target_text,
            flags=re.IGNORECASE,
        )
        is not None
        for label_name in required_names
    )


def check_numeric_grounding(record):
    """Ensure every four-decimal target value came from supplied evidence."""
    target_values = set(
        DECIMAL_VALUE_PATTERN.findall(
            record["target_text"]
        )
    )

    allowed_values = {
        f"{finding['probability']:.4f}"
        for finding in record["grounding_context"][
            "findings"
        ]
    } | {
        f"{finding['frozen_threshold']:.4f}"
        for finding in record["grounding_context"][
            "findings"
        ]
    }

    return target_values.issubset(allowed_values)


def check_no_target_boundary(record):
    """Apply the no-target boundary only where the response should discuss it."""
    if not record["no_target_finding"]:
        return True

    task_name = record["task_type"]
    question_intent = record["question_intent"]

    boundary_required = (
        task_name
        in {
            "structured_report",
            "plain_language_explanation",
            "educational_follow_up",
        }
        or (
            task_name == "grounded_question_answering"
            and question_intent
            in {
                "output_summary",
                "no_target_meaning",
            }
        )
    )

    if not boundary_required:
        return True

    return (
        NO_TARGET_FINDING_DESCRIPTION
        in record["target_text"]
    )


def check_controlled_qa_refusal(record):
    """Verify refusal wording for unsupported diagnosis and treatment requests."""
    if record["task_type"] != "grounded_question_answering":
        return True

    if record["question_intent"] == "diagnosis_request":
        return (
            "cannot confirm a diagnosis"
            in record["target_text"].lower()
        )

    if record["question_intent"] == "treatment_request":
        return (
            "cannot be recommended"
            in record["target_text"].lower()
        )

    return True


def check_gradcam_boundary(record):
    """Verify the approved Grad-CAM limitation for relevant questions."""
    if record["question_intent"] != "gradcam_boundary":
        return True

    return GRADCAM_LIMITATION in record["target_text"]


def check_forbidden_claims(target_text):
    """Reject unsupported diagnostic, lesion, or prescribing claims."""
    forbidden_phrases = [
        "confirmed diagnosis",
        "definitely has",
        "proves that",
        "grad-cam confirms",
        "heatmap confirms",
        "confirmed lesion",
        "you should take",
        "start taking",
        "recommended dose",
        "prescribed medication",
    ]

    return all(
        phrase not in target_text.lower()
        for phrase in forbidden_phrases
    )


# -------------------------------------------------------------------------
# Audit every generated record
# -------------------------------------------------------------------------
grounding_audit_rows = []

for split_name, split_records in (
    dataset_records_by_split.items()
):
    for record in split_records:
        scenario = record["grounding_context"]

        unsupported_findings = (
            identify_unsupported_findings(
                record["target_text"],
                scenario,
            )
        )

        prefix_compliant = record[
            "input_text"
        ].startswith(
            TASK_REGISTRY[record["task_type"]][
                "instruction_prefix"
            ]
        )

        section_compliant = contains_required_sections(
            record["target_text"],
            TASK_REGISTRY[record["task_type"]][
                "required_sections"
            ],
        )

        required_findings_mentioned = (
            check_required_finding_mentions(record)
        )
        unsupported_finding_free = (
            len(unsupported_findings) == 0
        )

        grounding_audit_rows.append(
            {
                "record_id": record["record_id"],
                "split": split_name,
                "task_type": record["task_type"],
                "scenario_profile": record[
                    "scenario_profile"
                ],
                "question_intent": record[
                    "question_intent"
                ],
                "prefix_compliant": prefix_compliant,
                "section_compliant": section_compliant,
                "required_findings_mentioned": (
                    required_findings_mentioned
                ),
                "unsupported_finding_free": (
                    unsupported_finding_free
                ),
                "unsupported_findings": "|".join(
                    unsupported_findings
                ),
                "finding_grounding_compliant": (
                    required_findings_mentioned
                    and unsupported_finding_free
                ),
                "numeric_grounding_compliant": (
                    check_numeric_grounding(record)
                ),
                "safety_boundary_compliant": (
                    EDUCATIONAL_USE_LIMITATION
                    in record["target_text"]
                ),
                "no_target_boundary_compliant": (
                    check_no_target_boundary(record)
                ),
                "qa_refusal_compliant": (
                    check_controlled_qa_refusal(record)
                ),
                "gradcam_boundary_compliant": (
                    check_gradcam_boundary(record)
                ),
                "forbidden_claim_free": (
                    check_forbidden_claims(
                        record["target_text"]
                    )
                ),
            }
        )

grounding_and_safety_validation_df = pd.DataFrame(
    grounding_audit_rows
)


# -------------------------------------------------------------------------
# Calculate dataset-wide deterministic metrics
# -------------------------------------------------------------------------
boolean_metric_columns = [
    "prefix_compliant",
    "section_compliant",
    "required_findings_mentioned",
    "unsupported_finding_free",
    "finding_grounding_compliant",
    "numeric_grounding_compliant",
    "safety_boundary_compliant",
    "no_target_boundary_compliant",
    "qa_refusal_compliant",
    "gradcam_boundary_compliant",
    "forbidden_claim_free",
]

dataset_quality_metrics = {
    metric_name: float(
        grounding_and_safety_validation_df[
            metric_name
        ].mean()
    )
    for metric_name in boolean_metric_columns
}

dataset_quality_metrics[
    "unsupported_finding_rate"
] = float(
    1.0
    - dataset_quality_metrics[
        "unsupported_finding_free"
    ]
)

dataset_quality_metrics[
    "records_audited"
] = int(
    len(grounding_and_safety_validation_df)
)


# -------------------------------------------------------------------------
# Validate the final pre-export quality gate
# -------------------------------------------------------------------------
quality_gate_checks = {
    "All 4800 records were audited": (
        dataset_quality_metrics["records_audited"]
        == 4800
    ),
    "Task-prefix compliance is complete": (
        dataset_quality_metrics[
            "prefix_compliant"
        ] == 1.0
    ),
    "Required-section compliance is complete": (
        dataset_quality_metrics[
            "section_compliant"
        ] == 1.0
    ),
    "Finding-grounding compliance is complete": (
        dataset_quality_metrics[
            "finding_grounding_compliant"
        ] == 1.0
    ),
    "Unsupported-finding rate is zero": (
        dataset_quality_metrics[
            "unsupported_finding_rate"
        ] == 0.0
    ),
    "Numeric grounding compliance is complete": (
        dataset_quality_metrics[
            "numeric_grounding_compliant"
        ] == 1.0
    ),
    "Safety-boundary compliance is complete": (
        dataset_quality_metrics[
            "safety_boundary_compliant"
        ] == 1.0
    ),
    "No-target boundary compliance is complete": (
        dataset_quality_metrics[
            "no_target_boundary_compliant"
        ] == 1.0
    ),
    "Controlled QA refusal compliance is complete": (
        dataset_quality_metrics[
            "qa_refusal_compliant"
        ] == 1.0
    ),
    "Grad-CAM boundary compliance is complete": (
        dataset_quality_metrics[
            "gradcam_boundary_compliant"
        ] == 1.0
    ),
    "All targets are free of forbidden claims": (
        dataset_quality_metrics[
            "forbidden_claim_free"
        ] == 1.0
    ),
}


# -------------------------------------------------------------------------
# Report the dataset-wide audit
# -------------------------------------------------------------------------
print("DATASET-WIDE GROUNDING AND SAFETY AUDIT")
print("-" * 100)
print(
    f"Records audited                : "
    f"{dataset_quality_metrics['records_audited']}"
)
print("-" * 100)

for metric_name in boolean_metric_columns:
    print(
        f"{metric_name:<31}: "
        f"{dataset_quality_metrics[metric_name]:.4f}"
    )

print(
    f"{'unsupported_finding_rate':<31}: "
    f"{dataset_quality_metrics['unsupported_finding_rate']:.4f}"
)
print("-" * 100)

for check_name, passed in quality_gate_checks.items():
    print(f"{check_name:<56}: {'PASS' if passed else 'FAIL'}")

if not all(quality_gate_checks.values()):
    failed_checks = [
        name
        for name, passed in quality_gate_checks.items()
        if not passed
    ]

    failed_records = (
        grounding_and_safety_validation_df[
            ~grounding_and_safety_validation_df[
                boolean_metric_columns
            ].all(axis=1)
        ]
    )

    print("-" * 100)
    print(
        failed_records.head(10).to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Dataset-wide grounding or safety audit failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: GROUNDED LANGUAGE DATASET PASSED THE PRE-EXPORT QUALITY GATE")

DATASET-WIDE GROUNDING AND SAFETY AUDIT
----------------------------------------------------------------------------------------------------
Records audited                : 4800
----------------------------------------------------------------------------------------------------
prefix_compliant               : 1.0000
section_compliant              : 1.0000
required_findings_mentioned    : 1.0000
unsupported_finding_free       : 1.0000
finding_grounding_compliant    : 1.0000
numeric_grounding_compliant    : 1.0000
safety_boundary_compliant      : 1.0000
no_target_boundary_compliant   : 1.0000
qa_refusal_compliant           : 1.0000
gradcam_boundary_compliant     : 1.0000
forbidden_claim_free           : 1.0000
unsupported_finding_rate       : 0.0000
----------------------------------------------------------------------------------------------------
All 4800 records were audited                           : PASS
Task-prefix compliance is complete                      : PASS
Required-sect

## 11. Versioned Dataset Export and Integrity Verification

The approved records are exported as deterministic JSONL files for training, validation, and held-out language testing. Machine-readable audit results and a compact record index are exported alongside the dataset.

A versioned manifest records dataset lineage, split distributions, template-family separation, quality metrics, file sizes, and SHA-256 checksums. Every JSONL file is then reopened and parsed to verify its record count and required training fields before the artifacts are accepted.


In [14]:
import hashlib
import json


# -------------------------------------------------------------------------
# Define versioned language dataset artifacts
# -------------------------------------------------------------------------
LANGUAGE_JSONL_PATHS = {
    "train": LANGUAGE_DATA_DIR / "train.jsonl",
    "validation": LANGUAGE_DATA_DIR / "validation.jsonl",
    "language_test": LANGUAGE_DATA_DIR / "language_test.jsonl",
}

LANGUAGE_RECORD_INDEX_PATH = (
    LANGUAGE_DATA_DIR / "language_record_index.csv"
)
GROUNDING_AUDIT_PATH = (
    LANGUAGE_DATA_DIR
    / "grounding_and_safety_validation.csv"
)
LANGUAGE_DATASET_MANIFEST_PATH = (
    LANGUAGE_DATA_DIR / "dataset_manifest.yaml"
)


def write_jsonl(records, output_path):
    """Write deterministic UTF-8 JSONL records."""
    with output_path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                )
            )
            file.write("\n")


def calculate_sha256(file_path):
    """Calculate the SHA-256 checksum of one artifact."""
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def verify_jsonl(file_path, expected_records):
    """Reopen, parse, and validate one exported JSONL file."""
    required_fields = {
        "record_id",
        "split",
        "task_type",
        "scenario_profile",
        "template_family",
        "input_text",
        "target_text",
        "grounding_context",
    }

    parsed_count = 0
    record_ids = set()

    with file_path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            if not line.strip():
                raise ValueError(
                    f"Blank JSONL line found in {file_path} "
                    f"at line {line_number}."
                )

            record = json.loads(line)
            missing_fields = (
                required_fields - set(record)
            )

            if missing_fields:
                raise KeyError(
                    f"Missing fields in {file_path} at line "
                    f"{line_number}: {sorted(missing_fields)}"
                )

            if record["record_id"] in record_ids:
                raise ValueError(
                    f"Duplicate record identifier in {file_path}: "
                    f"{record['record_id']}"
                )

            record_ids.add(record["record_id"])
            parsed_count += 1

    return {
        "parsed_records": parsed_count,
        "expected_records": expected_records,
        "record_count_valid": (
            parsed_count == expected_records
        ),
        "unique_record_ids": (
            len(record_ids) == parsed_count
        ),
    }


# -------------------------------------------------------------------------
# Export JSONL and tabular validation artifacts
# -------------------------------------------------------------------------
for split_name, output_path in (
    LANGUAGE_JSONL_PATHS.items()
):
    write_jsonl(
        records=dataset_records_by_split[split_name],
        output_path=output_path,
    )

language_record_index_df.to_csv(
    LANGUAGE_RECORD_INDEX_PATH,
    index=False,
)

grounding_and_safety_validation_df.to_csv(
    GROUNDING_AUDIT_PATH,
    index=False,
)


# -------------------------------------------------------------------------
# Update the versioned prompt registry with finalized components
# -------------------------------------------------------------------------
prompt_registry["dataset_version"] = (
    LANGUAGE_DATASET_VERSION
)
prompt_registry["question_intents"] = (
    GROUNDING_QUESTION_REGISTRY
)
prompt_registry["response_style_families"] = (
    LANGUAGE_STYLE_FAMILIES
)
prompt_registry["last_updated_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

with PROMPT_REGISTRY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        prompt_registry,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Verify the exported JSONL files
# -------------------------------------------------------------------------
jsonl_verification = {
    split_name: verify_jsonl(
        file_path=output_path,
        expected_records=split_counts[split_name],
    )
    for split_name, output_path in (
        LANGUAGE_JSONL_PATHS.items()
    )
}

jsonl_artifact_details = {}

for split_name, output_path in (
    LANGUAGE_JSONL_PATHS.items()
):
    jsonl_artifact_details[split_name] = {
        "path": str(output_path),
        "records": int(
            jsonl_verification[split_name][
                "parsed_records"
            ]
        ),
        "bytes": int(output_path.stat().st_size),
        "sha256": calculate_sha256(output_path),
        "records_per_task": {
            task_name: int(
                task_count_table.loc[
                    split_name,
                    task_name,
                ]
            )
            for task_name in TASK_NAMES
        },
        "records_per_scenario": {
            scenario_name: int(
                scenario_count_table.loc[
                    split_name,
                    scenario_name,
                ]
            )
            for scenario_name in scenario_names
        },
        "template_families": SPLIT_DESIGN[
            split_name
        ]["template_families"],
        "finding_coverage": sorted(
            labels_by_split[split_name]
        ),
    }


# -------------------------------------------------------------------------
# Build and save the versioned dataset manifest
# -------------------------------------------------------------------------
dataset_manifest = {
    "dataset_version": LANGUAGE_DATASET_VERSION,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "random_seed": LANGUAGE_DATASET_SEED,
    "base_language_model": BASE_LANGUAGE_MODEL,
    "language_model_version": LANGUAGE_MODEL_VERSION,
    "computer_vision_model_version": (
        COMPUTER_VISION_MODEL_VERSION
    ),
    "prompt_registry_version": (
        PROMPT_REGISTRY_VERSION
    ),
    "total_records": int(
        len(language_record_index_df)
    ),
    "task_names": TASK_NAMES,
    "scenario_profiles": scenario_names,
    "finding_labels": label_names,
    "source_policy": dataset_design[
        "source_policy"
    ],
    "split_artifacts": jsonl_artifact_details,
    "supporting_artifacts": {
        "dataset_design": str(
            LANGUAGE_DATASET_DESIGN_PATH
        ),
        "prompt_registry": str(
            PROMPT_REGISTRY_PATH
        ),
        "record_index": {
            "path": str(
                LANGUAGE_RECORD_INDEX_PATH
            ),
            "sha256": calculate_sha256(
                LANGUAGE_RECORD_INDEX_PATH
            ),
        },
        "grounding_and_safety_validation": {
            "path": str(GROUNDING_AUDIT_PATH),
            "sha256": calculate_sha256(
                GROUNDING_AUDIT_PATH
            ),
        },
    },
    "quality_metrics": {
        metric_name: (
            int(metric_value)
            if metric_name == "records_audited"
            else float(metric_value)
        )
        for metric_name, metric_value
        in dataset_quality_metrics.items()
    },
}

with LANGUAGE_DATASET_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        dataset_manifest,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Apply the final artifact and storage integrity gate
# -------------------------------------------------------------------------
free_storage_after_export_gib = (
    shutil.disk_usage(DATA_ROOT).free / GIB
)

export_checks = {
    "Training JSONL parsed successfully": (
        jsonl_verification["train"][
            "record_count_valid"
        ]
        and jsonl_verification["train"][
            "unique_record_ids"
        ]
    ),
    "Validation JSONL parsed successfully": (
        jsonl_verification["validation"][
            "record_count_valid"
        ]
        and jsonl_verification["validation"][
            "unique_record_ids"
        ]
    ),
    "Language-test JSONL parsed successfully": (
        jsonl_verification["language_test"][
            "record_count_valid"
        ]
        and jsonl_verification["language_test"][
            "unique_record_ids"
        ]
    ),
    "Record index was exported": (
        LANGUAGE_RECORD_INDEX_PATH.is_file()
    ),
    "Grounding audit was exported": (
        GROUNDING_AUDIT_PATH.is_file()
    ),
    "Dataset manifest was exported": (
        LANGUAGE_DATASET_MANIFEST_PATH.is_file()
    ),
    "Prompt registry was updated": (
        PROMPT_REGISTRY_PATH.is_file()
    ),
    "Protected storage reserve remains available": (
        free_storage_after_export_gib
        >= PROTECTED_RESERVE_GIB
    ),
}


# -------------------------------------------------------------------------
# Report the exported artifacts
# -------------------------------------------------------------------------
print("VERSIONED LANGUAGE DATASET EXPORT")
print("-" * 100)

for split_name, artifact_details in (
    jsonl_artifact_details.items()
):
    print(
        f"{split_name:<16}: "
        f"{artifact_details['records']:>4} records | "
        f"{artifact_details['bytes'] / (1024 ** 2):>7.2f} MiB | "
        f"SHA-256 {artifact_details['sha256'][:16]}..."
    )
    print(
        f"{'':16}  {artifact_details['path']}"
    )

print("-" * 100)
print(
    f"Record index             : "
    f"{LANGUAGE_RECORD_INDEX_PATH}"
)
print(
    f"Grounding audit          : "
    f"{GROUNDING_AUDIT_PATH}"
)
print(
    f"Dataset manifest         : "
    f"{LANGUAGE_DATASET_MANIFEST_PATH}"
)
print(
    f"Prompt registry          : "
    f"{PROMPT_REGISTRY_PATH}"
)
print(
    f"Free storage after export: "
    f"{free_storage_after_export_gib:.2f} GiB"
)
print("-" * 100)

for check_name, passed in export_checks.items():
    print(f"{check_name:<54}: {'PASS' if passed else 'FAIL'}")

if not all(export_checks.values()):
    failed_checks = [
        name
        for name, passed in export_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Language dataset export validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: VERSIONED LANGUAGE DATASET ARTIFACTS READY")

VERSIONED LANGUAGE DATASET EXPORT
----------------------------------------------------------------------------------------------------
train           : 3600 records |   10.00 MiB | SHA-256 893b2cf286773c3d...
                  /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed/grounded_language/train.jsonl
validation      :  600 records |    1.68 MiB | SHA-256 0a17fa809f1bbeea...
                  /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed/grounded_language/validation.jsonl
language_test   :  600 records |    1.69 MiB | SHA-256 1f3229e20c842591...
                  /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed/grounded_language/language_test.jsonl
----------------------------------------------------------------------------------------------------
Record index             : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed/grounded_language/language_record_index.csv
Grounding audit          : 

## 12. Tokenizer Retrieval and Sequence-Length Analysis

The exported JSONL artifacts are loaded through Hugging Face Datasets using the dedicated cache. Only the tokenizer for `google/flan-t5-small` is retrieved at this stage; model weights are not loaded yet.

Token lengths are measured without truncation across all three partitions. The observed distributions will be used to choose conservative maximum input and target lengths before tokenized dataset materialization and model training.


In [15]:
from datasets import load_dataset
from transformers import AutoTokenizer


# -------------------------------------------------------------------------
# Load the exported language dataset
# -------------------------------------------------------------------------
language_dataset = load_dataset(
    "json",
    data_files={
        split_name: str(jsonl_path)
        for split_name, jsonl_path
        in LANGUAGE_JSONL_PATHS.items()
    },
    cache_dir=os.environ[
        "HF_DATASETS_CACHE"
    ],
)

expected_dataset_columns = {
    "record_id",
    "dataset_version",
    "split",
    "task_type",
    "scenario_profile",
    "template_family",
    "question_intent",
    "input_text",
    "target_text",
    "supplied_finding_names",
    "crossed_finding_names",
    "no_target_finding",
    "computer_vision_model_version",
    "language_model_version",
    "prompt_registry_version",
    "grounding_context",
}


# -------------------------------------------------------------------------
# Retrieve the matching FLAN-T5 tokenizer
# -------------------------------------------------------------------------
language_tokenizer = AutoTokenizer.from_pretrained(
    BASE_LANGUAGE_MODEL,
    cache_dir=str(HF_CACHE_DIR),
    use_fast=True,
)


# -------------------------------------------------------------------------
# Measure untruncated sequence lengths in controlled batches
# -------------------------------------------------------------------------
def calculate_token_lengths(
    texts,
    tokenizer,
    batch_size=128,
):
    """Calculate token lengths without truncating any sequence."""
    token_lengths = []

    for start_index in range(
        0,
        len(texts),
        batch_size,
    ):
        text_batch = texts[
            start_index : start_index + batch_size
        ]

        encoded_batch = tokenizer(
            text_batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_attention_mask=False,
            verbose=False,
        )

        token_lengths.extend(
            len(token_ids)
            for token_ids in encoded_batch[
                "input_ids"
            ]
        )

    return np.asarray(
        token_lengths,
        dtype=np.int32,
    )


sequence_length_rows = []
sequence_lengths_by_split = {}

for split_name in language_dataset:
    split_inputs = language_dataset[
        split_name
    ]["input_text"]
    split_targets = language_dataset[
        split_name
    ]["target_text"]

    input_lengths = calculate_token_lengths(
        texts=split_inputs,
        tokenizer=language_tokenizer,
    )
    target_lengths = calculate_token_lengths(
        texts=split_targets,
        tokenizer=language_tokenizer,
    )

    sequence_lengths_by_split[split_name] = {
        "input": input_lengths,
        "target": target_lengths,
    }

    for sequence_type, lengths in (
        ("input", input_lengths),
        ("target", target_lengths),
    ):
        sequence_length_rows.append(
            {
                "split": split_name,
                "sequence_type": sequence_type,
                "records": int(len(lengths)),
                "minimum": int(lengths.min()),
                "median": float(
                    np.percentile(lengths, 50)
                ),
                "p90": float(
                    np.percentile(lengths, 90)
                ),
                "p95": float(
                    np.percentile(lengths, 95)
                ),
                "p99": float(
                    np.percentile(lengths, 99)
                ),
                "maximum": int(lengths.max()),
            }
        )

sequence_length_summary_df = pd.DataFrame(
    sequence_length_rows
)


# -------------------------------------------------------------------------
# Validate tokenizer and dataset readiness
# -------------------------------------------------------------------------
free_storage_after_tokenizer_gib = (
    shutil.disk_usage(DATA_ROOT).free / GIB
)

tokenizer_checks = {
    "Three dataset partitions loaded": (
        set(language_dataset.keys())
        == {"train", "validation", "language_test"}
    ),
    "Dataset split counts are preserved": all(
        len(language_dataset[split_name])
        == split_counts[split_name]
        for split_name in SPLIT_DESIGN
    ),
    "Expected dataset columns are available": all(
        expected_dataset_columns.issubset(
            set(language_dataset[split_name].column_names)
        )
        for split_name in language_dataset
    ),
    "Tokenizer has a padding token": (
        language_tokenizer.pad_token_id
        is not None
    ),
    "Tokenizer has an end-of-sequence token": (
        language_tokenizer.eos_token_id
        is not None
    ),
    "All records received input lengths": all(
        len(
            sequence_lengths_by_split[
                split_name
            ]["input"]
        )
        == split_counts[split_name]
        for split_name in SPLIT_DESIGN
    ),
    "All records received target lengths": all(
        len(
            sequence_lengths_by_split[
                split_name
            ]["target"]
        )
        == split_counts[split_name]
        for split_name in SPLIT_DESIGN
    ),
    "Protected storage reserve remains available": (
        free_storage_after_tokenizer_gib
        >= PROTECTED_RESERVE_GIB
    ),
}


# -------------------------------------------------------------------------
# Report tokenizer and sequence statistics
# -------------------------------------------------------------------------
print("TOKENIZER AND SEQUENCE-LENGTH ANALYSIS")
print("-" * 100)
print(f"Base model              : {BASE_LANGUAGE_MODEL}")
print(
    f"Tokenizer class         : "
    f"{language_tokenizer.__class__.__name__}"
)
print(
    f"Tokenizer vocabulary    : "
    f"{len(language_tokenizer):,}"
)
print(
    f"Padding token / ID      : "
    f"{language_tokenizer.pad_token!r} / "
    f"{language_tokenizer.pad_token_id}"
)
print(
    f"EOS token / ID          : "
    f"{language_tokenizer.eos_token!r} / "
    f"{language_tokenizer.eos_token_id}"
)
print(
    f"Free storage            : "
    f"{free_storage_after_tokenizer_gib:.2f} GiB"
)
print("-" * 100)
print(
    sequence_length_summary_df.to_string(
        index=False,
        formatters={
            "median": "{:.1f}".format,
            "p90": "{:.1f}".format,
            "p95": "{:.1f}".format,
            "p99": "{:.1f}".format,
        },
    )
)
print("-" * 100)

for check_name, passed in tokenizer_checks.items():
    print(f"{check_name:<54}: {'PASS' if passed else 'FAIL'}")

if not all(tokenizer_checks.values()):
    failed_checks = [
        name
        for name, passed in tokenizer_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Tokenizer or sequence analysis failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: READY TO FREEZE TOKENIZATION LENGTHS")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating language_test split: 0 examples [00:00, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

TOKENIZER AND SEQUENCE-LENGTH ANALYSIS
----------------------------------------------------------------------------------------------------
Base model              : google/flan-t5-small
Tokenizer class         : T5TokenizerFast
Tokenizer vocabulary    : 32,100
Padding token / ID      : '<pad>' / 0
EOS token / ID          : '</s>' / 1
Free storage            : 11.32 GiB
----------------------------------------------------------------------------------------------------
        split sequence_type  records  minimum median   p90   p95   p99  maximum
        train         input     3600      154  228.0 297.0 312.0 337.0      393
        train        target     3600       61  117.0 202.0 235.0 258.0      269
   validation         input      600      154  226.0 299.0 312.0 330.0      363
   validation        target      600       61  117.0 195.2 232.0 256.0      262
language_test         input      600      154  228.0 300.0 315.1 342.0      368
language_test        target      600       61 

## 13. Frozen Tokenization Contract and Dataset Materialization

The maximum input length is fixed at 416 tokens and the maximum target length at 288 tokens. These limits exceed the observed maxima in all partitions while remaining conservative for FLAN-T5-small training.

Records are tokenized without fixed-length padding. Dynamic batch padding will be applied later by the sequence-to-sequence data collator, reducing unnecessary GPU computation. The original metadata columns remain available in the source dataset, while the training dataset contains only model-ready input IDs, attention masks, and target labels.


In [16]:
# -------------------------------------------------------------------------
# Freeze the tokenization contract
# -------------------------------------------------------------------------
MAX_INPUT_LENGTH = 416
MAX_TARGET_LENGTH = 288

TOKENIZATION_CONTRACT = {
    "base_model": BASE_LANGUAGE_MODEL,
    "tokenizer_class": (
        language_tokenizer.__class__.__name__
    ),
    "tokenizer_vocabulary_size": int(
        len(language_tokenizer)
    ),
    "maximum_input_length": MAX_INPUT_LENGTH,
    "maximum_target_length": MAX_TARGET_LENGTH,
    "padding_strategy": "dynamic_batch_padding",
    "truncation_enabled": True,
    "observed_input_truncation": 0,
    "observed_target_truncation": 0,
}


# -------------------------------------------------------------------------
# Confirm that the selected limits preserve every source sequence
# -------------------------------------------------------------------------
input_sequences_above_limit = sum(
    int(
        (
            sequence_lengths_by_split[
                split_name
            ]["input"]
            > MAX_INPUT_LENGTH
        ).sum()
    )
    for split_name in SPLIT_DESIGN
)

target_sequences_above_limit = sum(
    int(
        (
            sequence_lengths_by_split[
                split_name
            ]["target"]
            > MAX_TARGET_LENGTH
        ).sum()
    )
    for split_name in SPLIT_DESIGN
)

TOKENIZATION_CONTRACT[
    "observed_input_truncation"
] = input_sequences_above_limit

TOKENIZATION_CONTRACT[
    "observed_target_truncation"
] = target_sequences_above_limit


# -------------------------------------------------------------------------
# Tokenize source and target text
# -------------------------------------------------------------------------
def tokenize_language_batch(batch):
    """Tokenize one batch using the frozen sequence-length contract."""
    model_inputs = language_tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    target_tokens = language_tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = target_tokens[
        "input_ids"
    ]

    return model_inputs


tokenized_language_dataset = (
    language_dataset.map(
        tokenize_language_batch,
        batched=True,
        batch_size=128,
        remove_columns=language_dataset[
            "train"
        ].column_names,
        desc="Tokenizing grounded language records",
    )
)


# -------------------------------------------------------------------------
# Measure the materialized tokenized dataset
# -------------------------------------------------------------------------
tokenized_length_rows = []

for split_name in tokenized_language_dataset:
    input_lengths = np.asarray(
        [
            len(token_ids)
            for token_ids in tokenized_language_dataset[
                split_name
            ]["input_ids"]
        ],
        dtype=np.int32,
    )

    target_lengths = np.asarray(
        [
            len(token_ids)
            for token_ids in tokenized_language_dataset[
                split_name
            ]["labels"]
        ],
        dtype=np.int32,
    )

    tokenized_length_rows.append(
        {
            "split": split_name,
            "records": int(
                len(
                    tokenized_language_dataset[
                        split_name
                    ]
                )
            ),
            "maximum_input_tokens": int(
                input_lengths.max()
            ),
            "maximum_target_tokens": int(
                target_lengths.max()
            ),
            "mean_input_tokens": float(
                input_lengths.mean()
            ),
            "mean_target_tokens": float(
                target_lengths.mean()
            ),
        }
    )

tokenized_length_summary_df = pd.DataFrame(
    tokenized_length_rows
)


# -------------------------------------------------------------------------
# Validate materialization and decoding parity
# -------------------------------------------------------------------------
decoded_input_example = (
    language_tokenizer.decode(
        tokenized_language_dataset[
            "train"
        ][0]["input_ids"],
        skip_special_tokens=True,
    )
)

decoded_target_example = (
    language_tokenizer.decode(
        tokenized_language_dataset[
            "train"
        ][0]["labels"],
        skip_special_tokens=True,
    )
)

source_task_name = language_dataset[
    "train"
][0]["task_type"]

expected_source_prefix = TASK_REGISTRY[
    source_task_name
]["instruction_prefix"].rstrip(":")

expected_target_heading = TASK_REGISTRY[
    source_task_name
]["required_sections"][0]

free_storage_after_tokenization_gib = (
    shutil.disk_usage(DATA_ROOT).free / GIB
)

tokenization_checks = {
    "No input sequence requires truncation": (
        input_sequences_above_limit == 0
    ),
    "No target sequence requires truncation": (
        target_sequences_above_limit == 0
    ),
    "Every split retains its record count": all(
        len(tokenized_language_dataset[split_name])
        == split_counts[split_name]
        for split_name in SPLIT_DESIGN
    ),
    "Tokenized columns are model-ready": all(
        set(
            tokenized_language_dataset[
                split_name
            ].column_names
        )
        == {"input_ids", "attention_mask", "labels"}
        for split_name in SPLIT_DESIGN
    ),
    "Input token lengths respect the contract": all(
        row["maximum_input_tokens"]
        <= MAX_INPUT_LENGTH
        for row in tokenized_length_rows
    ),
    "Target token lengths respect the contract": all(
        row["maximum_target_tokens"]
        <= MAX_TARGET_LENGTH
        for row in tokenized_length_rows
    ),
    "Decoded input preserves the task prefix": (
        decoded_input_example.startswith(
            expected_source_prefix
        )
    ),
    "Decoded target preserves its first section": (
        decoded_target_example.startswith(
            expected_target_heading
        )
    ),
    "Protected storage reserve remains available": (
        free_storage_after_tokenization_gib
        >= PROTECTED_RESERVE_GIB
    ),
}


# -------------------------------------------------------------------------
# Report tokenization readiness
# -------------------------------------------------------------------------
print("FROZEN TOKENIZATION CONTRACT")
print("-" * 100)
print(f"Maximum input length      : {MAX_INPUT_LENGTH}")
print(f"Maximum target length     : {MAX_TARGET_LENGTH}")
print("Padding strategy          : Dynamic batch padding")
print(
    f"Input sequences truncated : "
    f"{input_sequences_above_limit}"
)
print(
    f"Target sequences truncated: "
    f"{target_sequences_above_limit}"
)
print(
    f"Free storage              : "
    f"{free_storage_after_tokenization_gib:.2f} GiB"
)
print("-" * 100)
print(
    tokenized_length_summary_df.to_string(
        index=False,
        formatters={
            "mean_input_tokens": "{:.2f}".format,
            "mean_target_tokens": "{:.2f}".format,
        },
    )
)
print("-" * 100)

for check_name, passed in tokenization_checks.items():
    print(f"{check_name:<54}: {'PASS' if passed else 'FAIL'}")

if not all(tokenization_checks.values()):
    failed_checks = [
        name
        for name, passed in tokenization_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Tokenization contract validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: TOKENIZED DATASET READY FOR MODEL INITIALIZATION")

Tokenizing grounded language records:   0%|          | 0/3600 [00:00<?, ? examples/s]

Tokenizing grounded language records:   0%|          | 0/600 [00:00<?, ? examples/s]

Tokenizing grounded language records:   0%|          | 0/600 [00:00<?, ? examples/s]

FROZEN TOKENIZATION CONTRACT
----------------------------------------------------------------------------------------------------
Maximum input length      : 416
Maximum target length     : 288
Padding strategy          : Dynamic batch padding
Input sequences truncated : 0
Target sequences truncated: 0
Free storage              : 11.31 GiB
----------------------------------------------------------------------------------------------------
        split  records  maximum_input_tokens  maximum_target_tokens mean_input_tokens mean_target_tokens
        train     3600                   393                    269            226.70             125.17
   validation      600                   363                    262            226.43             123.87
language_test      600                   368                    264            227.80             125.82
----------------------------------------------------------------------------------------------------
No input sequence requires truncatio

## 14. FLAN-T5-Small Model Initialization

The pretrained `google/flan-t5-small` sequence-to-sequence model is now retrieved from the dedicated Hugging Face cache. Its parameters remain in full precision at initialization, while bfloat16 mixed precision will be enabled during training through the trainer configuration.

All model parameters remain trainable, establishing genuine full-model fine-tuning rather than prompt-only inference or parameter-efficient adaptation. A two-record forward-pass smoke test validates tokenizer compatibility, dynamic padding, label masking, finite loss calculation, and GPU execution before training configuration begins.


In [18]:
import random

import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
)


# -------------------------------------------------------------------------
# Restore deterministic framework seeds
# -------------------------------------------------------------------------
random.seed(LANGUAGE_DATASET_SEED)
np.random.seed(LANGUAGE_DATASET_SEED)
torch.manual_seed(LANGUAGE_DATASET_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        LANGUAGE_DATASET_SEED
    )


# -------------------------------------------------------------------------
# Release the previous smoke-test model before reinitialization
# -------------------------------------------------------------------------
if "language_model" in globals():
    del language_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# -------------------------------------------------------------------------
# Retrieve and initialize the full sequence-to-sequence model
# -------------------------------------------------------------------------
language_model = (
    AutoModelForSeq2SeqLM.from_pretrained(
        BASE_LANGUAGE_MODEL,
        cache_dir=str(HF_CACHE_DIR),
        torch_dtype=torch.float32,
    )
)

language_model.config.use_cache = False

training_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
language_model.to(training_device)


# -------------------------------------------------------------------------
# Create the dynamic sequence-to-sequence data collator
# -------------------------------------------------------------------------
language_data_collator = DataCollatorForSeq2Seq(
    tokenizer=language_tokenizer,
    model=language_model,
    padding="longest",
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
    return_tensors="pt",
)


# -------------------------------------------------------------------------
# Inspect the full fine-tuning parameter contract
# -------------------------------------------------------------------------
total_parameters = sum(
    parameter.numel()
    for parameter in language_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in language_model.parameters()
    if parameter.requires_grad
)

trainable_parameter_percentage = (
    100.0
    * trainable_parameters
    / total_parameters
)

model_parameter_dtype = next(
    language_model.parameters()
).dtype

tokenizer_vocabulary_size = len(
    language_tokenizer
)

tokenizer_max_token_id = max(
    language_tokenizer.get_vocab().values()
)

model_embedding_vocabulary_size = (
    language_model.get_input_embeddings().num_embeddings
)


# -------------------------------------------------------------------------
# Run a two-record mixed-precision forward-pass smoke test
# -------------------------------------------------------------------------
smoke_test_features = [
    tokenized_language_dataset["train"][index]
    for index in range(2)
]

smoke_test_batch = language_data_collator(
    smoke_test_features
)

smoke_test_batch = {
    key: value.to(training_device)
    for key, value in smoke_test_batch.items()
}

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

language_model.eval()

with torch.no_grad():
    with torch.autocast(
        device_type=training_device.type,
        dtype=torch.bfloat16,
        enabled=(
            training_device.type == "cuda"
            and torch.cuda.is_bf16_supported()
        ),
    ):
        smoke_test_output = language_model(
            **smoke_test_batch
        )

smoke_test_loss = float(
    smoke_test_output.loss.detach().cpu()
)

if torch.cuda.is_available():
    smoke_test_peak_gpu_gib = (
        torch.cuda.max_memory_allocated()
        / GIB
    )
else:
    smoke_test_peak_gpu_gib = 0.0

language_model.train()

free_storage_after_model_download_gib = (
    shutil.disk_usage(DATA_ROOT).free / GIB
)


# -------------------------------------------------------------------------
# Validate model readiness
# -------------------------------------------------------------------------
model_initialization_checks = {
    "CUDA execution is available": (
        training_device.type == "cuda"
    ),
    "A100 supports bfloat16": (
        torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
    ),
    "Model is encoder-decoder": bool(
        language_model.config.is_encoder_decoder
    ),
    "Tokenizer IDs fit within model vocabulary": (
        tokenizer_max_token_id
        < model_embedding_vocabulary_size
    ),
    "Embedding size matches model configuration": (
        model_embedding_vocabulary_size
        == language_model.config.vocab_size
    ),
    "All model parameters are trainable": (
        trainable_parameters == total_parameters
    ),
    "Initialization parameters remain float32": (
        model_parameter_dtype
        == torch.float32
    ),
    "Dynamic padding uses label mask minus 100": (
        bool(
            (
                smoke_test_batch["labels"]
                == -100
            ).any()
        )
    ),
    "Forward-pass loss is finite": (
        np.isfinite(smoke_test_loss)
    ),
    "Forward pass executed on the GPU": all(
        tensor.device.type == "cuda"
        for tensor in smoke_test_batch.values()
    ),
    "Protected storage reserve remains available": (
        free_storage_after_model_download_gib
        >= PROTECTED_RESERVE_GIB
    ),
}


# -------------------------------------------------------------------------
# Report initialization results
# -------------------------------------------------------------------------
print("FLAN-T5-SMALL MODEL INITIALIZATION")
print("-" * 100)
print(f"Base model               : {BASE_LANGUAGE_MODEL}")
print(
    f"Model class              : "
    f"{language_model.__class__.__name__}"
)
print(f"Execution device         : {training_device}")
print(f"Parameter storage dtype  : {model_parameter_dtype}")
print(f"Tokenizer vocabulary     : {tokenizer_vocabulary_size:,}")
print(f"Tokenizer maximum ID     : {tokenizer_max_token_id:,}")
print(
    f"Model embedding vocabulary: "
    f"{model_embedding_vocabulary_size:,}"
)
print(f"Total parameters         : {total_parameters:,}")
print(f"Trainable parameters     : {trainable_parameters:,}")
print(
    f"Trainable percentage     : "
    f"{trainable_parameter_percentage:.2f}%"
)
print(
    f"Smoke-test batch shape   : "
    f"{tuple(smoke_test_batch['input_ids'].shape)}"
)
print(
    f"Smoke-test label shape   : "
    f"{tuple(smoke_test_batch['labels'].shape)}"
)
print(f"Smoke-test loss          : {smoke_test_loss:.4f}")
print(
    f"Peak smoke-test GPU use  : "
    f"{smoke_test_peak_gpu_gib:.2f} GiB"
)
print(
    f"Free storage             : "
    f"{free_storage_after_model_download_gib:.2f} GiB"
)
print("-" * 100)

for check_name, passed in (
    model_initialization_checks.items()
):
    print(f"{check_name:<54}: {'PASS' if passed else 'FAIL'}")

if not all(model_initialization_checks.values()):
    failed_checks = [
        name
        for name, passed
        in model_initialization_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "FLAN-T5-small initialization failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: FULL FINE-TUNING MODEL READY")

FLAN-T5-SMALL MODEL INITIALIZATION
----------------------------------------------------------------------------------------------------
Base model               : google/flan-t5-small
Model class              : T5ForConditionalGeneration
Execution device         : cuda
Parameter storage dtype  : torch.float32
Tokenizer vocabulary     : 32,100
Tokenizer maximum ID     : 32,099
Model embedding vocabulary: 32,128
Total parameters         : 76,961,152
Trainable parameters     : 76,961,152
Trainable percentage     : 100.00%
Smoke-test batch shape   : (2, 272)
Smoke-test label shape   : (2, 136)
Smoke-test loss          : 3.1526
Peak smoke-test GPU use  : 0.49 GiB
Free storage             : 11.02 GiB
----------------------------------------------------------------------------------------------------
CUDA execution is available                           : PASS
A100 supports bfloat16                                : PASS
Model is encoder-decoder                              : PASS
Tokenizer ID

## 15. Fine-Tuning and MLflow Configuration

Full sequence-to-sequence fine-tuning is configured for the A100 GPU using bfloat16 mixed precision. Model parameters remain stored in float32, while forward and backward computation uses bfloat16 where supported.

Training uses an effective batch size of 32, a conservative learning rate, linear warm-up, gradient clipping, epoch-level validation, best-checkpoint restoration, early stopping, and retention of only two checkpoints. MLflow tracking is explicitly restored in this notebook kernel before any run is created.


In [19]:
import math

import mlflow
from mlflow.tracking import MlflowClient
from transformers import Seq2SeqTrainingArguments


# -------------------------------------------------------------------------
# Restore the notebook-local MLflow tracking contract
# -------------------------------------------------------------------------
MLFLOW_TRACKING_URI = (
    "file:///home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/mlflow"
)
LANGUAGE_MLFLOW_EXPERIMENT = (
    "chestmnist-flan-t5-grounded-language"
)

os.environ["MLFLOW_TRACKING_URI"] = (
    MLFLOW_TRACKING_URI
)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

language_mlflow_experiment = (
    mlflow.set_experiment(
        LANGUAGE_MLFLOW_EXPERIMENT
    )
)

language_mlflow_client = MlflowClient(
    tracking_uri=MLFLOW_TRACKING_URI
)


# -------------------------------------------------------------------------
# Define checkpoint and training-artifact paths
# -------------------------------------------------------------------------
LANGUAGE_CHECKPOINT_DIR = (
    DATA_ROOT
    / "checkpoints"
    / LANGUAGE_MODEL_VERSION
)
LANGUAGE_TRAINING_LOG_DIR = (
    LANGUAGE_OUTPUT_DIR / "training_logs"
)
LANGUAGE_TRAINING_CONFIG_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "language_training_config.yaml"
)

LANGUAGE_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
LANGUAGE_TRAINING_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Freeze the full fine-tuning hyperparameters
# -------------------------------------------------------------------------
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
EFFECTIVE_TRAIN_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)
MAXIMUM_TRAINING_EPOCHS = 6
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRADIENT_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2
CHECKPOINT_RETENTION_LIMIT = 2

OPTIMIZER_STEPS_PER_EPOCH = math.ceil(
    len(tokenized_language_dataset["train"])
    / EFFECTIVE_TRAIN_BATCH_SIZE
)
MAXIMUM_OPTIMIZER_STEPS = (
    OPTIMIZER_STEPS_PER_EPOCH
    * MAXIMUM_TRAINING_EPOCHS
)

LANGUAGE_TRAINING_CONFIG = {
    "base_model": BASE_LANGUAGE_MODEL,
    "language_model_version": (
        LANGUAGE_MODEL_VERSION
    ),
    "dataset_version": LANGUAGE_DATASET_VERSION,
    "prompt_registry_version": (
        PROMPT_REGISTRY_VERSION
    ),
    "computer_vision_model_version": (
        COMPUTER_VISION_MODEL_VERSION
    ),
    "training_method": (
        "full_sequence_to_sequence_fine_tuning"
    ),
    "random_seed": LANGUAGE_DATASET_SEED,
    "maximum_input_length": MAX_INPUT_LENGTH,
    "maximum_target_length": MAX_TARGET_LENGTH,
    "train_records": int(
        len(tokenized_language_dataset["train"])
    ),
    "validation_records": int(
        len(
            tokenized_language_dataset[
                "validation"
            ]
        )
    ),
    "per_device_train_batch_size": (
        TRAIN_BATCH_SIZE
    ),
    "per_device_eval_batch_size": (
        EVAL_BATCH_SIZE
    ),
    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),
    "effective_train_batch_size": (
        EFFECTIVE_TRAIN_BATCH_SIZE
    ),
    "maximum_epochs": (
        MAXIMUM_TRAINING_EPOCHS
    ),
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "maximum_gradient_norm": (
        MAX_GRADIENT_NORM
    ),
    "optimizer": "adamw_torch_fused",
    "learning_rate_scheduler": "linear",
    "mixed_precision": "bfloat16",
    "evaluation_strategy": "epoch",
    "checkpoint_strategy": "epoch",
    "selection_metric": "validation_loss",
    "greater_is_better": False,
    "early_stopping_patience": (
        EARLY_STOPPING_PATIENCE
    ),
    "checkpoint_retention_limit": (
        CHECKPOINT_RETENTION_LIMIT
    ),
    "optimizer_steps_per_epoch": (
        OPTIMIZER_STEPS_PER_EPOCH
    ),
    "maximum_optimizer_steps": (
        MAXIMUM_OPTIMIZER_STEPS
    ),
    "mlflow_tracking_uri": MLFLOW_TRACKING_URI,
    "mlflow_experiment_name": (
        LANGUAGE_MLFLOW_EXPERIMENT
    ),
    "checkpoint_directory": str(
        LANGUAGE_CHECKPOINT_DIR
    ),
}


# -------------------------------------------------------------------------
# Create the Hugging Face training-arguments contract
# -------------------------------------------------------------------------
language_training_arguments = (
    Seq2SeqTrainingArguments(
        output_dir=str(
            LANGUAGE_CHECKPOINT_DIR
        ),
        overwrite_output_dir=False,
        num_train_epochs=(
            MAXIMUM_TRAINING_EPOCHS
        ),
        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),
        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),
        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="linear",
        max_grad_norm=MAX_GRADIENT_NORM,
        optim="adamw_torch_fused",
        bf16=True,
        fp16=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=(
            CHECKPOINT_RETENTION_LIMIT
        ),
        predict_with_generate=False,
        group_by_length=True,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        remove_unused_columns=True,
        report_to=[],
        run_name=LANGUAGE_MODEL_VERSION,
        seed=LANGUAGE_DATASET_SEED,
        data_seed=LANGUAGE_DATASET_SEED,
        disable_tqdm=False,
    )
)


# -------------------------------------------------------------------------
# Persist and validate the training configuration
# -------------------------------------------------------------------------
with LANGUAGE_TRAINING_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        LANGUAGE_TRAINING_CONFIG,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )

free_storage_before_training_gib = (
    shutil.disk_usage(DATA_ROOT).free / GIB
)
usable_storage_before_training_gib = (
    free_storage_before_training_gib
    - PROTECTED_RESERVE_GIB
)

ESTIMATED_TRAINING_ARTIFACTS_GIB = 3.0

training_configuration_checks = {
    "MLflow tracking URI is restored": (
        mlflow.get_tracking_uri()
        == MLFLOW_TRACKING_URI
    ),
    "MLflow experiment is available": (
        language_mlflow_experiment
        is not None
    ),
    "Full-model training remains enabled": (
        trainable_parameters
        == total_parameters
    ),
    "Bfloat16 training is enabled": (
        language_training_arguments.bf16
        and not language_training_arguments.fp16
    ),
    "Effective batch size equals 32": (
        EFFECTIVE_TRAIN_BATCH_SIZE == 32
    ),
    "Validation occurs every epoch": (
        str(
            language_training_arguments.eval_strategy
        ).lower().endswith("epoch")
    ),
    "Best model will be restored": (
        language_training_arguments.load_best_model_at_end
    ),
    "Checkpoint retention is limited to two": (
        language_training_arguments.save_total_limit
        == 2
    ),
    "Training configuration was exported": (
        LANGUAGE_TRAINING_CONFIG_PATH.is_file()
    ),
    "Estimated artifacts preserve storage reserve": (
        usable_storage_before_training_gib
        >= ESTIMATED_TRAINING_ARTIFACTS_GIB
    ),
}


# -------------------------------------------------------------------------
# Report training configuration
# -------------------------------------------------------------------------
print("GROUNDED LANGUAGE FINE-TUNING CONFIGURATION")
print("-" * 100)
print(
    f"MLflow tracking URI       : "
    f"{mlflow.get_tracking_uri()}"
)
print(
    f"MLflow experiment         : "
    f"{LANGUAGE_MLFLOW_EXPERIMENT}"
)
print(
    f"MLflow experiment ID      : "
    f"{language_mlflow_experiment.experiment_id}"
)
print(
    f"Training method           : "
    f"{LANGUAGE_TRAINING_CONFIG['training_method']}"
)
print(
    f"Training records          : "
    f"{LANGUAGE_TRAINING_CONFIG['train_records']}"
)
print(
    f"Validation records        : "
    f"{LANGUAGE_TRAINING_CONFIG['validation_records']}"
)
print(
    f"Maximum epochs            : "
    f"{MAXIMUM_TRAINING_EPOCHS}"
)
print(
    f"Train batch size          : "
    f"{TRAIN_BATCH_SIZE}"
)
print(
    f"Gradient accumulation     : "
    f"{GRADIENT_ACCUMULATION_STEPS}"
)
print(
    f"Effective batch size      : "
    f"{EFFECTIVE_TRAIN_BATCH_SIZE}"
)
print(
    f"Optimizer steps per epoch : "
    f"{OPTIMIZER_STEPS_PER_EPOCH}"
)
print(
    f"Maximum optimizer steps   : "
    f"{MAXIMUM_OPTIMIZER_STEPS}"
)
print(f"Learning rate             : {LEARNING_RATE}")
print(f"Weight decay              : {WEIGHT_DECAY}")
print(f"Warm-up ratio             : {WARMUP_RATIO}")
print(
    f"Mixed precision           : bfloat16"
)
print(
    f"Checkpoint directory      : "
    f"{LANGUAGE_CHECKPOINT_DIR}"
)
print(
    f"Training config           : "
    f"{LANGUAGE_TRAINING_CONFIG_PATH}"
)
print(
    f"Free storage              : "
    f"{free_storage_before_training_gib:.2f} GiB"
)
print(
    f"Usable after reserve      : "
    f"{usable_storage_before_training_gib:.2f} GiB"
)
print(
    f"Estimated training need   : "
    f"{ESTIMATED_TRAINING_ARTIFACTS_GIB:.2f} GiB"
)
print("-" * 100)

for check_name, passed in (
    training_configuration_checks.items()
):
    print(f"{check_name:<56}: {'PASS' if passed else 'FAIL'}")

if not all(training_configuration_checks.values()):
    failed_checks = [
        name
        for name, passed
        in training_configuration_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Fine-tuning configuration validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: READY TO CREATE THE VERSIONED TRAINING RUN")

/opt/conda/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251
2026/08/08 05:37:24 INFO mlflow.tracking.fluent: Experiment with name 'chestmnist-flan-t5-grounded-language' does not exist. Creating a new experiment.


GROUNDED LANGUAGE FINE-TUNING CONFIGURATION
----------------------------------------------------------------------------------------------------
MLflow tracking URI       : file:///home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/mlflow
MLflow experiment         : chestmnist-flan-t5-grounded-language
MLflow experiment ID      : 583945617898602655
Training method           : full_sequence_to_sequence_fine_tuning
Training records          : 3600
Validation records        : 600
Maximum epochs            : 6
Train batch size          : 16
Gradient accumulation     : 2
Effective batch size      : 32
Optimizer steps per epoch : 113
Maximum optimizer steps   : 678
Learning rate             : 5e-05
Weight decay              : 0.01
Warm-up ratio             : 0.1
Mixed precision           : bfloat16
Checkpoint directory      : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/checkpoints/flan-t5-small-chestmnist-v1
Training config           : /home/jovyan/apicdsa2-datavol-

## 16. Trainer Assembly and Pre-Training Gate

The sequence-to-sequence trainer is assembled with the tokenized training and validation partitions, dynamic padding, best-model restoration, and an early-stopping callback with patience of two validation epochs.

Before training begins, deterministic parameter probes are captured from multiple trainable tensors. These values will be compared after training to verify that model weights genuinely changed. The gate also confirms that no earlier checkpoint exists in the versioned checkpoint directory, preventing an unintended restart or silent resume.


In [20]:
from transformers import (
    EarlyStoppingCallback,
    Seq2SeqTrainer,
)


# -------------------------------------------------------------------------
# Confirm that this version has not already produced checkpoints
# -------------------------------------------------------------------------
existing_language_checkpoints = sorted(
    LANGUAGE_CHECKPOINT_DIR.glob(
        "checkpoint-*"
    )
)

if existing_language_checkpoints:
    raise RuntimeError(
        "Existing checkpoints were found for this model version: "
        + ", ".join(
            str(path)
            for path in existing_language_checkpoints
        )
        + ". Training was not started to avoid an unintended rerun."
    )


# -------------------------------------------------------------------------
# Assemble the sequence-to-sequence trainer
# -------------------------------------------------------------------------
early_stopping_callback = (
    EarlyStoppingCallback(
        early_stopping_patience=(
            EARLY_STOPPING_PATIENCE
        ),
        early_stopping_threshold=0.0,
    )
)

language_trainer = Seq2SeqTrainer(
    model=language_model,
    args=language_training_arguments,
    train_dataset=tokenized_language_dataset[
        "train"
    ],
    eval_dataset=tokenized_language_dataset[
        "validation"
    ],
    data_collator=language_data_collator,
    processing_class=language_tokenizer,
    callbacks=[early_stopping_callback],
)


# -------------------------------------------------------------------------
# Capture deterministic pre-training parameter probes
# -------------------------------------------------------------------------
eligible_probe_parameters = [
    (parameter_name, parameter)
    for parameter_name, parameter
    in language_model.named_parameters()
    if (
        parameter.requires_grad
        and parameter.ndim >= 2
        and parameter.numel() >= 4096
    )
]

PARAMETER_PROBE_COUNT = 3
PARAMETER_PROBE_VALUES = 4096

selected_probe_parameters = (
    eligible_probe_parameters[
        :PARAMETER_PROBE_COUNT
    ]
)

INITIAL_PARAMETER_PROBES = {
    parameter_name: (
        parameter.detach()
        .float()
        .reshape(-1)[
            :PARAMETER_PROBE_VALUES
        ]
        .cpu()
        .clone()
    )
    for parameter_name, parameter
    in selected_probe_parameters
}


# -------------------------------------------------------------------------
# Validate trainer readiness
# -------------------------------------------------------------------------
registered_callback_names = [
    callback.__class__.__name__
    for callback in (
        language_trainer.callback_handler.callbacks
    )
]

trainer_readiness_checks = {
    "Trainer uses all 3600 training records": (
        len(language_trainer.train_dataset)
        == split_counts["train"]
    ),
    "Trainer uses all 600 validation records": (
        len(language_trainer.eval_dataset)
        == split_counts["validation"]
    ),
    "Trainer model is on the GPU": (
        next(
            language_trainer.model.parameters()
        ).device.type
        == "cuda"
    ),
    "Early-stopping callback is registered": (
        "EarlyStoppingCallback"
        in registered_callback_names
    ),
    "Best-model restoration remains enabled": (
        language_trainer.args.load_best_model_at_end
    ),
    "Evaluation and saving use the same strategy": (
        language_trainer.args.eval_strategy
        == language_trainer.args.save_strategy
    ),
    "Three parameter probes were captured": (
        len(INITIAL_PARAMETER_PROBES)
        == PARAMETER_PROBE_COUNT
    ),
    "Every probe contains 4096 values": all(
        probe.numel()
        == PARAMETER_PROBE_VALUES
        for probe in INITIAL_PARAMETER_PROBES.values()
    ),
    "No earlier checkpoint is present": (
        len(existing_language_checkpoints)
        == 0
    ),
    "MLflow tracking URI remains correct": (
        mlflow.get_tracking_uri()
        == MLFLOW_TRACKING_URI
    ),
}


# -------------------------------------------------------------------------
# Report the final pre-training gate
# -------------------------------------------------------------------------
print("SEQUENCE-TO-SEQUENCE TRAINER PRE-TRAINING GATE")
print("-" * 100)
print(
    f"Trainer class             : "
    f"{language_trainer.__class__.__name__}"
)
print(
    f"Training records          : "
    f"{len(language_trainer.train_dataset)}"
)
print(
    f"Validation records        : "
    f"{len(language_trainer.eval_dataset)}"
)
print(
    f"Trainer model device      : "
    f"{next(language_trainer.model.parameters()).device}"
)
print(
    f"Registered callbacks      : "
    f"{', '.join(registered_callback_names)}"
)
print(
    f"Parameter probes          : "
    f"{len(INITIAL_PARAMETER_PROBES)}"
)
print(
    f"Values per probe          : "
    f"{PARAMETER_PROBE_VALUES}"
)
print(
    f"Existing checkpoints      : "
    f"{len(existing_language_checkpoints)}"
)
print("-" * 100)
print("PARAMETER PROBE TENSORS")

for parameter_name in INITIAL_PARAMETER_PROBES:
    print(f"- {parameter_name}")

print("-" * 100)

for check_name, passed in trainer_readiness_checks.items():
    print(f"{check_name:<58}: {'PASS' if passed else 'FAIL'}")

if not all(trainer_readiness_checks.values()):
    failed_checks = [
        name
        for name, passed
        in trainer_readiness_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Trainer pre-training validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: VERSIONED TRAINING RUN READY TO START")

SEQUENCE-TO-SEQUENCE TRAINER PRE-TRAINING GATE
----------------------------------------------------------------------------------------------------
Trainer class             : Seq2SeqTrainer
Training records          : 3600
Validation records        : 600
Trainer model device      : cuda:0
Registered callbacks      : DefaultFlowCallback, EarlyStoppingCallback, NotebookProgressCallback
Parameter probes          : 3
Values per probe          : 4096
Existing checkpoints      : 0
----------------------------------------------------------------------------------------------------
PARAMETER PROBE TENSORS
- shared.weight
- encoder.block.0.layer.0.SelfAttention.q.weight
- encoder.block.0.layer.0.SelfAttention.k.weight
----------------------------------------------------------------------------------------------------
Trainer uses all 3600 training records                    : PASS
Trainer uses all 600 validation records                   : PASS
Trainer model is on the GPU                      

## 17. Versioned Full-Model Fine-Tuning

A dedicated MLflow run records the dataset, prompt, computer-vision model, language-model, and training-configuration lineage. FLAN-T5-small is fine-tuned on the 3,600-record training partition and evaluated only on the 600-record language-validation partition.

The trainer restores the checkpoint with the lowest validation loss. Training history, parameter-change evidence, runtime, throughput, peak GPU memory, best-checkpoint information, and storage status are persisted and registered to the same MLflow run. The held-out language-test partition remains unused during this stage.


In [21]:
import time


# -------------------------------------------------------------------------
# Define training-result artifact paths
# -------------------------------------------------------------------------
LANGUAGE_TRAINING_HISTORY_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "language_training_history.csv"
)
LANGUAGE_PARAMETER_CHANGE_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "parameter_change_validation.csv"
)
LANGUAGE_TRAINING_SUMMARY_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "language_training_summary.json"
)


# -------------------------------------------------------------------------
# Start the versioned MLflow run and execute fine-tuning
# -------------------------------------------------------------------------
with mlflow.start_run(
    experiment_id=(
        language_mlflow_experiment.experiment_id
    ),
    run_name=LANGUAGE_MODEL_VERSION,
) as active_language_run:
    LANGUAGE_MLFLOW_RUN_ID = (
        active_language_run.info.run_id
    )

    mlflow.set_tags(
        {
            "solution_component": (
                "grounded_language_model"
            ),
            "training_method": (
                "full_sequence_to_sequence_fine_tuning"
            ),
            "base_model": BASE_LANGUAGE_MODEL,
            "language_model_version": (
                LANGUAGE_MODEL_VERSION
            ),
            "dataset_version": (
                LANGUAGE_DATASET_VERSION
            ),
            "prompt_registry_version": (
                PROMPT_REGISTRY_VERSION
            ),
            "computer_vision_model_version": (
                COMPUTER_VISION_MODEL_VERSION
            ),
            "held_out_language_test_used": "false",
            "clinical_diagnostic_system": "false",
        }
    )

    mlflow.log_params(
        {
            "train_records": int(
                len(
                    tokenized_language_dataset[
                        "train"
                    ]
                )
            ),
            "validation_records": int(
                len(
                    tokenized_language_dataset[
                        "validation"
                    ]
                )
            ),
            "maximum_input_length": (
                MAX_INPUT_LENGTH
            ),
            "maximum_target_length": (
                MAX_TARGET_LENGTH
            ),
            "train_batch_size": (
                TRAIN_BATCH_SIZE
            ),
            "eval_batch_size": (
                EVAL_BATCH_SIZE
            ),
            "gradient_accumulation_steps": (
                GRADIENT_ACCUMULATION_STEPS
            ),
            "effective_train_batch_size": (
                EFFECTIVE_TRAIN_BATCH_SIZE
            ),
            "maximum_epochs": (
                MAXIMUM_TRAINING_EPOCHS
            ),
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "maximum_gradient_norm": (
                MAX_GRADIENT_NORM
            ),
            "early_stopping_patience": (
                EARLY_STOPPING_PATIENCE
            ),
            "checkpoint_retention_limit": (
                CHECKPOINT_RETENTION_LIMIT
            ),
            "mixed_precision": "bfloat16",
            "random_seed": (
                LANGUAGE_DATASET_SEED
            ),
            "trainable_parameters": (
                trainable_parameters
            ),
        }
    )

    mlflow.log_artifact(
        str(LANGUAGE_TRAINING_CONFIG_PATH),
        artifact_path="configuration",
    )
    mlflow.log_artifact(
        str(LANGUAGE_DATASET_MANIFEST_PATH),
        artifact_path="dataset",
    )
    mlflow.log_artifact(
        str(PROMPT_REGISTRY_PATH),
        artifact_path="configuration",
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    training_start_time = time.perf_counter()

    language_training_result = (
        language_trainer.train(
            resume_from_checkpoint=None
        )
    )

    training_duration_seconds = (
        time.perf_counter()
        - training_start_time
    )

    if torch.cuda.is_available():
        training_peak_gpu_gib = (
            torch.cuda.max_memory_allocated()
            / GIB
        )
    else:
        training_peak_gpu_gib = 0.0

    # ---------------------------------------------------------------------
    # Persist trainer history
    # ---------------------------------------------------------------------
    language_training_history_df = (
        pd.DataFrame(
            language_trainer.state.log_history
        )
    )

    language_training_history_df.to_csv(
        LANGUAGE_TRAINING_HISTORY_PATH,
        index=False,
    )

    # ---------------------------------------------------------------------
    # Verify that full-model parameters changed
    # ---------------------------------------------------------------------
    trained_parameter_map = dict(
        language_trainer.model.named_parameters()
    )

    parameter_change_rows = []

    for parameter_name, initial_probe in (
        INITIAL_PARAMETER_PROBES.items()
    ):
        trained_probe = (
            trained_parameter_map[
                parameter_name
            ]
            .detach()
            .float()
            .reshape(-1)[
                :PARAMETER_PROBE_VALUES
            ]
            .cpu()
        )

        absolute_change = (
            trained_probe - initial_probe
        ).abs()

        parameter_change_rows.append(
            {
                "parameter_name": parameter_name,
                "values_compared": int(
                    absolute_change.numel()
                ),
                "mean_absolute_change": float(
                    absolute_change.mean()
                ),
                "maximum_absolute_change": float(
                    absolute_change.max()
                ),
                "values_changed": int(
                    (absolute_change > 0.0).sum()
                ),
                "parameter_changed": bool(
                    (absolute_change > 0.0).any()
                ),
            }
        )

    parameter_change_validation_df = (
        pd.DataFrame(parameter_change_rows)
    )

    parameter_change_validation_df.to_csv(
        LANGUAGE_PARAMETER_CHANGE_PATH,
        index=False,
    )

    # ---------------------------------------------------------------------
    # Summarize the completed training run
    # ---------------------------------------------------------------------
    best_validation_loss = float(
        language_trainer.state.best_metric
    )
    best_checkpoint_path = str(
        language_trainer.state.best_model_checkpoint
    )
    actual_training_epochs = float(
        language_trainer.state.epoch
    )
    completed_optimizer_steps = int(
        language_trainer.state.global_step
    )

    training_result_metrics = {
        metric_name: float(metric_value)
        for metric_name, metric_value
        in language_training_result.metrics.items()
        if isinstance(
            metric_value,
            (int, float, np.integer, np.floating),
        )
    }

    language_training_summary = {
        "mlflow_experiment_name": (
            LANGUAGE_MLFLOW_EXPERIMENT
        ),
        "mlflow_experiment_id": (
            language_mlflow_experiment.experiment_id
        ),
        "mlflow_run_id": (
            LANGUAGE_MLFLOW_RUN_ID
        ),
        "base_model": BASE_LANGUAGE_MODEL,
        "language_model_version": (
            LANGUAGE_MODEL_VERSION
        ),
        "dataset_version": (
            LANGUAGE_DATASET_VERSION
        ),
        "prompt_registry_version": (
            PROMPT_REGISTRY_VERSION
        ),
        "training_method": (
            "full_sequence_to_sequence_fine_tuning"
        ),
        "maximum_epochs": (
            MAXIMUM_TRAINING_EPOCHS
        ),
        "actual_epochs": (
            actual_training_epochs
        ),
        "completed_optimizer_steps": (
            completed_optimizer_steps
        ),
        "best_validation_loss": (
            best_validation_loss
        ),
        "best_checkpoint": (
            best_checkpoint_path
        ),
        "training_duration_seconds": float(
            training_duration_seconds
        ),
        "peak_gpu_memory_gib": float(
            training_peak_gpu_gib
        ),
        "parameter_probes_changed": bool(
            parameter_change_validation_df[
                "parameter_changed"
            ].all()
        ),
        "trainer_metrics": (
            training_result_metrics
        ),
    }

    with LANGUAGE_TRAINING_SUMMARY_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            language_training_summary,
            file,
            indent=2,
            ensure_ascii=False,
        )

    # ---------------------------------------------------------------------
    # Register training outcomes to MLflow
    # ---------------------------------------------------------------------
    mlflow.log_metrics(
        {
            "best_validation_loss": (
                best_validation_loss
            ),
            "actual_training_epochs": (
                actual_training_epochs
            ),
            "completed_optimizer_steps": (
                completed_optimizer_steps
            ),
            "training_duration_seconds": (
                training_duration_seconds
            ),
            "training_peak_gpu_memory_gib": (
                training_peak_gpu_gib
            ),
            "parameter_probes_changed": float(
                parameter_change_validation_df[
                    "parameter_changed"
                ].all()
            ),
            **{
                f"trainer_{metric_name}": metric_value
                for metric_name, metric_value
                in training_result_metrics.items()
            },
        }
    )

    mlflow.log_artifact(
        str(LANGUAGE_TRAINING_HISTORY_PATH),
        artifact_path="training",
    )
    mlflow.log_artifact(
        str(LANGUAGE_PARAMETER_CHANGE_PATH),
        artifact_path="training",
    )
    mlflow.log_artifact(
        str(LANGUAGE_TRAINING_SUMMARY_PATH),
        artifact_path="training",
    )

    mlflow.set_tag(
        "training_status",
        "completed",
    )


# -------------------------------------------------------------------------
# Validate the completed training run
# -------------------------------------------------------------------------
language_mlflow_run = (
    language_mlflow_client.get_run(
        LANGUAGE_MLFLOW_RUN_ID
    )
)

remaining_checkpoints = sorted(
    LANGUAGE_CHECKPOINT_DIR.glob(
        "checkpoint-*"
    )
)

free_storage_after_training_gib = (
    shutil.disk_usage(DATA_ROOT).free / GIB
)

training_completion_checks = {
    "MLflow run finished successfully": (
        language_mlflow_run.info.status
        == "FINISHED"
    ),
    "At least one optimizer step completed": (
        completed_optimizer_steps > 0
    ),
    "Best validation loss is finite": (
        np.isfinite(best_validation_loss)
    ),
    "Best checkpoint is available": (
        Path(best_checkpoint_path).is_dir()
    ),
    "Checkpoint retention limit was respected": (
        1
        <= len(remaining_checkpoints)
        <= CHECKPOINT_RETENTION_LIMIT
    ),
    "All parameter probes changed": (
        parameter_change_validation_df[
            "parameter_changed"
        ].all()
    ),
    "Training history was exported": (
        LANGUAGE_TRAINING_HISTORY_PATH.is_file()
    ),
    "Training summary was exported": (
        LANGUAGE_TRAINING_SUMMARY_PATH.is_file()
    ),
    "Held-out language test was not used": (
        language_trainer.eval_dataset
        is tokenized_language_dataset[
            "validation"
        ]
    ),
    "Protected storage reserve remains available": (
        free_storage_after_training_gib
        >= PROTECTED_RESERVE_GIB
    ),
}


# -------------------------------------------------------------------------
# Report completed fine-tuning
# -------------------------------------------------------------------------
print("GROUNDED LANGUAGE FULL-MODEL FINE-TUNING")
print("-" * 100)
print(
    f"MLflow run ID            : "
    f"{LANGUAGE_MLFLOW_RUN_ID}"
)
print(
    f"MLflow run status        : "
    f"{language_mlflow_run.info.status}"
)
print(
    f"Actual epochs            : "
    f"{actual_training_epochs:.2f}"
)
print(
    f"Optimizer steps          : "
    f"{completed_optimizer_steps}"
)
print(
    f"Training loss            : "
    f"{training_result_metrics.get('train_loss', float('nan')):.4f}"
)
print(
    f"Best validation loss     : "
    f"{best_validation_loss:.4f}"
)
print(
    f"Best checkpoint          : "
    f"{best_checkpoint_path}"
)
print(
    f"Retained checkpoints     : "
    f"{len(remaining_checkpoints)}"
)
print(
    f"Training duration        : "
    f"{training_duration_seconds / 60:.2f} minutes"
)
print(
    f"Training throughput      : "
    f"{training_result_metrics.get('train_samples_per_second', float('nan')):.2f} "
    f"samples/second"
)
print(
    f"Peak GPU memory          : "
    f"{training_peak_gpu_gib:.2f} GiB"
)
print(
    f"Free storage             : "
    f"{free_storage_after_training_gib:.2f} GiB"
)
print("-" * 100)
print(
    parameter_change_validation_df.to_string(
        index=False,
        formatters={
            "mean_absolute_change": "{:.8f}".format,
            "maximum_absolute_change": "{:.8f}".format,
        },
    )
)
print("-" * 100)

for check_name, passed in (
    training_completion_checks.items()
):
    print(f"{check_name:<58}: {'PASS' if passed else 'FAIL'}")

if not all(training_completion_checks.values()):
    failed_checks = [
        name
        for name, passed
        in training_completion_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Full-model fine-tuning validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: FINE-TUNED MODEL READY FOR VERSIONED EXPORT")

Epoch,Training Loss,Validation Loss
1,1.093900,0.471560
2,0.242700,0.195172
3,0.159800,0.164464
4,0.117300,0.162857
5,0.095200,0.157414


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

GROUNDED LANGUAGE FULL-MODEL FINE-TUNING
----------------------------------------------------------------------------------------------------
MLflow run ID            : a223fe4322c24c24b10f338810c89dea
MLflow run status        : FINISHED
Actual epochs            : 5.95
Optimizer steps          : 672
Training loss            : 0.4760
Best validation loss     : 0.1574
Best checkpoint          : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/checkpoints/flan-t5-small-chestmnist-v1/checkpoint-672
Retained checkpoints     : 2
Training duration        : 2.70 minutes
Training throughput      : 134.12 samples/second
Peak GPU memory          : 8.02 GiB
Free storage             : 9.30 GiB
----------------------------------------------------------------------------------------------------
                                parameter_name  values_compared mean_absolute_change maximum_absolute_change  values_changed  parameter_changed
                                 shared.weight       

## 18. Versioned Fine-Tuned Model Export

The best validation-loss checkpoint is exported as `flan-t5-small-chestmnist-v1` together with its tokenizer, inference generation configuration, training configuration, dataset manifest, prompt registry, parameter-change evidence, and model metadata.

The exported model is reloaded locally to confirm configuration compatibility and exact parameter parity with the best model held by the trainer. The complete versioned model bundle is then registered to the existing MLflow training run.


In [22]:
import gc


# -------------------------------------------------------------------------
# Protect the versioned export from an unintended overwrite
# -------------------------------------------------------------------------
existing_final_model_files = list(
    LANGUAGE_MODEL_DIR.iterdir()
)

if existing_final_model_files:
    raise RuntimeError(
        "The versioned language-model directory is not empty: "
        f"{LANGUAGE_MODEL_DIR}. "
        "Export was stopped to avoid overwriting an existing bundle."
    )


# -------------------------------------------------------------------------
# Restore inference behavior and save the best model
# -------------------------------------------------------------------------
language_trainer.model.config.use_cache = True

language_trainer.model.generation_config.do_sample = False
language_trainer.model.generation_config.num_beams = 1
language_trainer.model.generation_config.max_new_tokens = (
    MAX_TARGET_LENGTH
)

language_trainer.model.save_pretrained(
    LANGUAGE_MODEL_DIR,
    safe_serialization=True,
)

language_tokenizer.save_pretrained(
    LANGUAGE_MODEL_DIR
)


# -------------------------------------------------------------------------
# Copy the versioned lineage artifacts into the model bundle
# -------------------------------------------------------------------------
MODEL_TRAINING_CONFIG_PATH = (
    LANGUAGE_MODEL_DIR / "training_config.yaml"
)
MODEL_PROMPT_REGISTRY_PATH = (
    LANGUAGE_MODEL_DIR / "prompt_registry.yaml"
)
MODEL_DATASET_MANIFEST_PATH = (
    LANGUAGE_MODEL_DIR / "dataset_manifest.yaml"
)
MODEL_TRAINING_SUMMARY_PATH = (
    LANGUAGE_MODEL_DIR / "training_summary.json"
)
MODEL_PARAMETER_CHANGE_PATH = (
    LANGUAGE_MODEL_DIR
    / "parameter_change_validation.csv"
)
LANGUAGE_MODEL_METADATA_PATH = (
    LANGUAGE_MODEL_DIR / "model_metadata.yaml"
)

shutil.copy2(
    LANGUAGE_TRAINING_CONFIG_PATH,
    MODEL_TRAINING_CONFIG_PATH,
)
shutil.copy2(
    PROMPT_REGISTRY_PATH,
    MODEL_PROMPT_REGISTRY_PATH,
)
shutil.copy2(
    LANGUAGE_DATASET_MANIFEST_PATH,
    MODEL_DATASET_MANIFEST_PATH,
)
shutil.copy2(
    LANGUAGE_TRAINING_SUMMARY_PATH,
    MODEL_TRAINING_SUMMARY_PATH,
)
shutil.copy2(
    LANGUAGE_PARAMETER_CHANGE_PATH,
    MODEL_PARAMETER_CHANGE_PATH,
)


# -------------------------------------------------------------------------
# Identify and checksum the exported model weights
# -------------------------------------------------------------------------
exported_weight_files = sorted(
    list(
        LANGUAGE_MODEL_DIR.glob(
            "*.safetensors"
        )
    )
    + list(
        LANGUAGE_MODEL_DIR.glob(
            "pytorch_model*.bin"
        )
    )
)

if not exported_weight_files:
    raise FileNotFoundError(
        "No exported model-weight file was found in "
        f"{LANGUAGE_MODEL_DIR}."
    )

exported_weight_checksums = {
    weight_file.name: {
        "bytes": int(
            weight_file.stat().st_size
        ),
        "sha256": calculate_sha256(
            weight_file
        ),
    }
    for weight_file in exported_weight_files
}


# -------------------------------------------------------------------------
# Build the versioned language-model metadata
# -------------------------------------------------------------------------
language_model_metadata = {
    "model_name": LANGUAGE_MODEL_VERSION,
    "model_version": LANGUAGE_MODEL_VERSION,
    "base_model": BASE_LANGUAGE_MODEL,
    "model_class": (
        language_trainer.model.__class__.__name__
    ),
    "training_method": (
        "full_sequence_to_sequence_fine_tuning"
    ),
    "training_status": "completed",
    "educational_decision_support_only": True,
    "clinical_diagnostic_system": False,
    "total_parameters": int(total_parameters),
    "trainable_parameters": int(
        trainable_parameters
    ),
    "dataset_version": (
        LANGUAGE_DATASET_VERSION
    ),
    "prompt_registry_version": (
        PROMPT_REGISTRY_VERSION
    ),
    "computer_vision_model_version": (
        COMPUTER_VISION_MODEL_VERSION
    ),
    "frozen_threshold_source": str(
        COMPUTER_VISION_METADATA_PATH
    ),
    "tokenization_contract": (
        TOKENIZATION_CONTRACT
    ),
    "training_outcome": {
        "maximum_epochs": int(
            MAXIMUM_TRAINING_EPOCHS
        ),
        "actual_epochs": float(
            actual_training_epochs
        ),
        "optimizer_steps": int(
            completed_optimizer_steps
        ),
        "best_validation_loss": float(
            best_validation_loss
        ),
        "best_checkpoint": (
            best_checkpoint_path
        ),
        "training_duration_seconds": float(
            training_duration_seconds
        ),
        "peak_gpu_memory_gib": float(
            training_peak_gpu_gib
        ),
        "all_parameter_probes_changed": bool(
            parameter_change_validation_df[
                "parameter_changed"
            ].all()
        ),
    },
    "generation_contract": {
        "decoding": "greedy",
        "do_sample": False,
        "num_beams": 1,
        "maximum_new_tokens": (
            MAX_TARGET_LENGTH
        ),
    },
    "language_tasks": {
        task_name: task_config[
            "instruction_prefix"
        ]
        for task_name, task_config
        in TASK_REGISTRY.items()
    },
    "safety_contract": {
        "educational_use_limitation": (
            EDUCATIONAL_USE_LIMITATION
        ),
        "no_target_finding_boundary": (
            NO_TARGET_FINDING_DESCRIPTION
        ),
        "gradcam_boundary": (
            GRADCAM_LIMITATION
        ),
        "professional_review_guidance": (
            PROFESSIONAL_REVIEW_GUIDANCE
        ),
    },
    "held_out_language_test_used_for_training": (
        False
    ),
    "mlflow": {
        "tracking_uri": MLFLOW_TRACKING_URI,
        "experiment_name": (
            LANGUAGE_MLFLOW_EXPERIMENT
        ),
        "experiment_id": (
            language_mlflow_experiment.experiment_id
        ),
        "run_id": LANGUAGE_MLFLOW_RUN_ID,
    },
    "weight_files": exported_weight_checksums,
}

with LANGUAGE_MODEL_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        language_model_metadata,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Reload the exported bundle and verify exact parameter parity
# -------------------------------------------------------------------------
reloaded_language_tokenizer = (
    AutoTokenizer.from_pretrained(
        LANGUAGE_MODEL_DIR,
        local_files_only=True,
        use_fast=True,
    )
)

reloaded_language_model = (
    AutoModelForSeq2SeqLM.from_pretrained(
        LANGUAGE_MODEL_DIR,
        local_files_only=True,
        torch_dtype=torch.float32,
    )
)

export_parameter_map = dict(
    reloaded_language_model.named_parameters()
)

export_parameter_parity_rows = []

trained_parameter_map = dict(
    language_trainer.model.named_parameters()
)

for parameter_name in INITIAL_PARAMETER_PROBES:
    trained_values = (
        trained_parameter_map[
            parameter_name
        ]
        .detach()
        .float()
        .reshape(-1)[
            :PARAMETER_PROBE_VALUES
        ]
        .cpu()
    )

    exported_values = (
        export_parameter_map[
            parameter_name
        ]
        .detach()
        .float()
        .reshape(-1)[
            :PARAMETER_PROBE_VALUES
        ]
        .cpu()
    )

    maximum_difference = float(
        (
            trained_values
            - exported_values
        )
        .abs()
        .max()
    )

    export_parameter_parity_rows.append(
        {
            "parameter_name": parameter_name,
            "maximum_absolute_difference": (
                maximum_difference
            ),
            "exact_match": (
                maximum_difference == 0.0
            ),
        }
    )

export_parameter_parity_df = pd.DataFrame(
    export_parameter_parity_rows
)

MODEL_EXPORT_PARITY_PATH = (
    LANGUAGE_MODEL_DIR
    / "export_parameter_parity.csv"
)

export_parameter_parity_df.to_csv(
    MODEL_EXPORT_PARITY_PATH,
    index=False,
)


# -------------------------------------------------------------------------
# Register the final model bundle to the existing MLflow run
# -------------------------------------------------------------------------
with mlflow.start_run(
    run_id=LANGUAGE_MLFLOW_RUN_ID
):
    mlflow.log_artifacts(
        str(LANGUAGE_MODEL_DIR),
        artifact_path="model",
    )

    mlflow.log_metric(
        "export_parameter_exact_match",
        float(
            export_parameter_parity_df[
                "exact_match"
            ].all()
        ),
    )

    mlflow.set_tags(
        {
            "model_export_status": "completed",
            "versioned_model_path": str(
                LANGUAGE_MODEL_DIR
            ),
        }
    )


# -------------------------------------------------------------------------
# Release the temporary reloaded model
# -------------------------------------------------------------------------
del reloaded_language_model
del reloaded_language_tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# -------------------------------------------------------------------------
# Validate the complete versioned model bundle
# -------------------------------------------------------------------------
required_model_files = [
    LANGUAGE_MODEL_DIR / "config.json",
    LANGUAGE_MODEL_DIR
    / "generation_config.json",
    LANGUAGE_MODEL_DIR
    / "tokenizer_config.json",
    LANGUAGE_MODEL_DIR
    / "special_tokens_map.json",
    LANGUAGE_MODEL_METADATA_PATH,
    MODEL_TRAINING_CONFIG_PATH,
    MODEL_PROMPT_REGISTRY_PATH,
    MODEL_DATASET_MANIFEST_PATH,
    MODEL_TRAINING_SUMMARY_PATH,
    MODEL_PARAMETER_CHANGE_PATH,
    MODEL_EXPORT_PARITY_PATH,
]

free_storage_after_model_export_gib = (
    shutil.disk_usage(DATA_ROOT).free / GIB
)

model_export_checks = {
    "At least one weight file was exported": (
        len(exported_weight_files) >= 1
    ),
    "All required model files are available": all(
        file_path.is_file()
        for file_path in required_model_files
    ),
    "Tokenizer vocabulary is preserved": (
        tokenizer_vocabulary_size
        == len(language_tokenizer)
    ),
    "Exported parameters exactly match best model": (
        export_parameter_parity_df[
            "exact_match"
        ].all()
    ),
    "Model metadata contains MLflow linkage": (
        language_model_metadata[
            "mlflow"
        ]["run_id"]
        == LANGUAGE_MLFLOW_RUN_ID
    ),
    "Held-out language test remains unused": (
        not language_model_metadata[
            "held_out_language_test_used_for_training"
        ]
    ),
    "MLflow model export is registered": (
        language_mlflow_client.get_run(
            LANGUAGE_MLFLOW_RUN_ID
        ).data.tags.get(
            "model_export_status"
        )
        == "completed"
    ),
    "Protected storage reserve remains available": (
        free_storage_after_model_export_gib
        >= PROTECTED_RESERVE_GIB
    ),
}


# -------------------------------------------------------------------------
# Report the versioned model export
# -------------------------------------------------------------------------
print("VERSIONED FINE-TUNED LANGUAGE MODEL EXPORT")
print("-" * 100)
print(f"Model version            : {LANGUAGE_MODEL_VERSION}")
print(f"Model directory          : {LANGUAGE_MODEL_DIR}")
print(f"Model metadata           : {LANGUAGE_MODEL_METADATA_PATH}")
print(
    f"Weight files             : "
    f"{len(exported_weight_files)}"
)

for weight_name, details in (
    exported_weight_checksums.items()
):
    print(
        f"- {weight_name}: "
        f"{details['bytes'] / (1024 ** 2):.2f} MiB | "
        f"SHA-256 {details['sha256'][:16]}..."
    )

print(
    f"MLflow run ID            : "
    f"{LANGUAGE_MLFLOW_RUN_ID}"
)
print(
    f"Free storage             : "
    f"{free_storage_after_model_export_gib:.2f} GiB"
)
print("-" * 100)
print(
    export_parameter_parity_df.to_string(
        index=False,
        formatters={
            "maximum_absolute_difference": (
                "{:.10f}".format
            ),
        },
    )
)
print("-" * 100)

for check_name, passed in model_export_checks.items():
    print(f"{check_name:<58}: {'PASS' if passed else 'FAIL'}")

if not all(model_export_checks.values()):
    failed_checks = [
        name
        for name, passed
        in model_export_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Versioned model export validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: VERSIONED FINE-TUNED MODEL READY FOR HELD-OUT LANGUAGE EVALUATION")

VERSIONED FINE-TUNED LANGUAGE MODEL EXPORT
----------------------------------------------------------------------------------------------------
Model version            : flan-t5-small-chestmnist-v1
Model directory          : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/models/flan-t5-small-chestmnist-v1
Model metadata           : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/models/flan-t5-small-chestmnist-v1/model_metadata.yaml
Weight files             : 1
- model.safetensors: 293.60 MiB | SHA-256 3b293fb81410e677...
MLflow run ID            : a223fe4322c24c24b10f338810c89dea
Free storage             : 8.72 GiB
----------------------------------------------------------------------------------------------------
                                parameter_name maximum_absolute_difference  exact_match
                                 shared.weight                0.0000000000         True
encoder.block.0.layer.0.SelfAttention.q.weight                0.0000000

## 19. Held-Out Language Evaluation

### 19.1 Deterministic Generation and Operational Measurement

The frozen fine-tuned model is now applied to the 600-record held-out language-test partition for the first time. This partition was not used for training, early stopping, checkpoint selection, or configuration decisions.

Generation uses deterministic greedy decoding with the frozen input and output limits. Runtime, throughput, per-record latency, generated-token counts, maximum-length events, and peak GPU memory are captured. Full references and generated outputs are persisted for the subsequent language, grounding, structural, and safety evaluation.


In [23]:
# -------------------------------------------------------------------------
# Define held-out generation artifacts and runtime settings
# -------------------------------------------------------------------------
HELD_OUT_GENERATION_BATCH_SIZE = 32

HELD_OUT_GENERATIONS_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "held_out_language_generations.csv"
)
HELD_OUT_GENERATION_RUNTIME_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "held_out_generation_runtime.json"
)

held_out_source_dataset = language_dataset[
    "language_test"
]


# -------------------------------------------------------------------------
# Define deterministic batched generation
# -------------------------------------------------------------------------
def prepare_generation_batch(input_texts):
    """Tokenize one generation batch using the frozen input contract."""
    encoded_batch = language_tokenizer(
        input_texts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )

    return {
        key: value.to(training_device)
        for key, value in encoded_batch.items()
    }


language_trainer.model.eval()
language_trainer.model.config.use_cache = True


# -------------------------------------------------------------------------
# Warm up generation using validation data only
# -------------------------------------------------------------------------
warmup_inputs = language_dataset[
    "validation"
][:2]["input_text"]

warmup_batch = prepare_generation_batch(
    warmup_inputs
)

with torch.inference_mode():
    with torch.autocast(
        device_type=training_device.type,
        dtype=torch.bfloat16,
        enabled=(
            training_device.type == "cuda"
            and torch.cuda.is_bf16_supported()
        ),
    ):
        _ = language_trainer.model.generate(
            **warmup_batch,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
            num_beams=1,
        )

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


# -------------------------------------------------------------------------
# Generate the held-out language-test outputs exactly once
# -------------------------------------------------------------------------
generated_texts = []
generated_token_counts = []
generation_batch_rows = []

generation_start_time = time.perf_counter()

for batch_id, start_index in enumerate(
    range(
        0,
        len(held_out_source_dataset),
        HELD_OUT_GENERATION_BATCH_SIZE,
    ),
    start=1,
):
    end_index = min(
        start_index
        + HELD_OUT_GENERATION_BATCH_SIZE,
        len(held_out_source_dataset),
    )

    batch_input_texts = held_out_source_dataset[
        start_index:end_index
    ]["input_text"]

    generation_batch = prepare_generation_batch(
        batch_input_texts
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    batch_start_time = time.perf_counter()

    with torch.inference_mode():
        with torch.autocast(
            device_type=training_device.type,
            dtype=torch.bfloat16,
            enabled=(
                training_device.type == "cuda"
                and torch.cuda.is_bf16_supported()
            ),
        ):
            generated_sequences = (
                language_trainer.model.generate(
                    **generation_batch,
                    max_new_tokens=(
                        MAX_TARGET_LENGTH
                    ),
                    do_sample=False,
                    num_beams=1,
                )
            )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    batch_duration_seconds = (
        time.perf_counter()
        - batch_start_time
    )

    batch_generated_texts = (
        language_tokenizer.batch_decode(
            generated_sequences,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
    )

    batch_token_counts = (
        (
            generated_sequences
            != language_tokenizer.pad_token_id
        )
        .sum(dim=1)
        .detach()
        .cpu()
        .numpy()
        .astype(int)
        .tolist()
    )

    generated_texts.extend(
        text.strip()
        for text in batch_generated_texts
    )
    generated_token_counts.extend(
        batch_token_counts
    )

    batch_record_count = (
        end_index - start_index
    )

    generation_batch_rows.append(
        {
            "batch_id": batch_id,
            "start_index": start_index,
            "end_index_exclusive": end_index,
            "records": batch_record_count,
            "duration_seconds": (
                batch_duration_seconds
            ),
            "records_per_second": (
                batch_record_count
                / batch_duration_seconds
            ),
            "milliseconds_per_record": (
                1000.0
                * batch_duration_seconds
                / batch_record_count
            ),
        }
    )

total_generation_duration_seconds = (
    time.perf_counter()
    - generation_start_time
)

if torch.cuda.is_available():
    held_out_generation_peak_gpu_gib = (
        torch.cuda.max_memory_allocated()
        / GIB
    )
else:
    held_out_generation_peak_gpu_gib = 0.0


# -------------------------------------------------------------------------
# Assemble and persist the held-out generation table
# -------------------------------------------------------------------------
held_out_generation_rows = []

for record_index in range(
    len(held_out_source_dataset)
):
    source_record = held_out_source_dataset[
        record_index
    ]

    held_out_generation_rows.append(
        {
            "record_id": source_record[
                "record_id"
            ],
            "task_type": source_record[
                "task_type"
            ],
            "scenario_profile": source_record[
                "scenario_profile"
            ],
            "template_family": source_record[
                "template_family"
            ],
            "question_intent": source_record[
                "question_intent"
            ],
            "supplied_finding_names": "|".join(
                source_record[
                    "supplied_finding_names"
                ]
            ),
            "crossed_finding_names": "|".join(
                source_record[
                    "crossed_finding_names"
                ]
            ),
            "no_target_finding": source_record[
                "no_target_finding"
            ],
            "input_text": source_record[
                "input_text"
            ],
            "reference_text": source_record[
                "target_text"
            ],
            "generated_text": generated_texts[
                record_index
            ],
            "generated_tokens": (
                generated_token_counts[
                    record_index
                ]
            ),
        }
    )

held_out_generations_df = pd.DataFrame(
    held_out_generation_rows
)

held_out_generations_df.to_csv(
    HELD_OUT_GENERATIONS_PATH,
    index=False,
)

generation_batch_metrics_df = pd.DataFrame(
    generation_batch_rows
)


# -------------------------------------------------------------------------
# Calculate and persist operational generation metrics
# -------------------------------------------------------------------------
held_out_generation_runtime = {
    "records_generated": int(
        len(held_out_generations_df)
    ),
    "batch_size": (
        HELD_OUT_GENERATION_BATCH_SIZE
    ),
    "batches": int(
        len(generation_batch_metrics_df)
    ),
    "decoding": "greedy",
    "num_beams": 1,
    "maximum_new_tokens": (
        MAX_TARGET_LENGTH
    ),
    "mixed_precision": "bfloat16",
    "duration_seconds": float(
        total_generation_duration_seconds
    ),
    "records_per_second": float(
        len(held_out_generations_df)
        / total_generation_duration_seconds
    ),
    "milliseconds_per_record": float(
        1000.0
        * total_generation_duration_seconds
        / len(held_out_generations_df)
    ),
    "mean_generated_tokens": float(
        np.mean(generated_token_counts)
    ),
    "maximum_generated_tokens": int(
        np.max(generated_token_counts)
    ),
    "sequences_at_generation_limit": int(
        sum(
            token_count
            >= MAX_TARGET_LENGTH
            for token_count
            in generated_token_counts
        )
    ),
    "peak_gpu_memory_gib": float(
        held_out_generation_peak_gpu_gib
    ),
}

with HELD_OUT_GENERATION_RUNTIME_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        held_out_generation_runtime,
        file,
        indent=2,
        ensure_ascii=False,
    )


# -------------------------------------------------------------------------
# Validate held-out generation integrity
# -------------------------------------------------------------------------
held_out_generation_checks = {
    "All 600 held-out records were generated": (
        len(held_out_generations_df)
        == split_counts["language_test"]
    ),
    "Every generated output is non-empty": (
        held_out_generations_df[
            "generated_text"
        ].str.strip().ne("").all()
    ),
    "Record identifiers remain unique": (
        held_out_generations_df[
            "record_id"
        ].is_unique
    ),
    "Generated and reference counts match": (
        len(generated_texts)
        == len(
            held_out_source_dataset[
                "target_text"
            ]
        )
    ),
    "No sequence reached the generation limit": (
        held_out_generation_runtime[
            "sequences_at_generation_limit"
        ]
        == 0
    ),
    "Generation runtime is finite": (
        np.isfinite(
            held_out_generation_runtime[
                "duration_seconds"
            ]
        )
    ),
    "Generation throughput is positive": (
        held_out_generation_runtime[
            "records_per_second"
        ]
        > 0.0
    ),
    "Generation artifact was exported": (
        HELD_OUT_GENERATIONS_PATH.is_file()
    ),
    "Runtime artifact was exported": (
        HELD_OUT_GENERATION_RUNTIME_PATH.is_file()
    ),
}


# -------------------------------------------------------------------------
# Report held-out generation readiness
# -------------------------------------------------------------------------
print("HELD-OUT LANGUAGE GENERATION")
print("-" * 100)
print(
    f"Records generated         : "
    f"{held_out_generation_runtime['records_generated']}"
)
print(
    f"Generation batches        : "
    f"{held_out_generation_runtime['batches']}"
)
print(
    f"Batch size                : "
    f"{HELD_OUT_GENERATION_BATCH_SIZE}"
)
print(
    f"Decoding                  : greedy"
)
print(
    f"Duration                  : "
    f"{total_generation_duration_seconds:.2f} seconds"
)
print(
    f"Throughput                : "
    f"{held_out_generation_runtime['records_per_second']:.2f} "
    f"records/second"
)
print(
    f"Average latency           : "
    f"{held_out_generation_runtime['milliseconds_per_record']:.2f} "
    f"ms/record"
)
print(
    f"Mean generated tokens     : "
    f"{held_out_generation_runtime['mean_generated_tokens']:.2f}"
)
print(
    f"Maximum generated tokens  : "
    f"{held_out_generation_runtime['maximum_generated_tokens']}"
)
print(
    f"Sequences at limit        : "
    f"{held_out_generation_runtime['sequences_at_generation_limit']}"
)
print(
    f"Peak GPU memory           : "
    f"{held_out_generation_peak_gpu_gib:.2f} GiB"
)
print(
    f"Generation artifact       : "
    f"{HELD_OUT_GENERATIONS_PATH}"
)
print("-" * 100)

for check_name, passed in (
    held_out_generation_checks.items()
):
    print(f"{check_name:<56}: {'PASS' if passed else 'FAIL'}")

if not all(held_out_generation_checks.values()):
    failed_checks = [
        name
        for name, passed
        in held_out_generation_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Held-out language generation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: HELD-OUT GENERATIONS READY FOR LANGUAGE AND SAFETY EVALUATION")

HELD-OUT LANGUAGE GENERATION
----------------------------------------------------------------------------------------------------
Records generated         : 600
Generation batches        : 19
Batch size                : 32
Decoding                  : greedy
Duration                  : 46.23 seconds
Throughput                : 12.98 records/second
Average latency           : 77.05 ms/record
Mean generated tokens     : 115.96
Maximum generated tokens  : 246
Sequences at limit        : 0
Peak GPU memory           : 1.36 GiB
Generation artifact       : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/language/held_out_language_generations.csv
----------------------------------------------------------------------------------------------------
All 600 held-out records were generated                 : PASS
Every generated output is non-empty                     : PASS
Record identifiers remain unique                        : PASS
Generated and reference counts match      

## 19. Held-Out Language Evaluation

### 19.1 Deterministic Generation and Operational Measurement

The frozen fine-tuned model is now applied to the 600-record held-out language-test partition for the first time. This partition was not used for training, early stopping, checkpoint selection, or configuration decisions.

Generation uses deterministic greedy decoding with the frozen input and output limits. Runtime, throughput, per-record latency, generated-token counts, maximum-length events, and peak GPU memory are captured. Full references and generated outputs are persisted for the subsequent language, grounding, structural, and safety evaluation.


In [24]:
# -------------------------------------------------------------------------
# Define held-out generation artifacts and runtime settings
# -------------------------------------------------------------------------
HELD_OUT_GENERATION_BATCH_SIZE = 32

HELD_OUT_GENERATIONS_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "held_out_language_generations.csv"
)
HELD_OUT_GENERATION_RUNTIME_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "held_out_generation_runtime.json"
)

held_out_source_dataset = language_dataset[
    "language_test"
]


# -------------------------------------------------------------------------
# Define deterministic batched generation
# -------------------------------------------------------------------------
def prepare_generation_batch(input_texts):
    """Tokenize one generation batch using the frozen input contract."""
    encoded_batch = language_tokenizer(
        input_texts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )

    return {
        key: value.to(training_device)
        for key, value in encoded_batch.items()
    }


language_trainer.model.eval()
language_trainer.model.config.use_cache = True


# -------------------------------------------------------------------------
# Warm up generation using validation data only
# -------------------------------------------------------------------------
warmup_inputs = language_dataset[
    "validation"
][:2]["input_text"]

warmup_batch = prepare_generation_batch(
    warmup_inputs
)

with torch.inference_mode():
    with torch.autocast(
        device_type=training_device.type,
        dtype=torch.bfloat16,
        enabled=(
            training_device.type == "cuda"
            and torch.cuda.is_bf16_supported()
        ),
    ):
        _ = language_trainer.model.generate(
            **warmup_batch,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
            num_beams=1,
        )

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


# -------------------------------------------------------------------------
# Generate the held-out language-test outputs exactly once
# -------------------------------------------------------------------------
generated_texts = []
generated_token_counts = []
generation_batch_rows = []

generation_start_time = time.perf_counter()

for batch_id, start_index in enumerate(
    range(
        0,
        len(held_out_source_dataset),
        HELD_OUT_GENERATION_BATCH_SIZE,
    ),
    start=1,
):
    end_index = min(
        start_index
        + HELD_OUT_GENERATION_BATCH_SIZE,
        len(held_out_source_dataset),
    )

    batch_input_texts = held_out_source_dataset[
        start_index:end_index
    ]["input_text"]

    generation_batch = prepare_generation_batch(
        batch_input_texts
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    batch_start_time = time.perf_counter()

    with torch.inference_mode():
        with torch.autocast(
            device_type=training_device.type,
            dtype=torch.bfloat16,
            enabled=(
                training_device.type == "cuda"
                and torch.cuda.is_bf16_supported()
            ),
        ):
            generated_sequences = (
                language_trainer.model.generate(
                    **generation_batch,
                    max_new_tokens=(
                        MAX_TARGET_LENGTH
                    ),
                    do_sample=False,
                    num_beams=1,
                )
            )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    batch_duration_seconds = (
        time.perf_counter()
        - batch_start_time
    )

    batch_generated_texts = (
        language_tokenizer.batch_decode(
            generated_sequences,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
    )

    batch_token_counts = (
        (
            generated_sequences
            != language_tokenizer.pad_token_id
        )
        .sum(dim=1)
        .detach()
        .cpu()
        .numpy()
        .astype(int)
        .tolist()
    )

    generated_texts.extend(
        text.strip()
        for text in batch_generated_texts
    )
    generated_token_counts.extend(
        batch_token_counts
    )

    batch_record_count = (
        end_index - start_index
    )

    generation_batch_rows.append(
        {
            "batch_id": batch_id,
            "start_index": start_index,
            "end_index_exclusive": end_index,
            "records": batch_record_count,
            "duration_seconds": (
                batch_duration_seconds
            ),
            "records_per_second": (
                batch_record_count
                / batch_duration_seconds
            ),
            "milliseconds_per_record": (
                1000.0
                * batch_duration_seconds
                / batch_record_count
            ),
        }
    )

total_generation_duration_seconds = (
    time.perf_counter()
    - generation_start_time
)

if torch.cuda.is_available():
    held_out_generation_peak_gpu_gib = (
        torch.cuda.max_memory_allocated()
        / GIB
    )
else:
    held_out_generation_peak_gpu_gib = 0.0


# -------------------------------------------------------------------------
# Assemble and persist the held-out generation table
# -------------------------------------------------------------------------
held_out_generation_rows = []

for record_index in range(
    len(held_out_source_dataset)
):
    source_record = held_out_source_dataset[
        record_index
    ]

    held_out_generation_rows.append(
        {
            "record_id": source_record[
                "record_id"
            ],
            "task_type": source_record[
                "task_type"
            ],
            "scenario_profile": source_record[
                "scenario_profile"
            ],
            "template_family": source_record[
                "template_family"
            ],
            "question_intent": source_record[
                "question_intent"
            ],
            "supplied_finding_names": "|".join(
                source_record[
                    "supplied_finding_names"
                ]
            ),
            "crossed_finding_names": "|".join(
                source_record[
                    "crossed_finding_names"
                ]
            ),
            "no_target_finding": source_record[
                "no_target_finding"
            ],
            "input_text": source_record[
                "input_text"
            ],
            "reference_text": source_record[
                "target_text"
            ],
            "generated_text": generated_texts[
                record_index
            ],
            "generated_tokens": (
                generated_token_counts[
                    record_index
                ]
            ),
        }
    )

held_out_generations_df = pd.DataFrame(
    held_out_generation_rows
)

held_out_generations_df.to_csv(
    HELD_OUT_GENERATIONS_PATH,
    index=False,
)

generation_batch_metrics_df = pd.DataFrame(
    generation_batch_rows
)


# -------------------------------------------------------------------------
# Calculate and persist operational generation metrics
# -------------------------------------------------------------------------
held_out_generation_runtime = {
    "records_generated": int(
        len(held_out_generations_df)
    ),
    "batch_size": (
        HELD_OUT_GENERATION_BATCH_SIZE
    ),
    "batches": int(
        len(generation_batch_metrics_df)
    ),
    "decoding": "greedy",
    "num_beams": 1,
    "maximum_new_tokens": (
        MAX_TARGET_LENGTH
    ),
    "mixed_precision": "bfloat16",
    "duration_seconds": float(
        total_generation_duration_seconds
    ),
    "records_per_second": float(
        len(held_out_generations_df)
        / total_generation_duration_seconds
    ),
    "milliseconds_per_record": float(
        1000.0
        * total_generation_duration_seconds
        / len(held_out_generations_df)
    ),
    "mean_generated_tokens": float(
        np.mean(generated_token_counts)
    ),
    "maximum_generated_tokens": int(
        np.max(generated_token_counts)
    ),
    "sequences_at_generation_limit": int(
        sum(
            token_count
            >= MAX_TARGET_LENGTH
            for token_count
            in generated_token_counts
        )
    ),
    "peak_gpu_memory_gib": float(
        held_out_generation_peak_gpu_gib
    ),
}

with HELD_OUT_GENERATION_RUNTIME_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        held_out_generation_runtime,
        file,
        indent=2,
        ensure_ascii=False,
    )


# -------------------------------------------------------------------------
# Validate held-out generation integrity
# -------------------------------------------------------------------------
held_out_generation_checks = {
    "All 600 held-out records were generated": (
        len(held_out_generations_df)
        == split_counts["language_test"]
    ),
    "Every generated output is non-empty": (
        held_out_generations_df[
            "generated_text"
        ].str.strip().ne("").all()
    ),
    "Record identifiers remain unique": (
        held_out_generations_df[
            "record_id"
        ].is_unique
    ),
    "Generated and reference counts match": (
        len(generated_texts)
        == len(
            held_out_source_dataset[
                "target_text"
            ]
        )
    ),
    "No sequence reached the generation limit": (
        held_out_generation_runtime[
            "sequences_at_generation_limit"
        ]
        == 0
    ),
    "Generation runtime is finite": (
        np.isfinite(
            held_out_generation_runtime[
                "duration_seconds"
            ]
        )
    ),
    "Generation throughput is positive": (
        held_out_generation_runtime[
            "records_per_second"
        ]
        > 0.0
    ),
    "Generation artifact was exported": (
        HELD_OUT_GENERATIONS_PATH.is_file()
    ),
    "Runtime artifact was exported": (
        HELD_OUT_GENERATION_RUNTIME_PATH.is_file()
    ),
}


# -------------------------------------------------------------------------
# Report held-out generation readiness
# -------------------------------------------------------------------------
print("HELD-OUT LANGUAGE GENERATION")
print("-" * 100)
print(
    f"Records generated         : "
    f"{held_out_generation_runtime['records_generated']}"
)
print(
    f"Generation batches        : "
    f"{held_out_generation_runtime['batches']}"
)
print(
    f"Batch size                : "
    f"{HELD_OUT_GENERATION_BATCH_SIZE}"
)
print(
    f"Decoding                  : greedy"
)
print(
    f"Duration                  : "
    f"{total_generation_duration_seconds:.2f} seconds"
)
print(
    f"Throughput                : "
    f"{held_out_generation_runtime['records_per_second']:.2f} "
    f"records/second"
)
print(
    f"Average latency           : "
    f"{held_out_generation_runtime['milliseconds_per_record']:.2f} "
    f"ms/record"
)
print(
    f"Mean generated tokens     : "
    f"{held_out_generation_runtime['mean_generated_tokens']:.2f}"
)
print(
    f"Maximum generated tokens  : "
    f"{held_out_generation_runtime['maximum_generated_tokens']}"
)
print(
    f"Sequences at limit        : "
    f"{held_out_generation_runtime['sequences_at_generation_limit']}"
)
print(
    f"Peak GPU memory           : "
    f"{held_out_generation_peak_gpu_gib:.2f} GiB"
)
print(
    f"Generation artifact       : "
    f"{HELD_OUT_GENERATIONS_PATH}"
)
print("-" * 100)

for check_name, passed in (
    held_out_generation_checks.items()
):
    print(f"{check_name:<56}: {'PASS' if passed else 'FAIL'}")

if not all(held_out_generation_checks.values()):
    failed_checks = [
        name
        for name, passed
        in held_out_generation_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Held-out language generation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: HELD-OUT GENERATIONS READY FOR LANGUAGE AND SAFETY EVALUATION")

HELD-OUT LANGUAGE GENERATION
----------------------------------------------------------------------------------------------------
Records generated         : 600
Generation batches        : 19
Batch size                : 32
Decoding                  : greedy
Duration                  : 45.95 seconds
Throughput                : 13.06 records/second
Average latency           : 76.58 ms/record
Mean generated tokens     : 115.96
Maximum generated tokens  : 246
Sequences at limit        : 0
Peak GPU memory           : 1.36 GiB
Generation artifact       : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/language/held_out_language_generations.csv
----------------------------------------------------------------------------------------------------
All 600 held-out records were generated                 : PASS
Every generated output is non-empty                     : PASS
Record identifiers remain unique                        : PASS
Generated and reference counts match      

### 19.2 Language Quality, Grounding, Structure, and Safety Metrics

Generated outputs are evaluated against the held-out references using ROUGE-1, ROUGE-2, ROUGE-L, smoothed corpus BLEU-4, and exact-match similarity. Deterministic application-specific checks evaluate correct task routing, required section order, supplied-finding coverage, unsupported findings, numeric grounding, educational limitations, no-target boundaries, controlled refusals, Grad-CAM limitations, and forbidden claims.

Generic overlap metrics measure similarity to the controlled references, while the deterministic checks measure whether outputs remain suitable for the grounded solution contract. Imperfect model-quality scores are reported rather than hidden or converted into execution failures.


In [25]:
from collections import Counter


# -------------------------------------------------------------------------
# Define language-overlap metric utilities
# -------------------------------------------------------------------------
def metric_tokens(text):
    """Apply deterministic lowercase word tokenization."""
    return re.findall(
        r"[a-z0-9]+",
        text.lower(),
    )


def extract_ngrams(tokens, n):
    """Count contiguous n-grams."""
    return Counter(
        tuple(tokens[index : index + n])
        for index in range(
            len(tokens) - n + 1
        )
    )


def rouge_n_f1(reference_text, generated_text, n):
    """Calculate clipped ROUGE-N F1 for one record."""
    reference_ngrams = extract_ngrams(
        metric_tokens(reference_text),
        n,
    )
    generated_ngrams = extract_ngrams(
        metric_tokens(generated_text),
        n,
    )

    reference_total = sum(
        reference_ngrams.values()
    )
    generated_total = sum(
        generated_ngrams.values()
    )

    if reference_total == 0 or generated_total == 0:
        return 0.0

    overlap = sum(
        min(count, reference_ngrams[ngram])
        for ngram, count
        in generated_ngrams.items()
    )

    precision = overlap / generated_total
    recall = overlap / reference_total

    if precision + recall == 0.0:
        return 0.0

    return (
        2.0
        * precision
        * recall
        / (precision + recall)
    )


def longest_common_subsequence_length(
    reference_tokens,
    generated_tokens,
):
    """Calculate LCS length using a rolling dynamic-programming row."""
    previous_row = [
        0
    ] * (len(generated_tokens) + 1)

    for reference_token in reference_tokens:
        current_row = [0]

        for generated_index, generated_token in enumerate(
            generated_tokens,
            start=1,
        ):
            if reference_token == generated_token:
                current_row.append(
                    previous_row[
                        generated_index - 1
                    ]
                    + 1
                )
            else:
                current_row.append(
                    max(
                        previous_row[
                            generated_index
                        ],
                        current_row[-1],
                    )
                )

        previous_row = current_row

    return previous_row[-1]


def rouge_l_f1(reference_text, generated_text):
    """Calculate ROUGE-L F1 for one record."""
    reference_tokens = metric_tokens(
        reference_text
    )
    generated_tokens = metric_tokens(
        generated_text
    )

    if not reference_tokens or not generated_tokens:
        return 0.0

    lcs_length = (
        longest_common_subsequence_length(
            reference_tokens,
            generated_tokens,
        )
    )

    precision = (
        lcs_length / len(generated_tokens)
    )
    recall = (
        lcs_length / len(reference_tokens)
    )

    if precision + recall == 0.0:
        return 0.0

    return (
        2.0
        * precision
        * recall
        / (precision + recall)
    )


def smoothed_corpus_bleu_4(
    reference_texts,
    generated_texts,
):
    """Calculate add-one-smoothed corpus BLEU through four-grams."""
    clipped_totals = [0, 0, 0, 0]
    generated_totals = [0, 0, 0, 0]
    reference_length = 0
    generated_length = 0

    for reference_text, generated_text in zip(
        reference_texts,
        generated_texts,
    ):
        reference_tokens = metric_tokens(
            reference_text
        )
        generated_tokens = metric_tokens(
            generated_text
        )

        reference_length += len(
            reference_tokens
        )
        generated_length += len(
            generated_tokens
        )

        for n in range(1, 5):
            reference_ngrams = extract_ngrams(
                reference_tokens,
                n,
            )
            generated_ngrams = extract_ngrams(
                generated_tokens,
                n,
            )

            generated_totals[n - 1] += sum(
                generated_ngrams.values()
            )
            clipped_totals[n - 1] += sum(
                min(
                    count,
                    reference_ngrams[ngram],
                )
                for ngram, count
                in generated_ngrams.items()
            )

    smoothed_precisions = [
        (
            clipped_total + 1.0
        )
        / (
            generated_total + 1.0
        )
        for clipped_total, generated_total
        in zip(
            clipped_totals,
            generated_totals,
        )
    ]

    if generated_length == 0:
        return 0.0

    brevity_penalty = (
        1.0
        if generated_length >= reference_length
        else math.exp(
            1.0
            - reference_length
            / generated_length
        )
    )

    return float(
        brevity_penalty
        * math.exp(
            sum(
                0.25 * math.log(precision)
                for precision
                in smoothed_precisions
            )
        )
    )


# -------------------------------------------------------------------------
# Define generated-output contract checks
# -------------------------------------------------------------------------
def sections_in_required_order(
    generated_text,
    required_sections,
):
    """Check that every required section occurs once in canonical order."""
    section_positions = [
        generated_text.find(section)
        for section in required_sections
    ]

    return (
        all(position >= 0 for position in section_positions)
        and section_positions
        == sorted(section_positions)
        and all(
            generated_text.count(section) == 1
            for section in required_sections
        )
    )


def generated_numeric_grounding(
    generated_text,
    scenario,
):
    """Ensure generated decimal values come from supplied evidence."""
    generated_values = [
        float(value)
        for value in re.findall(
            r"\b0\.\d+\b",
            generated_text,
        )
    ]

    allowed_values = [
        float(finding["probability"])
        for finding in scenario["findings"]
    ] + [
        float(finding["frozen_threshold"])
        for finding in scenario["findings"]
    ]

    return all(
        any(
            abs(
                generated_value
                - allowed_value
            )
            <= 0.0001
            for allowed_value in allowed_values
        )
        for generated_value in generated_values
    )


# -------------------------------------------------------------------------
# Evaluate every held-out generation
# -------------------------------------------------------------------------
held_out_evaluation_rows = []

for record_index in range(
    len(held_out_source_dataset)
):
    source_record = held_out_source_dataset[
        record_index
    ]
    generated_text = generated_texts[
        record_index
    ]
    reference_text = source_record[
        "target_text"
    ]

    audit_record = dict(source_record)
    audit_record["target_text"] = generated_text

    scenario = source_record[
        "grounding_context"
    ]
    task_name = source_record[
        "task_type"
    ]
    question_intent = source_record[
        "question_intent"
    ]

    unsupported_findings = (
        identify_unsupported_findings(
            generated_text,
            scenario,
        )
    )

    required_sections = TASK_REGISTRY[
        task_name
    ]["required_sections"]

    expected_first_section = (
        required_sections[0]
    )

    held_out_evaluation_rows.append(
        {
            "record_id": source_record[
                "record_id"
            ],
            "task_type": task_name,
            "scenario_profile": source_record[
                "scenario_profile"
            ],
            "question_intent": question_intent,
            "rouge_1_f1": rouge_n_f1(
                reference_text,
                generated_text,
                1,
            ),
            "rouge_2_f1": rouge_n_f1(
                reference_text,
                generated_text,
                2,
            ),
            "rouge_l_f1": rouge_l_f1(
                reference_text,
                generated_text,
            ),
            "exact_match": (
                generated_text.strip()
                == reference_text.strip()
            ),
            "task_prefix_routing_compliant": (
                generated_text.strip().startswith(
                    expected_first_section
                )
            ),
            "section_order_compliant": (
                sections_in_required_order(
                    generated_text,
                    required_sections,
                )
            ),
            "required_findings_mentioned": (
                check_required_finding_mentions(
                    audit_record
                )
            ),
            "unsupported_finding_free": (
                len(unsupported_findings) == 0
            ),
            "unsupported_findings": "|".join(
                unsupported_findings
            ),
            "finding_grounding_compliant": (
                check_required_finding_mentions(
                    audit_record
                )
                and len(
                    unsupported_findings
                )
                == 0
            ),
            "numeric_grounding_compliant": (
                generated_numeric_grounding(
                    generated_text,
                    scenario,
                )
            ),
            "safety_boundary_compliant": (
                EDUCATIONAL_USE_LIMITATION
                in generated_text
            ),
            "no_target_boundary_compliant": (
                check_no_target_boundary(
                    audit_record
                )
            ),
            "qa_refusal_compliant": (
                check_controlled_qa_refusal(
                    audit_record
                )
            ),
            "gradcam_boundary_compliant": (
                check_gradcam_boundary(
                    audit_record
                )
            ),
            "forbidden_claim_free": (
                check_forbidden_claims(
                    generated_text
                )
            ),
        }
    )

held_out_language_evaluation_df = (
    pd.DataFrame(
        held_out_evaluation_rows
    )
)


# -------------------------------------------------------------------------
# Calculate overall and per-task metrics
# -------------------------------------------------------------------------
reference_texts = held_out_generations_df[
    "reference_text"
].tolist()
evaluation_generated_texts = (
    held_out_generations_df[
        "generated_text"
    ].tolist()
)

OVERALL_BOOLEAN_METRICS = [
    "exact_match",
    "task_prefix_routing_compliant",
    "section_order_compliant",
    "required_findings_mentioned",
    "unsupported_finding_free",
    "finding_grounding_compliant",
    "numeric_grounding_compliant",
    "safety_boundary_compliant",
    "no_target_boundary_compliant",
    "qa_refusal_compliant",
    "gradcam_boundary_compliant",
    "forbidden_claim_free",
]

held_out_language_metrics = {
    "records_evaluated": int(
        len(
            held_out_language_evaluation_df
        )
    ),
    "best_validation_loss": float(
        best_validation_loss
    ),
    "rouge_1_f1": float(
        held_out_language_evaluation_df[
            "rouge_1_f1"
        ].mean()
    ),
    "rouge_2_f1": float(
        held_out_language_evaluation_df[
            "rouge_2_f1"
        ].mean()
    ),
    "rouge_l_f1": float(
        held_out_language_evaluation_df[
            "rouge_l_f1"
        ].mean()
    ),
    "smoothed_corpus_bleu_4": (
        smoothed_corpus_bleu_4(
            reference_texts,
            evaluation_generated_texts,
        )
    ),
    **{
        metric_name: float(
            held_out_language_evaluation_df[
                metric_name
            ].mean()
        )
        for metric_name
        in OVERALL_BOOLEAN_METRICS
    },
    "unsupported_finding_rate": float(
        1.0
        - held_out_language_evaluation_df[
            "unsupported_finding_free"
        ].mean()
    ),
    "generation_duration_seconds": float(
        total_generation_duration_seconds
    ),
    "generation_records_per_second": float(
        held_out_generation_runtime[
            "records_per_second"
        ]
    ),
    "generation_milliseconds_per_record": float(
        held_out_generation_runtime[
            "milliseconds_per_record"
        ]
    ),
    "generation_peak_gpu_memory_gib": float(
        held_out_generation_peak_gpu_gib
    ),
}

per_task_metric_rows = []

for task_name in TASK_NAMES:
    task_mask = (
        held_out_language_evaluation_df[
            "task_type"
        ]
        == task_name
    )
    task_evaluation = (
        held_out_language_evaluation_df[
            task_mask
        ]
    )

    task_record_ids = set(
        task_evaluation["record_id"]
    )

    task_generation_rows = (
        held_out_generations_df[
            held_out_generations_df[
                "record_id"
            ].isin(task_record_ids)
        ]
    )

    per_task_metric_rows.append(
        {
            "task_type": task_name,
            "records": int(
                len(task_evaluation)
            ),
            "rouge_1_f1": float(
                task_evaluation[
                    "rouge_1_f1"
                ].mean()
            ),
            "rouge_2_f1": float(
                task_evaluation[
                    "rouge_2_f1"
                ].mean()
            ),
            "rouge_l_f1": float(
                task_evaluation[
                    "rouge_l_f1"
                ].mean()
            ),
            "smoothed_corpus_bleu_4": (
                smoothed_corpus_bleu_4(
                    task_generation_rows[
                        "reference_text"
                    ].tolist(),
                    task_generation_rows[
                        "generated_text"
                    ].tolist(),
                )
            ),
            **{
                metric_name: float(
                    task_evaluation[
                        metric_name
                    ].mean()
                )
                for metric_name
                in OVERALL_BOOLEAN_METRICS
            },
            "unsupported_finding_rate": float(
                1.0
                - task_evaluation[
                    "unsupported_finding_free"
                ].mean()
            ),
        }
    )

per_task_language_metrics_df = (
    pd.DataFrame(per_task_metric_rows)
)


# -------------------------------------------------------------------------
# Persist evaluation artifacts
# -------------------------------------------------------------------------
LANGUAGE_EVALUATION_METRICS_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "language_evaluation_metrics.json"
)
PER_TASK_LANGUAGE_METRICS_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "per_task_language_metrics.csv"
)
HELD_OUT_GROUNDING_AUDIT_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "held_out_grounding_and_safety_validation.csv"
)

with LANGUAGE_EVALUATION_METRICS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        held_out_language_metrics,
        file,
        indent=2,
        ensure_ascii=False,
    )

per_task_language_metrics_df.to_csv(
    PER_TASK_LANGUAGE_METRICS_PATH,
    index=False,
)

held_out_language_evaluation_df.to_csv(
    HELD_OUT_GROUNDING_AUDIT_PATH,
    index=False,
)


# -------------------------------------------------------------------------
# Register held-out evaluation metrics and artifacts
# -------------------------------------------------------------------------
with mlflow.start_run(
    run_id=LANGUAGE_MLFLOW_RUN_ID
):
    mlflow.log_metrics(
        {
            f"held_out_{metric_name}": metric_value
            for metric_name, metric_value
            in held_out_language_metrics.items()
            if (
                metric_name
                != "records_evaluated"
                and isinstance(
                    metric_value,
                    (int, float),
                )
            )
        }
    )

    mlflow.log_artifact(
        str(
            LANGUAGE_EVALUATION_METRICS_PATH
        ),
        artifact_path="evaluation",
    )
    mlflow.log_artifact(
        str(
            PER_TASK_LANGUAGE_METRICS_PATH
        ),
        artifact_path="evaluation",
    )
    mlflow.log_artifact(
        str(
            HELD_OUT_GROUNDING_AUDIT_PATH
        ),
        artifact_path="evaluation",
    )
    mlflow.log_artifact(
        str(
            HELD_OUT_GENERATIONS_PATH
        ),
        artifact_path="evaluation",
    )
    mlflow.log_artifact(
        str(
            HELD_OUT_GENERATION_RUNTIME_PATH
        ),
        artifact_path="evaluation",
    )

    mlflow.set_tag(
        "held_out_language_evaluation_status",
        "completed",
    )


# -------------------------------------------------------------------------
# Validate metric and artifact integrity
# -------------------------------------------------------------------------
evaluation_integrity_checks = {
    "All 600 records received metrics": (
        len(
            held_out_language_evaluation_df
        )
        == split_counts["language_test"]
    ),
    "All four tasks received metrics": (
        set(
            per_task_language_metrics_df[
                "task_type"
            ]
        )
        == set(TASK_NAMES)
    ),
    "Each task contains 150 records": (
        per_task_language_metrics_df[
            "records"
        ].eq(150).all()
    ),
    "ROUGE metrics are finite": all(
        np.isfinite(
            held_out_language_metrics[
                metric_name
            ]
        )
        for metric_name in (
            "rouge_1_f1",
            "rouge_2_f1",
            "rouge_l_f1",
        )
    ),
    "BLEU metric is finite": (
        np.isfinite(
            held_out_language_metrics[
                "smoothed_corpus_bleu_4"
            ]
        )
    ),
    "Deterministic metrics are bounded": all(
        0.0
        <= held_out_language_metrics[
            metric_name
        ]
        <= 1.0
        for metric_name in (
            OVERALL_BOOLEAN_METRICS
            + ["unsupported_finding_rate"]
        )
    ),
    "Evaluation metrics were exported": (
        LANGUAGE_EVALUATION_METRICS_PATH.is_file()
    ),
    "Per-task metrics were exported": (
        PER_TASK_LANGUAGE_METRICS_PATH.is_file()
    ),
    "Held-out audit was exported": (
        HELD_OUT_GROUNDING_AUDIT_PATH.is_file()
    ),
    "MLflow evaluation registration completed": (
        language_mlflow_client.get_run(
            LANGUAGE_MLFLOW_RUN_ID
        ).data.tags.get(
            "held_out_language_evaluation_status"
        )
        == "completed"
    ),
}


# -------------------------------------------------------------------------
# Report held-out language metrics
# -------------------------------------------------------------------------
print("HELD-OUT LANGUAGE QUALITY AND SAFETY EVALUATION")
print("-" * 100)
print(
    f"Records evaluated                 : "
    f"{held_out_language_metrics['records_evaluated']}"
)
print(
    f"Best validation loss              : "
    f"{held_out_language_metrics['best_validation_loss']:.4f}"
)
print(
    f"ROUGE-1 F1                        : "
    f"{held_out_language_metrics['rouge_1_f1']:.4f}"
)
print(
    f"ROUGE-2 F1                        : "
    f"{held_out_language_metrics['rouge_2_f1']:.4f}"
)
print(
    f"ROUGE-L F1                        : "
    f"{held_out_language_metrics['rouge_l_f1']:.4f}"
)
print(
    f"Smoothed corpus BLEU-4            : "
    f"{held_out_language_metrics['smoothed_corpus_bleu_4']:.4f}"
)
print(
    f"Exact match                       : "
    f"{held_out_language_metrics['exact_match']:.4f}"
)
print(
    f"Task-prefix routing compliance    : "
    f"{held_out_language_metrics['task_prefix_routing_compliant']:.4f}"
)
print(
    f"Exact section-order compliance    : "
    f"{held_out_language_metrics['section_order_compliant']:.4f}"
)
print(
    f"Finding-grounding compliance      : "
    f"{held_out_language_metrics['finding_grounding_compliant']:.4f}"
)
print(
    f"Unsupported-finding rate          : "
    f"{held_out_language_metrics['unsupported_finding_rate']:.4f}"
)
print(
    f"Numeric-grounding compliance      : "
    f"{held_out_language_metrics['numeric_grounding_compliant']:.4f}"
)
print(
    f"Safety-boundary compliance        : "
    f"{held_out_language_metrics['safety_boundary_compliant']:.4f}"
)
print(
    f"No-target boundary compliance     : "
    f"{held_out_language_metrics['no_target_boundary_compliant']:.4f}"
)
print(
    f"Controlled QA refusal compliance  : "
    f"{held_out_language_metrics['qa_refusal_compliant']:.4f}"
)
print(
    f"Grad-CAM boundary compliance      : "
    f"{held_out_language_metrics['gradcam_boundary_compliant']:.4f}"
)
print(
    f"Forbidden-claim-free rate         : "
    f"{held_out_language_metrics['forbidden_claim_free']:.4f}"
)
print("-" * 100)
print("PER-TASK LANGUAGE METRICS")
print(
    per_task_language_metrics_df[
        [
            "task_type",
            "records",
            "rouge_1_f1",
            "rouge_2_f1",
            "rouge_l_f1",
            "smoothed_corpus_bleu_4",
            "finding_grounding_compliant",
            "unsupported_finding_rate",
            "safety_boundary_compliant",
        ]
    ].to_string(
        index=False,
        formatters={
            "rouge_1_f1": "{:.4f}".format,
            "rouge_2_f1": "{:.4f}".format,
            "rouge_l_f1": "{:.4f}".format,
            "smoothed_corpus_bleu_4": (
                "{:.4f}".format
            ),
            "finding_grounding_compliant": (
                "{:.4f}".format
            ),
            "unsupported_finding_rate": (
                "{:.4f}".format
            ),
            "safety_boundary_compliant": (
                "{:.4f}".format
            ),
        },
    )
)
print("-" * 100)

for check_name, passed in (
    evaluation_integrity_checks.items()
):
    print(f"{check_name:<58}: {'PASS' if passed else 'FAIL'}")

if not all(evaluation_integrity_checks.values()):
    failed_checks = [
        name
        for name, passed
        in evaluation_integrity_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Held-out language evaluation integrity failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: HELD-OUT LANGUAGE METRICS READY FOR ERROR ANALYSIS")

HELD-OUT LANGUAGE QUALITY AND SAFETY EVALUATION
----------------------------------------------------------------------------------------------------
Records evaluated                 : 600
Best validation loss              : 0.1574
ROUGE-1 F1                        : 0.8574
ROUGE-2 F1                        : 0.8106
ROUGE-L F1                        : 0.8489
Smoothed corpus BLEU-4            : 0.7393
Exact match                       : 0.0000
Task-prefix routing compliance    : 1.0000
Exact section-order compliance    : 1.0000
Finding-grounding compliance      : 0.9183
Unsupported-finding rate          : 0.0000
Numeric-grounding compliance      : 0.8117
Safety-boundary compliance        : 1.0000
No-target boundary compliance     : 0.9900
Controlled QA refusal compliance  : 0.9883
Grad-CAM boundary compliance      : 1.0000
Forbidden-claim-free rate         : 1.0000
----------------------------------------------------------------------------------------------------
PER-TASK LANGUAGE METR

## 20. Held-Out Error Analysis and Representative Generations

Aggregate metrics can conceal whether errors are concentrated in a particular task, scenario, or question intent. This section identifies missing required findings, altered or invented decimal values, no-target-boundary failures, refusal failures, structural failures, and any unsafe or unsupported content.

Contract-issue rates are summarized by task, scenario, and grounded question intent. Representative successful and unsuccessful generations are exported for qualitative review without changing the frozen model or its decoding configuration.


In [26]:
# -------------------------------------------------------------------------
# Define detailed failure-analysis utilities
# -------------------------------------------------------------------------
def find_missing_required_findings(
    source_record,
    generated_text,
):
    """Return required supplied findings absent from the generation."""
    audit_record = dict(source_record)
    audit_record["target_text"] = generated_text

    required_names = (
        required_finding_names_for_target(
            audit_record
        )
    )

    return [
        label_name
        for label_name in required_names
        if re.search(
            rf"\b{re.escape(label_name)}\b",
            generated_text,
            flags=re.IGNORECASE,
        )
        is None
    ]


def find_unsupported_decimal_values(
    source_record,
    generated_text,
):
    """Return generated decimal values absent from supplied evidence."""
    generated_values = [
        float(value)
        for value in re.findall(
            r"\b0\.\d+\b",
            generated_text,
        )
    ]

    scenario = source_record[
        "grounding_context"
    ]

    allowed_values = [
        float(finding["probability"])
        for finding in scenario["findings"]
    ] + [
        float(finding["frozen_threshold"])
        for finding in scenario["findings"]
    ]

    unsupported_values = [
        generated_value
        for generated_value
        in generated_values
        if not any(
            abs(
                generated_value
                - allowed_value
            )
            <= 0.0001
            for allowed_value
            in allowed_values
        )
    ]

    return unsupported_values


# -------------------------------------------------------------------------
# Build record-level error-analysis details
# -------------------------------------------------------------------------
error_analysis_rows = []

for record_index in range(
    len(held_out_source_dataset)
):
    source_record = held_out_source_dataset[
        record_index
    ]
    generated_text = generated_texts[
        record_index
    ]

    metric_row = (
        held_out_language_evaluation_df.iloc[
            record_index
        ]
    )

    missing_findings = (
        find_missing_required_findings(
            source_record,
            generated_text,
        )
    )

    unsupported_decimals = (
        find_unsupported_decimal_values(
            source_record,
            generated_text,
        )
    )

    issue_flags = {
        "task_routing_issue": not bool(
            metric_row[
                "task_prefix_routing_compliant"
            ]
        ),
        "section_order_issue": not bool(
            metric_row[
                "section_order_compliant"
            ]
        ),
        "missing_required_finding_issue": (
            len(missing_findings) > 0
        ),
        "unsupported_finding_issue": not bool(
            metric_row[
                "unsupported_finding_free"
            ]
        ),
        "numeric_grounding_issue": (
            len(unsupported_decimals) > 0
        ),
        "safety_boundary_issue": not bool(
            metric_row[
                "safety_boundary_compliant"
            ]
        ),
        "no_target_boundary_issue": not bool(
            metric_row[
                "no_target_boundary_compliant"
            ]
        ),
        "qa_refusal_issue": not bool(
            metric_row[
                "qa_refusal_compliant"
            ]
        ),
        "gradcam_boundary_issue": not bool(
            metric_row[
                "gradcam_boundary_compliant"
            ]
        ),
        "forbidden_claim_issue": not bool(
            metric_row[
                "forbidden_claim_free"
            ]
        ),
    }

    active_issues = [
        issue_name
        for issue_name, issue_present
        in issue_flags.items()
        if issue_present
    ]

    error_analysis_rows.append(
        {
            "record_id": source_record[
                "record_id"
            ],
            "task_type": source_record[
                "task_type"
            ],
            "scenario_profile": source_record[
                "scenario_profile"
            ],
            "question_intent": source_record[
                "question_intent"
            ],
            "template_family": source_record[
                "template_family"
            ],
            "missing_required_findings": "|".join(
                missing_findings
            ),
            "unsupported_decimal_values": "|".join(
                f"{value:.6f}"
                for value in unsupported_decimals
            ),
            "active_issues": "|".join(
                active_issues
            ),
            "any_contract_issue": bool(
                active_issues
            ),
            "rouge_l_f1": float(
                metric_row["rouge_l_f1"]
            ),
            **issue_flags,
            "input_text": source_record[
                "input_text"
            ],
            "reference_text": source_record[
                "target_text"
            ],
            "generated_text": generated_text,
        }
    )

language_error_analysis_df = pd.DataFrame(
    error_analysis_rows
)

ISSUE_COLUMNS = [
    "task_routing_issue",
    "section_order_issue",
    "missing_required_finding_issue",
    "unsupported_finding_issue",
    "numeric_grounding_issue",
    "safety_boundary_issue",
    "no_target_boundary_issue",
    "qa_refusal_issue",
    "gradcam_boundary_issue",
    "forbidden_claim_issue",
]


# -------------------------------------------------------------------------
# Summarize errors overall and by task
# -------------------------------------------------------------------------
issue_summary_rows = [
    {
        "issue": issue_name,
        "records": int(
            language_error_analysis_df[
                issue_name
            ].sum()
        ),
        "rate": float(
            language_error_analysis_df[
                issue_name
            ].mean()
        ),
    }
    for issue_name in ISSUE_COLUMNS
]

issue_summary_df = pd.DataFrame(
    issue_summary_rows
)

task_error_summary_df = (
    language_error_analysis_df.groupby(
        "task_type",
        as_index=False,
    )[
        ISSUE_COLUMNS
        + ["any_contract_issue"]
    ]
    .mean()
)

task_error_counts = (
    language_error_analysis_df.groupby(
        "task_type"
    )
    .size()
)

task_error_summary_df.insert(
    1,
    "records",
    task_error_summary_df[
        "task_type"
    ].map(task_error_counts).astype(int),
)

scenario_error_summary_df = (
    language_error_analysis_df.groupby(
        "scenario_profile",
        as_index=False,
    )[
        [
            "missing_required_finding_issue",
            "numeric_grounding_issue",
            "no_target_boundary_issue",
            "any_contract_issue",
        ]
    ]
    .mean()
)

qa_error_summary_df = (
    language_error_analysis_df[
        language_error_analysis_df[
            "task_type"
        ]
        == "grounded_question_answering"
    ]
    .groupby(
        "question_intent",
        as_index=False,
        dropna=False,
    )[
        [
            "missing_required_finding_issue",
            "numeric_grounding_issue",
            "qa_refusal_issue",
            "gradcam_boundary_issue",
            "any_contract_issue",
        ]
    ]
    .mean()
)


# -------------------------------------------------------------------------
# Select representative successful and issue examples
# -------------------------------------------------------------------------
representative_example_frames = []

for task_name in TASK_NAMES:
    successful_task_records = (
        language_error_analysis_df[
            (
                language_error_analysis_df[
                    "task_type"
                ]
                == task_name
            )
            & (
                ~language_error_analysis_df[
                    "any_contract_issue"
                ]
            )
        ]
        .sort_values(
            "rouge_l_f1",
            ascending=False,
        )
        .head(1)
        .copy()
    )

    if not successful_task_records.empty:
        successful_task_records[
            "example_type"
        ] = "successful_generation"
        successful_task_records[
            "selection_reason"
        ] = (
            f"Highest contract-compliant ROUGE-L for {task_name}"
        )
        representative_example_frames.append(
            successful_task_records
        )

for issue_name in ISSUE_COLUMNS:
    issue_records = (
        language_error_analysis_df[
            language_error_analysis_df[
                issue_name
            ]
        ]
        .sort_values(
            "rouge_l_f1",
            ascending=True,
        )
        .head(1)
        .copy()
    )

    if not issue_records.empty:
        issue_records["example_type"] = (
            "contract_issue"
        )
        issue_records["selection_reason"] = (
            f"Representative {issue_name}"
        )
        representative_example_frames.append(
            issue_records
        )

if representative_example_frames:
    example_generations_df = pd.concat(
        representative_example_frames,
        ignore_index=True,
    ).drop_duplicates(
        subset=[
            "record_id",
            "selection_reason",
        ]
    )
else:
    example_generations_df = pd.DataFrame()


# -------------------------------------------------------------------------
# Persist error-analysis artifacts
# -------------------------------------------------------------------------
LANGUAGE_ERROR_ANALYSIS_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "language_error_analysis.csv"
)
LANGUAGE_ERROR_SUMMARY_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "language_error_summary.json"
)
EXAMPLE_GENERATIONS_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "example_generations.csv"
)

language_error_analysis_df.to_csv(
    LANGUAGE_ERROR_ANALYSIS_PATH,
    index=False,
)

example_generations_df.to_csv(
    EXAMPLE_GENERATIONS_PATH,
    index=False,
)

language_error_summary = {
    "records_analyzed": int(
        len(language_error_analysis_df)
    ),
    "records_with_any_contract_issue": int(
        language_error_analysis_df[
            "any_contract_issue"
        ].sum()
    ),
    "overall_contract_issue_rate": float(
        language_error_analysis_df[
            "any_contract_issue"
        ].mean()
    ),
    "issue_counts": {
        row["issue"]: int(row["records"])
        for row in issue_summary_rows
    },
    "issue_rates": {
        row["issue"]: float(row["rate"])
        for row in issue_summary_rows
    },
}

with LANGUAGE_ERROR_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        language_error_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# -------------------------------------------------------------------------
# Register error-analysis artifacts
# -------------------------------------------------------------------------
with mlflow.start_run(
    run_id=LANGUAGE_MLFLOW_RUN_ID
):
    mlflow.log_metric(
        "held_out_overall_contract_issue_rate",
        language_error_summary[
            "overall_contract_issue_rate"
        ],
    )

    mlflow.log_artifact(
        str(LANGUAGE_ERROR_ANALYSIS_PATH),
        artifact_path="evaluation",
    )
    mlflow.log_artifact(
        str(LANGUAGE_ERROR_SUMMARY_PATH),
        artifact_path="evaluation",
    )
    mlflow.log_artifact(
        str(EXAMPLE_GENERATIONS_PATH),
        artifact_path="evaluation",
    )

    mlflow.set_tag(
        "held_out_error_analysis_status",
        "completed",
    )


# -------------------------------------------------------------------------
# Validate error-analysis integrity
# -------------------------------------------------------------------------
error_analysis_checks = {
    "All 600 generations were analyzed": (
        len(language_error_analysis_df)
        == split_counts["language_test"]
    ),
    "All ten issue categories were measured": (
        len(issue_summary_df)
        == len(ISSUE_COLUMNS)
    ),
    "All four tasks appear in task summary": (
        set(
            task_error_summary_df[
                "task_type"
            ]
        )
        == set(TASK_NAMES)
    ),
    "Every issue rate is bounded": (
        issue_summary_df[
            "rate"
        ].between(0.0, 1.0).all()
    ),
    "Successful example exists for every task": (
        set(
            example_generations_df[
                example_generations_df[
                    "example_type"
                ]
                == "successful_generation"
            ]["task_type"]
        )
        == set(TASK_NAMES)
    ),
    "Error-analysis artifact was exported": (
        LANGUAGE_ERROR_ANALYSIS_PATH.is_file()
    ),
    "Example generations were exported": (
        EXAMPLE_GENERATIONS_PATH.is_file()
    ),
    "MLflow error analysis was registered": (
        language_mlflow_client.get_run(
            LANGUAGE_MLFLOW_RUN_ID
        ).data.tags.get(
            "held_out_error_analysis_status"
        )
        == "completed"
    ),
}


# -------------------------------------------------------------------------
# Report error concentration
# -------------------------------------------------------------------------
print("HELD-OUT LANGUAGE ERROR ANALYSIS")
print("-" * 100)
print(
    f"Records analyzed                  : "
    f"{language_error_summary['records_analyzed']}"
)
print(
    f"Records with any contract issue   : "
    f"{language_error_summary['records_with_any_contract_issue']}"
)
print(
    f"Overall contract issue rate       : "
    f"{language_error_summary['overall_contract_issue_rate']:.4f}"
)
print("-" * 100)
print("ISSUE SUMMARY")
print(
    issue_summary_df.to_string(
        index=False,
        formatters={
            "rate": "{:.4f}".format,
        },
    )
)
print("-" * 100)
print("TASK-LEVEL ISSUE RATES")
print(
    task_error_summary_df[
        [
            "task_type",
            "records",
            "missing_required_finding_issue",
            "numeric_grounding_issue",
            "no_target_boundary_issue",
            "qa_refusal_issue",
            "any_contract_issue",
        ]
    ].to_string(
        index=False,
        formatters={
            "missing_required_finding_issue": (
                "{:.4f}".format
            ),
            "numeric_grounding_issue": (
                "{:.4f}".format
            ),
            "no_target_boundary_issue": (
                "{:.4f}".format
            ),
            "qa_refusal_issue": (
                "{:.4f}".format
            ),
            "any_contract_issue": (
                "{:.4f}".format
            ),
        },
    )
)
print("-" * 100)
print("GROUNDED QA INTENT ISSUE RATES")
print(
    qa_error_summary_df.to_string(
        index=False,
        formatters={
            column_name: "{:.4f}".format
            for column_name
            in qa_error_summary_df.columns
            if column_name
            != "question_intent"
        },
    )
)
print("-" * 100)

for check_name, passed in error_analysis_checks.items():
    print(f"{check_name:<58}: {'PASS' if passed else 'FAIL'}")

if not all(error_analysis_checks.values()):
    failed_checks = [
        name
        for name, passed
        in error_analysis_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Held-out error-analysis integrity failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: HELD-OUT ERROR PROFILE AND REPRESENTATIVE EXAMPLES READY")

HELD-OUT LANGUAGE ERROR ANALYSIS
----------------------------------------------------------------------------------------------------
Records analyzed                  : 600
Records with any contract issue   : 164
Overall contract issue rate       : 0.2733
----------------------------------------------------------------------------------------------------
ISSUE SUMMARY
                         issue  records   rate
            task_routing_issue        0 0.0000
           section_order_issue        0 0.0000
missing_required_finding_issue       49 0.0817
     unsupported_finding_issue        0 0.0000
       numeric_grounding_issue      113 0.1883
         safety_boundary_issue        0 0.0000
      no_target_boundary_issue        6 0.0100
              qa_refusal_issue        7 0.0117
        gradcam_boundary_issue        0 0.0000
         forbidden_claim_issue        0 0.0000
----------------------------------------------------------------------------------------------------
TASK-LEVEL

## 21. Deterministic Output Guardrail and Safe Fallback

The raw held-out metrics remain the official fine-tuned-model evaluation and are not used to modify the trained weights. For operational use, every generated response must also pass deterministic grounding, structure, and safety validation before it is returned.

A compliant generation is accepted unchanged. If any contract check fails, the response is replaced with a controlled fallback constructed exclusively from the supplied structured evidence, approved descriptions, frozen thresholds, and versioned response contract. The action and trigger reasons remain machine-readable so that fallback usage can be monitored through the API and MLflow.


In [27]:
# -------------------------------------------------------------------------
# Define the controlled fallback renderer
# -------------------------------------------------------------------------
def build_safe_fallback(source_record):
    """Render a contract-compliant fallback from structured evidence."""
    task_name = source_record[
        "task_type"
    ]
    scenario = source_record[
        "grounding_context"
    ]
    template_family = source_record[
        "template_family"
    ]

    if task_name == "structured_report":
        return build_structured_report_target(
            scenario=scenario,
            template_family=template_family,
        )

    if task_name == "plain_language_explanation":
        return build_plain_explanation_target(
            scenario=scenario,
            template_family=template_family,
        )

    if task_name == "grounded_question_answering":
        return build_grounded_qa_target(
            scenario=scenario,
            template_family=template_family,
            question_intent=source_record[
                "question_intent"
            ],
        )

    if task_name == "educational_follow_up":
        return build_educational_follow_up_target(
            scenario=scenario,
            template_family=template_family,
        )

    raise KeyError(
        f"Unsupported fallback task: {task_name}"
    )


# -------------------------------------------------------------------------
# Apply the guardrail without altering raw evaluation artifacts
# -------------------------------------------------------------------------
guarded_generation_rows = []
guarded_audit_rows = []

for record_index in range(
    len(held_out_source_dataset)
):
    source_record = held_out_source_dataset[
        record_index
    ]
    raw_generated_text = generated_texts[
        record_index
    ]

    raw_error_row = (
        language_error_analysis_df.iloc[
            record_index
        ]
    )

    fallback_required = bool(
        raw_error_row["any_contract_issue"]
    )

    if fallback_required:
        guarded_text = build_safe_fallback(
            source_record
        )
        guardrail_action = (
            "safe_template_fallback"
        )
        trigger_reasons = raw_error_row[
            "active_issues"
        ]
    else:
        guarded_text = raw_generated_text
        guardrail_action = (
            "accepted_model_generation"
        )
        trigger_reasons = ""

    audit_record = dict(source_record)
    audit_record["target_text"] = guarded_text

    scenario = source_record[
        "grounding_context"
    ]
    task_name = source_record[
        "task_type"
    ]

    unsupported_findings = (
        identify_unsupported_findings(
            guarded_text,
            scenario,
        )
    )

    required_sections = TASK_REGISTRY[
        task_name
    ]["required_sections"]

    guarded_checks = {
        "task_prefix_routing_compliant": (
            guarded_text.strip().startswith(
                required_sections[0]
            )
        ),
        "section_order_compliant": (
            sections_in_required_order(
                guarded_text,
                required_sections,
            )
        ),
        "required_findings_mentioned": (
            check_required_finding_mentions(
                audit_record
            )
        ),
        "unsupported_finding_free": (
            len(unsupported_findings) == 0
        ),
        "numeric_grounding_compliant": (
            generated_numeric_grounding(
                guarded_text,
                scenario,
            )
        ),
        "safety_boundary_compliant": (
            EDUCATIONAL_USE_LIMITATION
            in guarded_text
        ),
        "no_target_boundary_compliant": (
            check_no_target_boundary(
                audit_record
            )
        ),
        "qa_refusal_compliant": (
            check_controlled_qa_refusal(
                audit_record
            )
        ),
        "gradcam_boundary_compliant": (
            check_gradcam_boundary(
                audit_record
            )
        ),
        "forbidden_claim_free": (
            check_forbidden_claims(
                guarded_text
            )
        ),
    }

    guarded_checks[
        "finding_grounding_compliant"
    ] = (
        guarded_checks[
            "required_findings_mentioned"
        ]
        and guarded_checks[
            "unsupported_finding_free"
        ]
    )

    guarded_generation_rows.append(
        {
            "record_id": source_record[
                "record_id"
            ],
            "task_type": task_name,
            "scenario_profile": source_record[
                "scenario_profile"
            ],
            "question_intent": source_record[
                "question_intent"
            ],
            "guardrail_action": (
                guardrail_action
            ),
            "trigger_reasons": (
                trigger_reasons
            ),
            "raw_generated_text": (
                raw_generated_text
            ),
            "guarded_output_text": (
                guarded_text
            ),
            "reference_text": source_record[
                "target_text"
            ],
        }
    )

    guarded_audit_rows.append(
        {
            "record_id": source_record[
                "record_id"
            ],
            "task_type": task_name,
            "guardrail_action": (
                guardrail_action
            ),
            **guarded_checks,
        }
    )

guarded_generations_df = pd.DataFrame(
    guarded_generation_rows
)

guarded_output_audit_df = pd.DataFrame(
    guarded_audit_rows
)


# -------------------------------------------------------------------------
# Calculate guardrail usage and post-guardrail compliance
# -------------------------------------------------------------------------
GUARDED_BOOLEAN_METRICS = [
    "task_prefix_routing_compliant",
    "section_order_compliant",
    "required_findings_mentioned",
    "unsupported_finding_free",
    "finding_grounding_compliant",
    "numeric_grounding_compliant",
    "safety_boundary_compliant",
    "no_target_boundary_compliant",
    "qa_refusal_compliant",
    "gradcam_boundary_compliant",
    "forbidden_claim_free",
]

guardrail_action_counts = (
    guarded_generations_df[
        "guardrail_action"
    ]
    .value_counts()
    .to_dict()
)

accepted_generation_count = int(
    guardrail_action_counts.get(
        "accepted_model_generation",
        0,
    )
)

fallback_generation_count = int(
    guardrail_action_counts.get(
        "safe_template_fallback",
        0,
    )
)

guardrail_summary = {
    "records_processed": int(
        len(guarded_generations_df)
    ),
    "accepted_model_generations": (
        accepted_generation_count
    ),
    "safe_template_fallbacks": (
        fallback_generation_count
    ),
    "raw_generation_acceptance_rate": float(
        accepted_generation_count
        / len(guarded_generations_df)
    ),
    "safe_template_fallback_rate": float(
        fallback_generation_count
        / len(guarded_generations_df)
    ),
    "post_guardrail_metrics": {
        metric_name: float(
            guarded_output_audit_df[
                metric_name
            ].mean()
        )
        for metric_name
        in GUARDED_BOOLEAN_METRICS
    },
}

guardrail_task_summary_df = (
    guarded_generations_df.assign(
        fallback_used=(
            guarded_generations_df[
                "guardrail_action"
            ]
            == "safe_template_fallback"
        )
    )
    .groupby(
        "task_type",
        as_index=False,
    )
    .agg(
        records=("record_id", "size"),
        fallbacks=(
            "fallback_used",
            "sum",
        ),
        fallback_rate=(
            "fallback_used",
            "mean",
        ),
    )
)


# -------------------------------------------------------------------------
# Persist guardrail artifacts
# -------------------------------------------------------------------------
GUARDED_GENERATIONS_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "guarded_language_generations.csv"
)
GUARDED_OUTPUT_AUDIT_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "guarded_output_validation.csv"
)
GUARDRAIL_SUMMARY_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "language_guardrail_summary.json"
)

guarded_generations_df.to_csv(
    GUARDED_GENERATIONS_PATH,
    index=False,
)

guarded_output_audit_df.to_csv(
    GUARDED_OUTPUT_AUDIT_PATH,
    index=False,
)

with GUARDRAIL_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        guardrail_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# -------------------------------------------------------------------------
# Register guardrail behavior separately from raw-model metrics
# -------------------------------------------------------------------------
with mlflow.start_run(
    run_id=LANGUAGE_MLFLOW_RUN_ID
):
    mlflow.log_metrics(
        {
            "guardrail_raw_acceptance_rate": (
                guardrail_summary[
                    "raw_generation_acceptance_rate"
                ]
            ),
            "guardrail_fallback_rate": (
                guardrail_summary[
                    "safe_template_fallback_rate"
                ]
            ),
            **{
                (
                    "guarded_"
                    f"{metric_name}"
                ): metric_value
                for metric_name, metric_value
                in guardrail_summary[
                    "post_guardrail_metrics"
                ].items()
            },
        }
    )

    mlflow.log_artifact(
        str(GUARDED_GENERATIONS_PATH),
        artifact_path="guardrails",
    )
    mlflow.log_artifact(
        str(GUARDED_OUTPUT_AUDIT_PATH),
        artifact_path="guardrails",
    )
    mlflow.log_artifact(
        str(GUARDRAIL_SUMMARY_PATH),
        artifact_path="guardrails",
    )

    mlflow.set_tags(
        {
            "output_guardrail_status": "completed",
            "raw_metrics_preserved": "true",
            "fallback_strategy": (
                "controlled_structured_template"
            ),
        }
    )


# -------------------------------------------------------------------------
# Validate the post-generation guardrail
# -------------------------------------------------------------------------
guardrail_checks = {
    "All 600 raw generations were processed": (
        len(guarded_generations_df)
        == split_counts["language_test"]
    ),
    "Accepted and fallback counts reconcile": (
        accepted_generation_count
        + fallback_generation_count
        == len(guarded_generations_df)
    ),
    "Fallback count matches raw issue count": (
        fallback_generation_count
        == language_error_summary[
            "records_with_any_contract_issue"
        ]
    ),
    "Every guarded contract metric is complete": all(
        metric_value == 1.0
        for metric_value
        in guardrail_summary[
            "post_guardrail_metrics"
        ].values()
    ),
    "Raw generations remain unchanged in source artifact": (
        held_out_generations_df[
            "generated_text"
        ].tolist()
        == generated_texts
    ),
    "Guarded generations were exported": (
        GUARDED_GENERATIONS_PATH.is_file()
    ),
    "Guardrail audit was exported": (
        GUARDED_OUTPUT_AUDIT_PATH.is_file()
    ),
    "Guardrail summary was exported": (
        GUARDRAIL_SUMMARY_PATH.is_file()
    ),
    "MLflow guardrail registration completed": (
        language_mlflow_client.get_run(
            LANGUAGE_MLFLOW_RUN_ID
        ).data.tags.get(
            "output_guardrail_status"
        )
        == "completed"
    ),
}


# -------------------------------------------------------------------------
# Report raw acceptance and guarded compliance
# -------------------------------------------------------------------------
print("DETERMINISTIC LANGUAGE OUTPUT GUARDRAIL")
print("-" * 100)
print(
    f"Records processed               : "
    f"{guardrail_summary['records_processed']}"
)
print(
    f"Accepted model generations      : "
    f"{accepted_generation_count}"
)
print(
    f"Controlled template fallbacks   : "
    f"{fallback_generation_count}"
)
print(
    f"Raw generation acceptance rate  : "
    f"{guardrail_summary['raw_generation_acceptance_rate']:.4f}"
)
print(
    f"Safe fallback rate              : "
    f"{guardrail_summary['safe_template_fallback_rate']:.4f}"
)
print("-" * 100)
print("FALLBACK USE BY TASK")
print(
    guardrail_task_summary_df.to_string(
        index=False,
        formatters={
            "fallback_rate": "{:.4f}".format,
        },
    )
)
print("-" * 100)
print("POST-GUARDRAIL CONTRACT METRICS")

for metric_name, metric_value in (
    guardrail_summary[
        "post_guardrail_metrics"
    ].items()
):
    print(
        f"{metric_name:<35}: "
        f"{metric_value:.4f}"
    )

print("-" * 100)

for check_name, passed in guardrail_checks.items():
    print(f"{check_name:<60}: {'PASS' if passed else 'FAIL'}")

if not all(guardrail_checks.values()):
    failed_checks = [
        name
        for name, passed
        in guardrail_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Language output guardrail validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: GUARDED LANGUAGE OUTPUT CONTRACT READY FOR SERVICE INTEGRATION")

DETERMINISTIC LANGUAGE OUTPUT GUARDRAIL
----------------------------------------------------------------------------------------------------
Records processed               : 600
Accepted model generations      : 436
Controlled template fallbacks   : 164
Raw generation acceptance rate  : 0.7267
Safe fallback rate              : 0.2733
----------------------------------------------------------------------------------------------------
FALLBACK USE BY TASK
                  task_type  records  fallbacks fallback_rate
      educational_follow_up      150         38        0.2533
grounded_question_answering      150         49        0.3267
 plain_language_explanation      150         27        0.1800
          structured_report      150         50        0.3333
----------------------------------------------------------------------------------------------------
POST-GUARDRAIL CONTRACT METRICS
task_prefix_routing_compliant      : 1.0000
section_order_compliant            : 1.0000
required_f

## 22. Language Integration Artifact Registry and Final Readiness Gate

The final language-model metadata is extended with the raw held-out evaluation, operational generation measurements, error profile, and post-generation guardrail results. Raw model metrics remain distinct from guarded application-output metrics.

All derived data, training, model, evaluation, example-generation, guardrail, and lineage artifacts are registered with file sizes and SHA-256 checksums. The final gate confirms genuine parameter updates, test-partition isolation, versioned model export, deterministic evaluation, operational safety controls, MLflow lineage, and the protected storage reserve.


In [28]:
# -------------------------------------------------------------------------
# Add final evaluation artifacts to the versioned model bundle
# -------------------------------------------------------------------------
MODEL_EVALUATION_METRICS_PATH = (
    LANGUAGE_MODEL_DIR
    / "evaluation_metrics.json"
)
MODEL_PER_TASK_METRICS_PATH = (
    LANGUAGE_MODEL_DIR
    / "per_task_language_metrics.csv"
)
MODEL_ERROR_SUMMARY_PATH = (
    LANGUAGE_MODEL_DIR
    / "error_summary.json"
)
MODEL_GUARDRAIL_SUMMARY_PATH = (
    LANGUAGE_MODEL_DIR
    / "guardrail_summary.json"
)

shutil.copy2(
    LANGUAGE_EVALUATION_METRICS_PATH,
    MODEL_EVALUATION_METRICS_PATH,
)
shutil.copy2(
    PER_TASK_LANGUAGE_METRICS_PATH,
    MODEL_PER_TASK_METRICS_PATH,
)
shutil.copy2(
    LANGUAGE_ERROR_SUMMARY_PATH,
    MODEL_ERROR_SUMMARY_PATH,
)
shutil.copy2(
    GUARDRAIL_SUMMARY_PATH,
    MODEL_GUARDRAIL_SUMMARY_PATH,
)


# -------------------------------------------------------------------------
# Extend the model metadata without replacing raw-model results
# -------------------------------------------------------------------------
language_model_metadata[
    "held_out_language_evaluation"
] = {
    "evaluation_partition": (
        "language_test"
    ),
    "records": int(
        held_out_language_metrics[
            "records_evaluated"
        ]
    ),
    "raw_model_metrics": (
        held_out_language_metrics
    ),
    "operational_generation": (
        held_out_generation_runtime
    ),
    "error_profile": (
        language_error_summary
    ),
}

language_model_metadata[
    "output_guardrail"
] = {
    "strategy": (
        "deterministic_validation_with_"
        "controlled_structured_fallback"
    ),
    "raw_generation_acceptance_rate": (
        guardrail_summary[
            "raw_generation_acceptance_rate"
        ]
    ),
    "safe_template_fallback_rate": (
        guardrail_summary[
            "safe_template_fallback_rate"
        ]
    ),
    "post_guardrail_metrics": (
        guardrail_summary[
            "post_guardrail_metrics"
        ]
    ),
    "raw_metrics_preserved": True,
}

language_model_metadata[
    "service_integration_contract"
] = {
    "validate_every_generation": True,
    "return_guardrail_action": True,
    "return_trigger_reasons": True,
    "return_language_model_version": True,
    "return_prompt_registry_version": True,
    "return_computer_vision_model_version": True,
    "return_request_id": True,
    "return_generation_latency": True,
}

language_model_metadata[
    "last_updated_utc"
] = datetime.now(
    timezone.utc
).isoformat()

with LANGUAGE_MODEL_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        language_model_metadata,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Define the complete language-stage artifact inventory
# -------------------------------------------------------------------------
LANGUAGE_ARTIFACT_REGISTRY_PATH = (
    LANGUAGE_OUTPUT_DIR
    / "language_artifact_registry.yaml"
)

language_artifact_groups = {
    "derived_dataset": [
        LANGUAGE_JSONL_PATHS["train"],
        LANGUAGE_JSONL_PATHS["validation"],
        LANGUAGE_JSONL_PATHS["language_test"],
        LANGUAGE_DATASET_DESIGN_PATH,
        LANGUAGE_DATASET_MANIFEST_PATH,
        LANGUAGE_RECORD_INDEX_PATH,
        GROUNDING_AUDIT_PATH,
        PROMPT_REGISTRY_PATH,
    ],
    "training": [
        LANGUAGE_TRAINING_CONFIG_PATH,
        LANGUAGE_TRAINING_HISTORY_PATH,
        LANGUAGE_PARAMETER_CHANGE_PATH,
        LANGUAGE_TRAINING_SUMMARY_PATH,
    ],
    "versioned_model": [
        LANGUAGE_MODEL_DIR / "config.json",
        LANGUAGE_MODEL_DIR
        / "generation_config.json",
        LANGUAGE_MODEL_DIR
        / "model.safetensors",
        LANGUAGE_MODEL_DIR
        / "tokenizer_config.json",
        LANGUAGE_MODEL_DIR
        / "special_tokens_map.json",
        LANGUAGE_MODEL_DIR
        / "spiece.model",
        LANGUAGE_MODEL_METADATA_PATH,
        MODEL_TRAINING_CONFIG_PATH,
        MODEL_PROMPT_REGISTRY_PATH,
        MODEL_DATASET_MANIFEST_PATH,
        MODEL_TRAINING_SUMMARY_PATH,
        MODEL_PARAMETER_CHANGE_PATH,
        MODEL_EXPORT_PARITY_PATH,
        MODEL_EVALUATION_METRICS_PATH,
        MODEL_PER_TASK_METRICS_PATH,
        MODEL_ERROR_SUMMARY_PATH,
        MODEL_GUARDRAIL_SUMMARY_PATH,
    ],
    "held_out_evaluation": [
        HELD_OUT_GENERATIONS_PATH,
        HELD_OUT_GENERATION_RUNTIME_PATH,
        LANGUAGE_EVALUATION_METRICS_PATH,
        PER_TASK_LANGUAGE_METRICS_PATH,
        HELD_OUT_GROUNDING_AUDIT_PATH,
        LANGUAGE_ERROR_ANALYSIS_PATH,
        LANGUAGE_ERROR_SUMMARY_PATH,
        EXAMPLE_GENERATIONS_PATH,
    ],
    "operational_guardrails": [
        GUARDED_GENERATIONS_PATH,
        GUARDED_OUTPUT_AUDIT_PATH,
        GUARDRAIL_SUMMARY_PATH,
    ],
}


# -------------------------------------------------------------------------
# Validate files before calculating artifact checksums
# -------------------------------------------------------------------------
missing_language_artifacts = [
    str(file_path)
    for artifact_paths in (
        language_artifact_groups.values()
    )
    for file_path in artifact_paths
    if not file_path.is_file()
]

if missing_language_artifacts:
    raise FileNotFoundError(
        "Required language-stage artifacts are missing: "
        + ", ".join(missing_language_artifacts)
    )

language_artifact_registry = {
    "registry_version": (
        "grounded-language-artifacts-v1"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "language_model_version": (
        LANGUAGE_MODEL_VERSION
    ),
    "dataset_version": (
        LANGUAGE_DATASET_VERSION
    ),
    "prompt_registry_version": (
        PROMPT_REGISTRY_VERSION
    ),
    "mlflow": {
        "tracking_uri": MLFLOW_TRACKING_URI,
        "experiment_name": (
            LANGUAGE_MLFLOW_EXPERIMENT
        ),
        "experiment_id": (
            language_mlflow_experiment.experiment_id
        ),
        "run_id": LANGUAGE_MLFLOW_RUN_ID,
    },
    "artifact_groups": {
        group_name: [
            {
                "path": str(file_path),
                "bytes": int(
                    file_path.stat().st_size
                ),
                "sha256": calculate_sha256(
                    file_path
                ),
            }
            for file_path in artifact_paths
        ]
        for group_name, artifact_paths
        in language_artifact_groups.items()
    },
}

with LANGUAGE_ARTIFACT_REGISTRY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        language_artifact_registry,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Register the finalized metadata and artifact registry
# -------------------------------------------------------------------------
with mlflow.start_run(
    run_id=LANGUAGE_MLFLOW_RUN_ID
):
    mlflow.log_artifact(
        str(LANGUAGE_MODEL_METADATA_PATH),
        artifact_path="model",
    )
    mlflow.log_artifact(
        str(LANGUAGE_ARTIFACT_REGISTRY_PATH),
        artifact_path="artifact_registry",
    )

    mlflow.set_tags(
        {
            "language_artifact_registry_status": (
                "completed"
            ),
            "notebook_06_status": "completed",
            "service_integration_readiness": (
                "ready"
            ),
        }
    )


# -------------------------------------------------------------------------
# Apply the final language-integration readiness gate
# -------------------------------------------------------------------------
final_language_run = (
    language_mlflow_client.get_run(
        LANGUAGE_MLFLOW_RUN_ID
    )
)

total_registered_artifacts = sum(
    len(artifact_paths)
    for artifact_paths
    in language_artifact_groups.values()
)

free_storage_final_gib = (
    shutil.disk_usage(DATA_ROOT).free / GIB
)

final_language_readiness_checks = {
    "Genuine full-model fine-tuning completed": (
        trainable_parameters
        == total_parameters
        and parameter_change_validation_df[
            "parameter_changed"
        ].all()
    ),
    "Training and validation partitions were used correctly": (
        len(
            tokenized_language_dataset[
                "train"
            ]
        )
        == 3600
        and len(
            tokenized_language_dataset[
                "validation"
            ]
        )
        == 600
    ),
    "Held-out language test remained isolated until evaluation": (
        language_model_metadata[
            "held_out_language_test_used_for_training"
        ]
        is False
    ),
    "Versioned model export is complete": (
        LANGUAGE_MODEL_METADATA_PATH.is_file()
        and len(exported_weight_files) >= 1
    ),
    "All four language tasks were evaluated": (
        set(
            per_task_language_metrics_df[
                "task_type"
            ]
        )
        == set(TASK_NAMES)
    ),
    "Raw outputs contain no unsupported findings": (
        held_out_language_metrics[
            "unsupported_finding_rate"
        ]
        == 0.0
    ),
    "Raw outputs preserve the safety boundary": (
        held_out_language_metrics[
            "safety_boundary_compliant"
        ]
        == 1.0
    ),
    "Raw outputs contain no forbidden claims": (
        held_out_language_metrics[
            "forbidden_claim_free"
        ]
        == 1.0
    ),
    "Guardrail preserves every output contract": all(
        metric_value == 1.0
        for metric_value
        in guardrail_summary[
            "post_guardrail_metrics"
        ].values()
    ),
    "Generation latency and throughput are available": (
        held_out_generation_runtime[
            "milliseconds_per_record"
        ]
        > 0.0
        and held_out_generation_runtime[
            "records_per_second"
        ]
        > 0.0
    ),
    "Artifact registry contains every required file": (
        not missing_language_artifacts
        and total_registered_artifacts > 0
    ),
    "MLflow run remains finished": (
        final_language_run.info.status
        == "FINISHED"
    ),
    "MLflow final readiness tag is complete": (
        final_language_run.data.tags.get(
            "service_integration_readiness"
        )
        == "ready"
    ),
    "Protected storage reserve remains available": (
        free_storage_final_gib
        >= PROTECTED_RESERVE_GIB
    ),
}


# -------------------------------------------------------------------------
# Report final language-stage readiness
# -------------------------------------------------------------------------
print("GROUNDED LANGUAGE INTEGRATION READINESS")
print("-" * 100)
print(
    f"Language model version        : "
    f"{LANGUAGE_MODEL_VERSION}"
)
print(
    f"Dataset version               : "
    f"{LANGUAGE_DATASET_VERSION}"
)
print(
    f"Prompt registry version       : "
    f"{PROMPT_REGISTRY_VERSION}"
)
print(
    f"MLflow experiment ID          : "
    f"{language_mlflow_experiment.experiment_id}"
)
print(
    f"MLflow run ID                 : "
    f"{LANGUAGE_MLFLOW_RUN_ID}"
)
print(
    f"MLflow run status             : "
    f"{final_language_run.info.status}"
)
print(
    f"Registered artifact files     : "
    f"{total_registered_artifacts}"
)
print(
    f"Raw ROUGE-L F1                : "
    f"{held_out_language_metrics['rouge_l_f1']:.4f}"
)
print(
    f"Raw BLEU-4                    : "
    f"{held_out_language_metrics['smoothed_corpus_bleu_4']:.4f}"
)
print(
    f"Raw finding grounding         : "
    f"{held_out_language_metrics['finding_grounding_compliant']:.4f}"
)
print(
    f"Raw unsupported-finding rate  : "
    f"{held_out_language_metrics['unsupported_finding_rate']:.4f}"
)
print(
    f"Raw safety compliance         : "
    f"{held_out_language_metrics['safety_boundary_compliant']:.4f}"
)
print(
    f"Raw generation acceptance     : "
    f"{guardrail_summary['raw_generation_acceptance_rate']:.4f}"
)
print(
    f"Guarded contract compliance   : "
    f"{min(guardrail_summary['post_guardrail_metrics'].values()):.4f}"
)
print(
    f"Generation throughput         : "
    f"{held_out_generation_runtime['records_per_second']:.2f} "
    f"records/second"
)
print(
    f"Average generation latency    : "
    f"{held_out_generation_runtime['milliseconds_per_record']:.2f} "
    f"ms/record"
)
print(
    f"Free storage                  : "
    f"{free_storage_final_gib:.2f} GiB"
)
print(
    f"Artifact registry             : "
    f"{LANGUAGE_ARTIFACT_REGISTRY_PATH}"
)
print("-" * 100)

for check_name, passed in (
    final_language_readiness_checks.items()
):
    print(f"{check_name:<62}: {'PASS' if passed else 'FAIL'}")

if not all(final_language_readiness_checks.values()):
    failed_checks = [
        name
        for name, passed
        in final_language_readiness_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Final grounded language readiness gate failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("FINAL STATUS: READY FOR API SERVICE INTEGRATION")

GROUNDED LANGUAGE INTEGRATION READINESS
----------------------------------------------------------------------------------------------------
Language model version        : flan-t5-small-chestmnist-v1
Dataset version               : chestmnist-grounded-language-v1
Prompt registry version       : grounded-language-prompts-v1
MLflow experiment ID          : 583945617898602655
MLflow run ID                 : a223fe4322c24c24b10f338810c89dea
MLflow run status             : FINISHED
Registered artifact files     : 40
Raw ROUGE-L F1                : 0.8489
Raw BLEU-4                    : 0.7393
Raw finding grounding         : 0.9183
Raw unsupported-finding rate  : 0.0000
Raw safety compliance         : 1.0000
Raw generation acceptance     : 0.7267
Guarded contract compliance   : 1.0000
Generation throughput         : 13.06 records/second
Average generation latency    : 76.58 ms/record
Free storage                  : 8.71 GiB
Artifact registry             : /home/jovyan/apicdsa2-datavol-1/che

## 23. Grounded Language Integration Summary

A genuinely fine-tuned `google/flan-t5-small` model was developed and versioned as `flan-t5-small-chestmnist-v1`. Full sequence-to-sequence fine-tuning was completed using 3,600 derived training records and 600 validation records covering structured report generation, plain-language explanation, grounded question answering, and controlled educational follow-up. The 600-record held-out language-test partition remained isolated until the model and training configuration were frozen.

The raw model achieved a ROUGE-L F1 of 0.8489 and a smoothed corpus BLEU-4 score of 0.7393. It maintained 100% task-routing compliance, section-order compliance, safety-boundary compliance, Grad-CAM boundary compliance, and forbidden-claim avoidance. The unsupported-finding rate was 0.0000, demonstrating that the model did not introduce ChestMNIST findings absent from the supplied context.

Raw finding-grounding compliance was 0.9183, while exact numeric-grounding compliance was 0.8117. A deterministic output guardrail was therefore established to validate every generated response before delivery. It accepted 72.67% of raw model generations unchanged and routed the remaining 27.33% through a controlled structured fallback. The guarded workflow achieved complete compliance across all defined grounding, structure, threshold, safety, refusal, and explainability boundaries.

The frozen language component produces educational decision-support content only. It does not independently inspect chest X-ray images, confirm medical conditions, interpret Grad-CAM as lesion evidence, prescribe treatment, or replace professional medical review. The state **no target finding** continues to mean only that none of the 14 supported findings crossed its frozen threshold; it does not establish that an image is clinically normal.

The versioned model, tokenizer, prompt registry, dataset manifest, training history, evaluation results, representative generations, guardrail evidence, model metadata, and MLflow lineage were successfully registered. Generation throughput reached 13.06 records per second with an average batch-mode latency of 76.58 milliseconds per record.

**Final status: Ready for API service integration.**
